# Houston Rockets 2025-26 Season Review

This notebook produces a multi-section HTML report analyzing the Houston Rockets' 2025-26 season.

## Table of Contents

This notebook is organized in execution order. The conceptual sections of the final report appear in a different order, since some assets (like the Overview timeline) were built last.

**Foundational work:**
1. [Setup & Data Pull](#setup) — NBA API caching, pulling team/player/PBP/shot data
2. [Lineup Reconstruction](#lineup-reconstruction) — Building 5-man lineup data from ESPN PBP (used by multiple report sections)

**Report sections (built in this order):**
3. [Section 1: Playstyle Identity](#section-1) — Interior strengths + 3-point problem + shot heatmaps
4. [Section 2: The Close-Out Problem](#section-2) — Clutch comparison + four collapses + KD-off scatter + OT record
5. [Section 3: KD Reliance](#section-3) — Wheels Came Off table + headline card + aging stars + scoring involvement + gravity
6. [Section 4: Lineup Construction](#section-4) — Starting lineup + lineup ratings + team gravity
7. [Section 5: The Playoffs](#section-5) — Game 3 timeline + Path Forward
8. [Section 0: Overview](#section-0) — Numbered timeline (built last)

**Output:**
9. [Full Report Stitcher](#stitcher) — Combines all asset HTML files into one tabbed report
10. [Appendix: Verification & Fact Checks](#appendix) — End-of-build cells used to confirm specific stats

## Data sources

All data pulled from public NBA APIs:
- NBA Stats API via `nba_api` (team/player game logs, advanced stats, clutch stats, shot charts, play-by-play)
- ESPN API (play-by-play with quarter-break substitutions, used for lineup reconstruction)
- NBA.com Inside the Game tracking (gravity metric, manually transcribed)

## Output structure

Each report section produces standalone HTML asset files in `report_assets/`. The stitcher (Section 9 below) combines them into a single tabbed HTML report.


<a id="setup"></a>
# Setup & Data Pull

This first part pulls and caches all data needed for the rest of the notebook. Every NBA API response is cached as a parquet file in `data/raw/` so re-running cells is instant after the first pull. To force a refresh, delete the corresponding file.

**What gets pulled:**
1. Team game logs (per-game team stats)
2. Player rosters
3. Player game logs (per-game player stats)
4. Season-level league stats (team and player advanced metrics)
5. Shot charts (per-player shot location data)
6. Play-by-play data (used for clutch analysis, lineup reconstruction, KD-off stretches)
7. Clutch / scoring / four-factors team stats (for clutch comparison table)

## Imports, paths, and constants

In [3]:
from pathlib import Path
import time, json, hashlib, logging
from functools import wraps
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from nba_api.stats.endpoints import (
    teamgamelog, playergamelog, playbyplayv3, shotchartdetail,
    leaguedashplayerstats, leaguedashteamstats, commonteamroster,
)
from nba_api.stats.static import teams

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ROCKETS = next(t for t in teams.get_teams() if t["abbreviation"] == "HOU")
ROCKETS_ID = ROCKETS["id"]
SEASONS = ["2024-25", "2025-26"]
SEASON_TYPES = ["Regular Season", "Playoffs"]
SLEEP_BETWEEN_CALLS = 0.6
MAX_RETRIES = 3
RETRY_BACKOFF = 2.0

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

print(f"Rockets team_id: {ROCKETS_ID}")
print(f"Raw cache dir: {RAW_DIR}")

Rockets team_id: 1610612745
Raw cache dir: /Users/stevenyan/Desktop/rockets_season_review/data/raw


/Users/stevenyan/Desktop/rockets_season_review/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
def _cache_key(endpoint_name, params):
    param_str = json.dumps(params, sort_keys=True, default=str)
    short_hash = hashlib.md5(param_str.encode()).hexdigest()[:8]
    readable = "_".join(f"{k}={v}" for k, v in sorted(params.items()))
    readable = readable.replace("/", "-").replace(" ", "")[:80]
    return f"{endpoint_name}__{readable}__{short_hash}.parquet"

def _cached(endpoint_name):
    def decorator(func):
        @wraps(func)
        def wrapper(**params):
            fname = _cache_key(endpoint_name, params)
            fpath = RAW_DIR / fname
            if fpath.exists():
                return pd.read_parquet(fpath)
            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    df = func(**params)
                    time.sleep(SLEEP_BETWEEN_CALLS)
                    df.to_parquet(fpath, index=False)
                    return df
                except Exception as e:
                    wait = RETRY_BACKOFF ** attempt
                    log.warning(f"{endpoint_name} failed (attempt {attempt}): {e}. Retry in {wait}s")
                    time.sleep(wait)
            raise RuntimeError(f"{endpoint_name} failed after {MAX_RETRIES} attempts: {params}")
        return wrapper
    return decorator

@_cached("team_gamelog")
def get_team_gamelog(team_id, season, season_type):
    return teamgamelog.TeamGameLog(team_id=team_id, season=season, season_type_all_star=season_type).get_data_frames()[0]

@_cached("player_gamelog")
def get_player_gamelog(player_id, season, season_type):
    return playergamelog.PlayerGameLog(player_id=player_id, season=season, season_type_all_star=season_type).get_data_frames()[0]

@_cached("pbp")
def get_play_by_play(game_id):
    return playbyplayv3.PlayByPlayV3(game_id=game_id).get_data_frames()[0]

@_cached("shotchart")
def get_shotchart(player_id, season, season_type):
    return shotchartdetail.ShotChartDetail(
        team_id=0, player_id=player_id, season_nullable=season,
        season_type_all_star=season_type, context_measure_simple="FGA",
    ).get_data_frames()[0]

@_cached("league_player_stats")
def get_league_player_stats(season, season_type, measure_type):
    return leaguedashplayerstats.LeagueDashPlayerStats(
        season=season, season_type_all_star=season_type,
        measure_type_detailed_defense=measure_type,
    ).get_data_frames()[0]

@_cached("league_team_stats")
def get_league_team_stats(season, season_type, measure_type):
    return leaguedashteamstats.LeagueDashTeamStats(
        season=season, season_type_all_star=season_type,
        measure_type_detailed_defense=measure_type,
    ).get_data_frames()[0]

@_cached("roster")
def get_roster(team_id, season):
    return commonteamroster.CommonTeamRoster(team_id=team_id, season=season).get_data_frames()[0]

print("API wrappers loaded")

API wrappers loaded


## Sanity check — one API call to confirm the pipeline works

Before kicking off the full pull, run one small request to confirm the API is reachable and the cache decorator is functioning.

In [5]:
# Quick test - this is the smallest possible pull
test_df = get_team_gamelog(team_id=ROCKETS_ID, season="2025-26", season_type="Regular Season")
print(f"Got {len(test_df)} games for 2025-26 regular season")
test_df.head()

Got 82 games for 2025-26 regular season


,Team_ID,Game_ID,GAME_DATE,MATCHUP,WL,W,L,W_PCT,MIN,FGM,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS
0,1610612745,0022501194,"APR 12, 2026",HOU vs. MEM,W,52,30,0.634,240,48,...,0.733,21,43,64,23,10,8,11,10,132
1,1610612745,0022501178,"APR 10, 2026",HOU vs. MIN,L,51,30,0.630,240,53,...,0.786,11,28,39,29,4,8,12,18,132
2,1610612745,0022501169,"APR 09, 2026",HOU vs. PHI,W,51,29,0.638,240,42,...,0.692,16,37,53,21,12,1,15,18,113
3,1610612745,0022501157,"APR 07, 2026",HOU @ PHX,W,50,29,0.633,240,45,...,0.938,24,31,55,26,8,3,17,30,119
4,1610612745,0022501142,"APR 05, 2026",HOU @ GSW,W,49,29,0.628,240,44,...,0.842,8,30,38,30,4,7,13,16,117


If that returned ~82 rows, the API is working. The result is now cached — running the cell again is instant.

## 2. Pull all team game logs

This is fast (4 calls total).

## Pull all team game logs (2024-25 and 2025-26, regular season + playoffs)

In [6]:
team_gamelogs = []
for season in SEASONS:
    for stype in SEASON_TYPES:
        print(f"Pulling team gamelog: {season} {stype}")
        df = get_team_gamelog(team_id=ROCKETS_ID, season=season, season_type=stype)
        df["SEASON"] = season
        df["SEASON_TYPE"] = stype
        team_gamelogs.append(df)

team_gamelogs = pd.concat(team_gamelogs, ignore_index=True)
team_gamelogs.to_parquet(PROCESSED_DIR / "rockets_team_gamelogs.parquet", index=False)
print(f"\nTotal Rockets games: {len(team_gamelogs)}")
team_gamelogs.groupby(['SEASON', 'SEASON_TYPE']).size()

Pulling team gamelog: 2024-25 Regular Season
Pulling team gamelog: 2024-25 Playoffs
Pulling team gamelog: 2025-26 Regular Season
Pulling team gamelog: 2025-26 Playoffs

Total Rockets games: 177


SEASON   SEASON_TYPE   
2024-25  Playoffs           7
         Regular Season    82
2025-26  Playoffs           6
         Regular Season    82
dtype: int64

## Pull team rosters (to find every Rockets player across both seasons)

In [7]:
rosters = []
for season in SEASONS:
    print(f"Pulling roster: {season}")
    df = get_roster(team_id=ROCKETS_ID, season=season)
    df["SEASON"] = season
    rosters.append(df)

rosters = pd.concat(rosters, ignore_index=True)
print(f"\nRoster rows: {len(rosters)}")
print(f"Unique players across both seasons: {rosters['PLAYER_ID'].nunique()}")
rosters[["SEASON", "PLAYER", "POSITION", "AGE"]].head(10)

Pulling roster: 2024-25
Pulling roster: 2025-26

Roster rows: 35
Unique players across both seasons: 25


,SEASON,PLAYER,POSITION,AGE
0,2024-25,Aaron Holiday,G,28.0
1,2024-25,Amen Thompson,G-F,22.0
2,2024-25,Jock Landale,C,29.0
3,2024-25,N'Faly Dante,C,23.0
4,2024-25,Jalen Green,G,23.0
5,2024-25,Fred VanVleet,G,31.0
6,2024-25,Cam Whitmore,F,20.0
7,2024-25,Jae'Sean Tate,F,29.0
8,2024-25,Dillon Brooks,G-F,29.0
9,2024-25,Jabari Smith Jr.,F,22.0


In [8]:
# Get unique player IDs
unique_players = rosters[["PLAYER_ID", "PLAYER"]].drop_duplicates(subset="PLAYER_ID")
unique_players.to_parquet(PROCESSED_DIR / "rockets_players.parquet", index=False)
print(f"{len(unique_players)} unique players to pull data for")
unique_players.head(15)

25 unique players to pull data for


,PLAYER_ID,PLAYER
0,1628988,Aaron Holiday
1,1641708,Amen Thompson
2,1629111,Jock Landale
3,1642368,N'Faly Dante
4,1630224,Jalen Green
5,1627832,Fred VanVleet
6,1641715,Cam Whitmore
7,1630256,Jae'Sean Tate
8,1628415,Dillon Brooks
9,1631095,Jabari Smith Jr.


## Pull player game logs

One API call per (player × season × season_type). With ~25 players × 2 seasons × 2 season types, this is the slowest cached section after PBP.

In [9]:
player_gamelogs = []
errors = []

for _, row in tqdm(unique_players.iterrows(), total=len(unique_players), desc="Players"):
    pid = int(row["PLAYER_ID"])
    pname = row["PLAYER"]
    for season in SEASONS:
        for stype in SEASON_TYPES:
            try:
                df = get_player_gamelog(player_id=pid, season=season, season_type=stype)
                if len(df) > 0:
                    df["SEASON"] = season
                    df["SEASON_TYPE"] = stype
                    player_gamelogs.append(df)
            except Exception as e:
                errors.append((pname, season, stype, str(e)))

player_gamelogs = pd.concat(player_gamelogs, ignore_index=True) if player_gamelogs else pd.DataFrame()
player_gamelogs.to_parquet(PROCESSED_DIR / "rockets_player_gamelogs.parquet", index=False)
print(f"\nTotal player-game rows: {len(player_gamelogs)}")
print(f"Errors: {len(errors)}")
if errors:
    for e in errors[:5]:
        print(f"  {e}")

Players:   0%|          | 0/25 [00:00<?, ?it/s]


Total player-game rows: 2345
Errors: 0


**Heads up:** `playergamelog` returns games for the player **across all teams that season**. If a player was traded, you'll see games for both teams. We filter to just Rockets games using the team_gamelogs game_ids.

In [10]:
# Filter player gamelogs to only Rockets games
rockets_game_ids = set(team_gamelogs["Game_ID"].unique())
print(f"Rockets game IDs: {len(rockets_game_ids)}")

player_gamelogs_rockets = player_gamelogs[player_gamelogs["Game_ID"].isin(rockets_game_ids)].copy()
print(f"Player-game rows before filter: {len(player_gamelogs)}")
print(f"Player-game rows after filter (Rockets games only): {len(player_gamelogs_rockets)}")

player_gamelogs_rockets.to_parquet(
    PROCESSED_DIR / "rockets_player_gamelogs_filtered.parquet", index=False
)

Rockets game IDs: 177
Player-game rows before filter: 2345
Player-game rows after filter (Rockets games only): 1855


## Pull season-level league stats (team and player advanced metrics)

In [11]:
# Team stats (for league ranking comparisons)
team_stats = {}
for season in SEASONS:
    for mt in ["Base", "Advanced"]:
        print(f"Pulling league team stats: {season} {mt}")
        df = get_league_team_stats(season=season, season_type="Regular Season", measure_type=mt)
        team_stats[(season, mt)] = df

print(f"\nGot team stats for {len(team_stats)} (season, measure_type) combos")
team_stats[("2025-26", "Advanced")][["TEAM_NAME", "OFF_RATING", "DEF_RATING", "NET_RATING"]].sort_values("NET_RATING", ascending=False).head(10)

Pulling league team stats: 2024-25 Base
Pulling league team stats: 2024-25 Advanced
Pulling league team stats: 2025-26 Base
Pulling league team stats: 2025-26 Advanced

Got team stats for 4 (season, measure_type) combos


,TEAM_NAME,OFF_RATING,DEF_RATING,NET_RATING
20,Oklahoma City Thunder,117.6,106.5,11.1
26,San Antonio Spurs,118.7,110.4,8.4
8,Detroit Pistons,117.3,108.9,8.4
1,Boston Celtics,120.0,111.7,8.3
19,New York Knicks,118.7,112.3,6.4
10,Houston Rockets,117.5,112.1,5.4
7,Denver Nuggets,121.2,116.0,5.2
3,Charlotte Hornets,118.4,113.5,4.9
5,Cleveland Cavaliers,118.3,114.1,4.1
17,Minnesota Timberwolves,115.6,112.5,3.1


In [12]:
# Player stats (advanced metrics like usage rate, TS%, etc.)
player_stats = {}
for season in SEASONS:
    for mt in ["Base", "Advanced"]:
        print(f"Pulling league player stats: {season} {mt}")
        df = get_league_player_stats(season=season, season_type="Regular Season", measure_type=mt)
        player_stats[(season, mt)] = df

print(f"\nGot player stats for {len(player_stats)} combos")

Pulling league player stats: 2024-25 Base
Pulling league player stats: 2024-25 Advanced
Pulling league player stats: 2025-26 Base
Pulling league player stats: 2025-26 Advanced

Got player stats for 4 combos


## Pull shot charts for every Rockets player

In [13]:
shot_chart_frames = []
errors = []

for _, row in tqdm(unique_players.iterrows(), total=len(unique_players), desc="Shot charts"):
    pid = int(row["PLAYER_ID"])
    pname = row["PLAYER"]
    for season in SEASONS:
        for stype in SEASON_TYPES:
            try:
                df = get_shotchart(player_id=pid, season=season, season_type=stype)
                if len(df) > 0:
                    df["SEASON_LABEL"] = season
                    df["SEASON_TYPE"] = stype
                    shot_chart_frames.append(df)
            except Exception as e:
                errors.append((pname, season, stype, str(e)))

if shot_chart_frames:
    shot_charts = pd.concat(shot_chart_frames, ignore_index=True)
    shot_charts.to_parquet(PROCESSED_DIR / "rockets_shot_charts.parquet", index=False)
    print(f"\nTotal shots logged: {len(shot_charts)}")
else:
    print("No shot data returned")

print(f"Errors: {len(errors)}")

Shot charts:   0%|          | 0/25 [00:00<?, ?it/s]


Total shots logged: 20954
Errors: 0


## Pull play-by-play for all 82 Rockets games

This is the slowest section — one API call per game. Cached as parquet after the first run. Used downstream for:
- Clutch / blown leads analysis (Section 2)
- KD-off stretches (Section 3)
- Lineup reconstruction (foundational work, switched to ESPN later)

In [14]:
pbp_frames = []
game_ids = sorted(team_gamelogs["Game_ID"].unique().tolist())
print(f"Pulling play-by-play for {len(game_ids)} games...")

for gid in tqdm(game_ids, desc="PBP"):
    gid_str = str(gid).zfill(10)
    try:
        df = get_play_by_play(game_id=gid_str)
        pbp_frames.append(df)
    except Exception as e:
        print(f"Failed {gid_str}: {e}")

play_by_play = pd.concat(pbp_frames, ignore_index=True)
play_by_play.to_parquet(PROCESSED_DIR / "rockets_play_by_play.parquet", index=False)
print(f"\nTotal PBP events: {len(play_by_play):,}")
print(f"Avg events per game: {len(play_by_play) / len(game_ids):.0f}")

Pulling play-by-play for 177 games...


PBP:   0%|          | 0/177 [00:00<?, ?it/s]


Total PBP events: 89,540
Avg events per game: 506


## Pull additional team measure types (Misc, Scoring, Four Factors)

For metrics like paint points, 2nd chance points, %FGA from three.

In [15]:
# Pull team stats with all the measure types we need
# Base, Advanced: already pulled
# Misc: paint points, 2nd chance points
# Scoring: %FGA 3PT
# Four Factors: optional but useful

EXTRA_MEASURE_TYPES = ["Misc", "Scoring", "Four Factors"]

for season in SEASONS:
    for mt in EXTRA_MEASURE_TYPES:
        print(f"Pulling team stats: {season} {mt}")
        get_league_team_stats(season=season, season_type="Regular Season", measure_type=mt)

print("Extra team measure types done")

Pulling team stats: 2024-25 Misc
Pulling team stats: 2024-25 Scoring
Pulling team stats: 2024-25 Four Factors
Pulling team stats: 2025-26 Misc
Pulling team stats: 2025-26 Scoring
Pulling team stats: 2025-26 Four Factors
Extra team measure types done


## Pull clutch + scoring league dashboards

Used in Section 2's clutch comparison table.

In [16]:
from nba_api.stats.endpoints import (
    leaguedashteamclutch,
    leaguedashplayerclutch,
)

@_cached("league_team_clutch")
def get_league_team_clutch(season, season_type, measure_type):
    return leaguedashteamclutch.LeagueDashTeamClutch(
        season=season,
        season_type_all_star=season_type,
        measure_type_detailed_defense=measure_type,
        clutch_time="Last 5 Minutes",
        ahead_behind="Ahead or Behind",
        point_diff=5,
    ).get_data_frames()[0]

@_cached("league_player_clutch")
def get_league_player_clutch(season, season_type, measure_type):
    return leaguedashplayerclutch.LeagueDashPlayerClutch(
        season=season,
        season_type_all_star=season_type,
        measure_type_detailed_defense=measure_type,
        clutch_time="Last 5 Minutes",
        ahead_behind="Ahead or Behind",
        point_diff=5,
    ).get_data_frames()[0]

# Pull clutch data
for season in SEASONS:
    for mt in ["Base", "Advanced"]:
        print(f"Team clutch: {season} {mt}")
        get_league_team_clutch(season=season, season_type="Regular Season", measure_type=mt)
        print(f"Player clutch: {season} {mt}")
        get_league_player_clutch(season=season, season_type="Regular Season", measure_type=mt)

print("Clutch data done")

Team clutch: 2024-25 Base
Player clutch: 2024-25 Base
Team clutch: 2024-25 Advanced
Player clutch: 2024-25 Advanced
Team clutch: 2025-26 Base
Player clutch: 2025-26 Base
Team clutch: 2025-26 Advanced
Player clutch: 2025-26 Advanced
Clutch data done


## Data quality check — inventory what we've cached

Final sanity check before moving on to analysis. Confirms all expected parquet files exist and have reasonable row counts.

In [18]:
# Inventory everything we've saved
print("=" * 60)
print("PROCESSED FILES")
print("=" * 60)
for f in sorted(PROCESSED_DIR.glob("*.parquet")):
    df = pd.read_parquet(f)
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:50s}  {len(df):>7,} rows  {size_kb:>8,.0f} KB")

print()
print("=" * 60)
print("RAW CACHE FILES")
print("=" * 60)
raw_files = list(RAW_DIR.glob("*.parquet"))
raw_total_kb = sum(f.stat().st_size for f in raw_files) / 1024
print(f"  {len(raw_files)} cached API responses, total {raw_total_kb:,.0f} KB")

PROCESSED FILES
  rockets_player_gamelogs.parquet                       2,345 rows        59 KB
  rockets_player_gamelogs_filtered.parquet              1,855 rows        46 KB
  rockets_players.parquet                                  25 rows         2 KB
  rockets_shot_charts.parquet                          20,954 rows       219 KB
  rockets_team_gamelogs.parquet                           177 rows        25 KB

RAW CACHE FILES
  405 cached API responses, total 11,117 KB


In [ ]:
# Team game logs check
team_gl = pd.read_parquet(PROCESSED_DIR / "rockets_team_gamelogs.parquet")
print(f"Total rows: {len(team_gl)}")
print(team_gl.groupby(['SEASON', 'SEASON_TYPE']).size())
print(f"\nDate range: {team_gl['GAME_DATE'].min()} to {team_gl['GAME_DATE'].max()}")

Total rows: 177
SEASON   SEASON_TYPE   
2024-25  Playoffs           7
         Regular Season    82
2025-26  Playoffs           6
         Regular Season    82
dtype: int64

Date range: APR 01, 2026 to OCT 31, 2024


In [20]:
player_gl = pd.read_parquet(PROCESSED_DIR / "rockets_player_gamelogs_filtered.parquet")
print(f"Total player-game rows: {len(player_gl)}")
print(f"Unique players: {player_gl['Player_ID'].nunique()}")
print(f"\nTop players by games played:")
print(player_gl.groupby('Player_ID').size().sort_values(ascending=False).head(10))

Total player-game rows: 1855
Unique players: 25

Top players by games played:
Player_ID
1641708    161
1630578    161
1631095    147
1642263    143
1631106    130
1628988    128
1630256    103
203500      97
1630224     90
1628415     86
dtype: int64


In [21]:
shots = pd.read_parquet(PROCESSED_DIR / "rockets_shot_charts.parquet")
print(f"Total shots: {len(shots):,}")
print(f"Unique players: {shots['PLAYER_NAME'].nunique()}")
print(f"\nShots by player (top 10):")
print(shots.groupby('PLAYER_NAME').size().sort_values(ascending=False).head(10))

Total shots: 20,954
Unique players: 25

Shots by player (top 10):
PLAYER_NAME
Kevin Durant        2512
Alperen Sengun      2496
Jalen Green         2118
Dillon Brooks       2002
Amen Thompson       1912
Jabari Smith Jr.    1660
Reed Sheppard       1275
Tari Eason          1248
Fred VanVleet        852
Jock Landale         702
dtype: int64


In [24]:
pbp = pd.read_parquet(PROCESSED_DIR / "rockets_play_by_play.parquet")
print(f"Total PBP events: {len(pbp):,}")
print(f"Unique games: {pbp['gameId'].nunique()}")
print(f"\nColumns: {pbp.columns.tolist()}")

Total PBP events: 89,540
Unique games: 177

Columns: ['gameId', 'actionNumber', 'clock', 'period', 'teamId', 'teamTricode', 'personId', 'playerName', 'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult', 'isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal', 'location', 'description', 'actionType', 'subType', 'videoAvailable', 'shotValue', 'actionId']


<a id="section-1"></a>
# Section 1: Playstyle Identity

Builds the 6 assets for the Playstyle Identity tab of the report:

| File | What it shows |
|---|---|
| `01a_interior_identity.html` | Strengths table (paint, transition, free throws) |
| `01b_three_point_problem.html` | 3-point weakness comparison |
| `01c_three_ball_didnt_fall.html` | Sub-10 3PM games callout |
| `01d_heatmap_starters.html` | Amen + Sengun shot heatmaps |
| `01e_heatmap_supporting_cast.html` | DFS + Eason (pre/post ASB) heatmaps |
| `01f_heatmap_reliable_shooters.html` | KD + Jabari + Reed heatmaps + trio breakdown bars |

## Load data sources for Section 1

In [ ]:
# Load the data sources we need for Section 1
team_advanced_2526 = pd.read_parquet(
    next(RAW_DIR.glob("league_team_stats__measure_type=Advanced_season=2025-26*"))
)
team_advanced_2425 = pd.read_parquet(
    next(RAW_DIR.glob("league_team_stats__measure_type=Advanced_season=2024-25*"))
)
team_base_2526 = pd.read_parquet(
    next(RAW_DIR.glob("league_team_stats__measure_type=Base_season=2025-26*"))
)
team_base_2425 = pd.read_parquet(
    next(RAW_DIR.glob("league_team_stats__measure_type=Base_season=2024-25*"))
)
team_misc_2526 = pd.read_parquet(
    next(RAW_DIR.glob("league_team_stats__measure_type=Misc_season=2025-26*"))
)
team_misc_2425 = pd.read_parquet(
    next(RAW_DIR.glob("league_team_stats__measure_type=Misc_season=2024-25*"))
)
team_scoring_2526 = pd.read_parquet(
    next(RAW_DIR.glob("league_team_stats__measure_type=Scoring_season=2025-26*"))
)
team_scoring_2425 = pd.read_parquet(
    next(RAW_DIR.glob("league_team_stats__measure_type=Scoring_season=2024-25*"))
)



In [26]:
print("Advanced 2025-26 — Houston:")
print(team_advanced_2526[team_advanced_2526['TEAM_ID'] == ROCKETS_ID].iloc[0][
    ['TEAM_NAME', 'GP', 'OFF_RATING', 'DEF_RATING', 'NET_RATING', 'OREB_PCT', 'OREB_PCT_RANK']
])

Advanced 2025-26 — Houston:
TEAM_NAME        Houston Rockets
GP                            82
OFF_RATING                 117.5
DEF_RATING                 112.1
NET_RATING                   5.4
OREB_PCT                   0.388
OREB_PCT_RANK                  1
Name: 10, dtype: object


In [27]:
# What columns does each measure type give us?
print("BASE columns:", team_base_2526.columns.tolist()[:30])
print()
print("ADVANCED columns:", team_advanced_2526.columns.tolist()[:30])
print()
print("MISC columns:", team_misc_2526.columns.tolist()[:30])
print()
print("SCORING columns:", team_scoring_2526.columns.tolist()[:30])

BASE columns: ['TEAM_ID', 'TEAM_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'GP_RANK', 'W_RANK']

ADVANCED columns: ['TEAM_ID', 'TEAM_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'E_OFF_RATING', 'OFF_RATING', 'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT', 'EFG_PCT', 'TS_PCT', 'E_PACE', 'PACE', 'PACE_PER40', 'POSS', 'PIE', 'GP_RANK', 'W_RANK', 'L_RANK']

MISC columns: ['TEAM_ID', 'TEAM_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'PTS_OFF_TOV', 'PTS_2ND_CHANCE', 'PTS_FB', 'PTS_PAINT', 'OPP_PTS_OFF_TOV', 'OPP_PTS_2ND_CHANCE', 'OPP_PTS_FB', 'OPP_PTS_PAINT', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'PTS_OFF_TOV_RANK', 'PTS_2ND_CHANCE_RANK', 'PTS_FB_RANK', 'PTS_PAINT_RANK', 'OPP_PTS_OFF_TOV_RANK', 'OPP_PTS_2ND_CHANCE_RANK', 'OPP_PTS

## Build the playstyle tables (interior identity + 3-point problem + sub-10 callout)

### Three asset files in this group

- `01a_interior_identity.html` — Strengths table
- `01b_three_point_problem.html` — 3PT struggles table
- `01c_three_ball_didnt_fall.html` — Sub-10 3PM callout

In [40]:
from nba_api.stats.static import teams as nba_teams

all_teams = nba_teams.get_teams()
print(f"Pulling 2025-26 game logs for all {len(all_teams)} teams...")

league_gamelogs_2526 = []
for team in tqdm(all_teams, desc="Teams"):
    try:
        df = get_team_gamelog(
            team_id=team['id'], 
            season="2025-26", 
            season_type="Regular Season"
        )
        if len(df) > 0:
            df = df.copy()
            df["TEAM_NAME"] = team['full_name']
            df["TEAM_ABBR"] = team['abbreviation']
            league_gamelogs_2526.append(df)
    except Exception as e:
        print(f"Failed: {team['abbreviation']}: {e}")

league_gamelogs_2526 = pd.concat(league_gamelogs_2526, ignore_index=True)
league_gamelogs_2526.to_parquet(PROCESSED_DIR / "league_team_gamelogs_2526.parquet", index=False)
print(f"\nTotal team-games: {len(league_gamelogs_2526):,}")

Pulling 2025-26 game logs for all 30 teams...


Teams:   0%|          | 0/30 [00:00<?, ?it/s]


Total team-games: 2,460


In [41]:
# How many sub-10 3PM games per team in 2025-26?
sub10_counts = (
    league_gamelogs_2526
    .groupby("TEAM_ABBR")
    .apply(lambda g: (g["FG3M"] < 10).sum())
    .sort_values(ascending=False)
)

print("Teams with most sub-10 3PM games in 2025-26:")
print(sub10_counts.head(10))
print(f"\nLeague average: {sub10_counts.mean():.1f} games per team")
print(f"League median: {sub10_counts.median():.1f}")

# Houston specifics
hou = league_gamelogs_2526[league_gamelogs_2526["TEAM_ABBR"] == "HOU"]
hou_low = hou[hou["FG3M"] < 10]
hou_high = hou[hou["FG3M"] >= 10]

print(f"\n--- Houston 2025-26 ---")
print(f"Total games: {len(hou)}")
print(f"Games under 10 3PM: {len(hou_low)} ({len(hou_low)/len(hou)*100:.0f}%)")
print(f"Houston W-L in sub-10 3PM games: {(hou_low['WL']=='W').sum()}-{(hou_low['WL']=='L').sum()}")
print(f"Houston W-L in 10+ 3PM games: {(hou_high['WL']=='W').sum()}-{(hou_high['WL']=='L').sum()}")

Teams with most sub-10 3PM games in 2025-26:
TEAM_ABBR
SAC    32
DAL    29
NOP    29
DET    28
TOR    26
HOU    24
LAL    22
LAC    21
ORL    21
PHI    20
dtype: int64

League average: 13.7 games per team
League median: 11.0

--- Houston 2025-26 ---
Total games: 82
Games under 10 3PM: 24 (29%)
Houston W-L in sub-10 3PM games: 12-12
Houston W-L in 10+ 3PM games: 40-18


/var/folders/y9/138bp6nn7mb_sjlqtbfl6_fh0000gn/T/ipykernel_77227/526577975.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  league_gamelogs_2526


In [45]:
def hou_row(df):
    return df[df['TEAM_ID'] == ROCKETS_ID].iloc[0]

def get_rank(df, col, per_game=False):
    work = df.copy()
    if per_game:
        work[col] = work[col] / work['GP']
    sorted_df = work.sort_values(col, ascending=False).reset_index(drop=True)
    rank = sorted_df[sorted_df['TEAM_ID'] == ROCKETS_ID].index[0] + 1
    value = work[work['TEAM_ID'] == ROCKETS_ID].iloc[0][col]
    return rank, value

# Now with 3PM and 3PT% added
metrics = [
    ("Offensive Rebound %",    team_advanced_2425, team_advanced_2526, "OREB_PCT",       False),
    ("2nd Chance Points/G",    team_misc_2425,     team_misc_2526,     "PTS_2ND_CHANCE", True),
    ("Paint Points/G",         team_misc_2425,     team_misc_2526,     "PTS_PAINT",      True),
    ("3PA per Game",           team_base_2425,     team_base_2526,     "FG3A",           True),
    ("3PM per Game",           team_base_2425,     team_base_2526,     "FG3M",           True),
    ("3PT %",                  team_base_2425,     team_base_2526,     "FG3_PCT",        False),
    ("% of FGA from 3PT",      team_scoring_2425,  team_scoring_2526,  "PCT_FGA_3PT",    False),
]

rows = []
for label, df24, df25, col, is_count in metrics:
    rank24, val24 = get_rank(df24, col, per_game=is_count)
    rank25, val25 = get_rank(df25, col, per_game=is_count)
    rows.append({"Metric": label, "Season": "2024-25", "Rank": rank24, "Value": round(val24, 2)})
    rows.append({"Metric": label, "Season": "2025-26", "Rank": rank25, "Value": round(val25, 2)})

playstyle_df = pd.DataFrame(rows)
playstyle_df

,Metric,Season,Rank,Value
0,Offensive Rebound %,2024-25,1,0.36
1,Offensive Rebound %,2025-26,1,0.39
2,2nd Chance Points/G,2024-25,1,18.10
3,2nd Chance Points/G,2025-26,3,17.78
4,Paint Points/G,2024-25,7,51.49
5,Paint Points/G,2025-26,5,53.12
6,3PA per Game,2024-25,20,35.83
7,3PA per Game,2025-26,28,31.46
8,3PM per Game,2024-25,22,12.66
9,3PM per Game,2025-26,25,11.46


In [334]:
# ============================================================
# Section 1 assets: 2 tables + 3-card callout
# Each saved as a separate HTML file
# ============================================================

def fmt_value(metric, value):
    if "%" in metric and value < 1:
        return f"{value*100:.1f}%"
    elif "Points" in metric or "3PA" in metric or "3PM" in metric:
        return f"{value:.1f}"
    else:
        return f"{value:.2f}"

def fmt_rank(rank):
    suffix = "th"
    if rank % 10 == 1 and rank != 11: suffix = "st"
    elif rank % 10 == 2 and rank != 12: suffix = "nd"
    elif rank % 10 == 3 and rank != 13: suffix = "rd"
    return f"{rank}{suffix}"

def build_table_rows(metric_list):
    rows = ""
    for metric in metric_list:
        row = playstyle_df[playstyle_df["Metric"] == metric]
        r24 = row[row["Season"] == "2024-25"].iloc[0]
        r25 = row[row["Season"] == "2025-26"].iloc[0]
        v24 = f"{fmt_value(metric, r24['Value'])} ({fmt_rank(r24['Rank'])})"
        v25 = f"{fmt_value(metric, r25['Value'])} ({fmt_rank(r25['Rank'])})"
        rows += f"""
    <tr>
      <td class="metric">{metric}</td>
      <td class="value">{v24}</td>
      <td class="value">{v25}</td>
    </tr>"""
    return rows

# ---------- Shared CSS ----------
SHARED_CSS = """
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    font-family: 'Inter', sans-serif;
    background: #FAFAF7;
    color: #1A1A1A;
    padding: 60px 40px;
    line-height: 1.5;
  }
  .wrapper { max-width: 720px; margin: 0 auto; }
  .table-label {
    font-family: 'JetBrains Mono', monospace;
    font-size: 11px;
    letter-spacing: 0.15em;
    text-transform: uppercase;
    color: #CE1141;
    margin-bottom: 8px;
  }
  h2 {
    font-family: 'Fraunces', Georgia, serif;
    font-size: 26px;
    font-weight: 600;
    letter-spacing: -0.015em;
    margin-bottom: 8px;
  }
  .table-subtitle {
    color: #6B7280;
    font-size: 15px;
    margin-bottom: 20px;
  }
  table {
    width: 100%;
    border-collapse: collapse;
    background: white;
    box-shadow: 0 1px 3px rgba(0,0,0,0.04), 0 1px 2px rgba(0,0,0,0.06);
    border-radius: 6px;
    overflow: hidden;
  }
  thead { background: #1A1A1A; color: white; }
  thead th {
    padding: 14px 24px;
    font-weight: 600;
    font-size: 12px;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    text-align: right;
  }
  thead th:first-child { text-align: left; }
  tbody td {
    padding: 14px 24px;
    font-size: 15px;
    border-bottom: 1px solid #F0EDE5;
    text-align: right;
  }
  tbody tr:last-child td { border-bottom: none; }
  tbody tr:hover { background: #FAFAF7; }
  td.metric { text-align: left; font-weight: 500; }
  td.value {
    font-family: 'JetBrains Mono', 'SF Mono', monospace;
    font-feature-settings: "tnum";
    color: #2D2D2D;
  }
  /* Callout cards */
  .callout-grid {
    display: grid;
    grid-template-columns: 1fr 1fr 1fr;
    gap: 16px;
    margin-top: 20px;
  }
  .card {
    background: white;
    border: 1px solid #E5E2DA;
    border-left: 3px solid #CE1141;
    padding: 24px 20px;
    border-radius: 4px;
  }
  .card .big {
    font-family: 'JetBrains Mono', 'SF Mono', monospace;
    font-feature-settings: "tnum";
    font-size: 36px;
    font-weight: 700;
    line-height: 1;
    color: #1A1A1A;
    margin-bottom: 10px;
  }
  .card .label {
    font-family: 'Inter', sans-serif;
    font-size: 13px;
    color: #6B7280;
    margin-bottom: 12px;
    line-height: 1.4;
    font-weight: 500;
  }
  .card .sub {
    font-family: 'JetBrains Mono', monospace;
    font-size: 12px;
    color: #2D2D2D;
  }
  .callout-footer {
    margin-top: 14px;
    font-size: 13px;
    color: #6B7280;
    font-style: italic;
  }
  .table-footnote {
    margin-top: 12px;
    font-size: 12px;
    color: #9CA3AF;
    font-style: italic;
  }
  .player-prose {
    margin-top: 28px;
    padding: 20px 24px;
    background: white;
    border-left: 3px solid #CE1141;
    border-radius: 4px;
  }
  .player-prose .player-name {
    font-family: 'Fraunces', Georgia, serif;
    font-size: 18px;
    font-weight: 600;
    margin-bottom: 8px;
    color: #1A1A1A;
  }
  .player-prose p {
    font-size: 14px;
    line-height: 1.65;
    color: #374151;
  }
"""

FONTS_LINK = """<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@500;700&display=swap" rel="stylesheet">"""


def wrap_html(title, body_content):
    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>{title}</title>
{FONTS_LINK}
<style>{SHARED_CSS}</style>
</head>
<body>
<div class="wrapper">
{body_content}
</div>
</body>
</html>"""


# ============================================================
# File 1: Interior Identity table
# ============================================================
table1_rows = build_table_rows([
    "Offensive Rebound %",
    "2nd Chance Points/G",
    "Paint Points/G",
])

table1_body = f"""
  <div class="table-label">Playstyle, Section 01</div>
  <h2>Interior Identity</h2>
  <p class="table-subtitle">Houston led the league in offensive rebounding and stayed elite at generating second chance opportunities and points in the paint across both seasons.</p>
  <table>
    <thead>
      <tr>
        <th>Metric</th>
        <th>2024,25</th>
        <th>2025,26</th>
      </tr>
    </thead>
    <tbody>{table1_rows}
    </tbody>
  </table>
"""
# Fix the season labels (replace the comma I just added with en-dash, since dashes between years are conventional)
table1_body = table1_body.replace("2024,25", "2024–25").replace("2025,26", "2025–26")

with open(ASSETS_DIR / "01a_interior_identity.html", "w") as f:
    f.write(wrap_html("Interior Identity", table1_body))


# ============================================================
# File 2: The 3-Point Problem table
# ============================================================
table2_rows = build_table_rows([
    "3PA per Game",
    "3PM per Game",
    "3PT %",
    "% of FGA from 3PT",
])

table2_body = f"""
  <div class="table-label">Playstyle, Section 01</div>
  <h2>The 3-Point Problem</h2>
  <p class="table-subtitle">Houston ranked among the bottom of the NBA in both 3-point volume and accuracy, and the trend got worse in 2025–26 after losing Green, Brooks, and VanVleet.</p>
  <table>
    <thead>
      <tr>
        <th>Metric</th>
        <th>2024–25</th>
        <th>2025–26</th>
      </tr>
    </thead>
    <tbody>{table2_rows}
    </tbody>
  </table>
"""

with open(ASSETS_DIR / "01b_three_point_problem.html", "w") as f:
    f.write(wrap_html("The 3-Point Problem", table2_body))


# ============================================================
# File 3: When the 3-ball didn't fall callout
# ============================================================
hou = league_gamelogs_2526[league_gamelogs_2526["TEAM_ABBR"] == "HOU"]
hou_low = hou[hou["FG3M"] < 10]
hou_high = hou[hou["FG3M"] >= 10]

n_low = len(hou_low)
pct_low = n_low / len(hou) * 100
low_w = (hou_low["WL"] == "W").sum()
low_l = (hou_low["WL"] == "L").sum()
high_w = (hou_high["WL"] == "W").sum()
high_l = (hou_high["WL"] == "L").sum()
low_wpct = low_w / (low_w + low_l) if (low_w + low_l) > 0 else 0
high_wpct = high_w / (high_w + high_l) if (high_w + high_l) > 0 else 0

league_avg = (
    league_gamelogs_2526.groupby("TEAM_ABBR")["FG3M"]
    .apply(lambda s: (s < 10).sum())
    .mean()
)

callout_body = f"""
  <div class="table-label">Playstyle, Section 01</div>
  <h2>When the 3-Ball Didn't Fall</h2>
  <p class="table-subtitle">Houston made under 10 threes in nearly a third of their games in 2025–26, and their record reflected it.</p>
  <div class="callout-grid">
    <div class="card">
      <div class="label">Games under 10 3PM made</div>
      <div class="big">{n_low}</div>
      <div class="sub">{pct_low:.0f}% of regular season</div>
    </div>
    <div class="card">
      <div class="label">Record when under 10 3PM</div>
      <div class="big">{low_w}–{low_l}</div>
      <div class="sub">.{int(round(low_wpct*1000)):03d} win %</div>
    </div>
    <div class="card">
      <div class="label">Record when 10+ 3PM made</div>
      <div class="big">{high_w}–{high_l}</div>
      <div class="sub">.{int(round(high_wpct*1000)):03d} win %</div>
    </div>
  </div>
  <p class="callout-footer">League average: {league_avg:.1f} games per team under 10 3PM in 2025–26.</p>
"""

with open(ASSETS_DIR / "01c_three_ball_didnt_fall.html", "w") as f:
    f.write(wrap_html("When the 3-Ball Didn't Fall", callout_body))


print("Saved:")
print(f"  {ASSETS_DIR / '01a_interior_identity.html'}")
print(f"  {ASSETS_DIR / '01b_three_point_problem.html'}")
print(f"  {ASSETS_DIR / '01c_three_ball_didnt_fall.html'}")

Saved:
  /Users/stevenyan/Desktop/rockets_season_review/report_assets/01a_interior_identity.html
  /Users/stevenyan/Desktop/rockets_season_review/report_assets/01b_three_point_problem.html
  /Users/stevenyan/Desktop/rockets_season_review/report_assets/01c_three_ball_didnt_fall.html


## Build shot chart heatmaps

In [47]:
## shot chart
shots = pd.read_parquet(PROCESSED_DIR / "rockets_shot_charts.parquet")
print(f"Total shots: {len(shots):,}")
print(f"\nColumns: {shots.columns.tolist()}")
print(f"\nShots by player (2025-26 regular season):")
mask = (shots["SEASON_LABEL"] == "2025-26") & (shots["SEASON_TYPE"] == "Regular Season")
print(shots[mask].groupby("PLAYER_NAME").size().sort_values(ascending=False).head(15))

Total shots: 20,954

Columns: ['GRID_TYPE', 'GAME_ID', 'GAME_EVENT_ID', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_NAME', 'PERIOD', 'MINUTES_REMAINING', 'SECONDS_REMAINING', 'EVENT_TYPE', 'ACTION_TYPE', 'SHOT_TYPE', 'SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA', 'SHOT_ZONE_RANGE', 'SHOT_DISTANCE', 'LOC_X', 'LOC_Y', 'SHOT_ATTEMPTED_FLAG', 'SHOT_MADE_FLAG', 'GAME_DATE', 'HTM', 'VTM', 'SEASON_LABEL', 'SEASON_TYPE']

Shots by player (2025-26 regular season):
PLAYER_NAME
Kevin Durant           1376
Alperen Sengun         1123
Amen Thompson          1044
Jabari Smith Jr.        975
Dillon Brooks           959
Reed Sheppard           946
Tari Eason              580
Jock Landale            548
Jalen Green             512
Josh Okogie             294
Aaron Holiday           252
Clint Capela            227
Cam Whitmore            169
Steven Adams            139
Dorian Finney-Smith     132
dtype: int64


### Heatmap utilities + trio breakdown chart

The next few cells build the shared heatmap renderer used across all player heatmaps, and the trio breakdown stacked bars (KD / Sheppard / Smith Jr. vs. all other Rockets).

In [327]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Trio 3PT breakdown data (Reed > Jabari > KD by attempts)
# ============================================================
TRIO_BREAKDOWN = [
    {"name": "Sheppard",   "color": "#CE1141", "attempts": 576, "makes": 227, "pct_3pa": 22.3, "pct_3pm": 24.1},
    {"name": "Smith Jr.",  "color": "#E8869B", "attempts": 488, "makes": 177, "pct_3pa": 18.9, "pct_3pm": 18.8},
    {"name": "Durant",     "color": "#F2B5C2", "attempts": 450, "makes": 186, "pct_3pa": 17.4, "pct_3pm": 19.8},
]
TRIO_OTHER = {"pct_3pa": 41.3, "pct_3pm": 37.2, "attempts": 1066, "makes": 350}
TRIO_TOTAL_3PA_PCT = sum(p["pct_3pa"] for p in TRIO_BREAKDOWN)
TRIO_TOTAL_3PM_PCT = sum(p["pct_3pm"] for p in TRIO_BREAKDOWN)


def build_trio_breakdown_html():
    """Build the stacked-bars HTML block for the Reliable Shooters page."""
    def segments(metric_key):
        segs = ""
        for p in TRIO_BREAKDOWN:
            pct = p[metric_key]
            segs += f'<div style="width: {pct:.2f}%; background: {p["color"]}; display: flex; align-items: center; justify-content: center; color: white; font-size: 12px; font-weight: 600;">{pct:.1f}%</div>'
        other_pct = TRIO_OTHER[metric_key]
        segs += f'<div style="width: {other_pct:.2f}%; background: #E5E2DA; display: flex; align-items: center; justify-content: center; color: #6B7280; font-size: 12px; font-weight: 500;">{other_pct:.1f}%</div>'
        return segs
    
    def legend(count_key):
        items = ""
        for p in TRIO_BREAKDOWN:
            items += f'<div style="display: flex; align-items: center; gap: 6px;"><span style="width: 11px; height: 11px; background: {p["color"]}; border-radius: 2px;"></span><span>{p["name"]} ({p[count_key]:,})</span></div>'
        items += f'<div style="display: flex; align-items: center; gap: 6px;"><span style="width: 11px; height: 11px; background: #E5E2DA; border-radius: 2px;"></span><span>All other Rockets ({TRIO_OTHER[count_key]:,})</span></div>'
        return items
    
    return f"""
<div style="margin-top: 36px; padding: 24px; background: white; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.04);">
  <h3 style="font-family: 'Fraunces', Georgia, serif; font-size: 22px; font-weight: 600; letter-spacing: -0.015em; color: #1A1A1A; margin-bottom: 6px;">Where Houston's Threes Came From</h3>
  <p style="font-size: 14px; color: #6B7280; margin-bottom: 24px;">Durant, Sheppard, Smith Jr. vs. all other Rockets.</p>
  <div style="margin-bottom: 28px;">
    <div style="display: flex; align-items: baseline; gap: 12px; margin-bottom: 14px;">
      <div style="font-family: 'Fraunces', Georgia, serif; font-size: 38px; font-weight: 600; color: #CE1141; line-height: 1;">{TRIO_TOTAL_3PA_PCT:.1f}%</div>
      <div style="font-family: 'JetBrains Mono', monospace; font-size: 11px; letter-spacing: 0.12em; color: #6B7280; text-transform: uppercase;">of Houston's 3-point attempts</div>
    </div>
    <div style="display: flex; height: 32px; border-radius: 6px; overflow: hidden;">{segments('pct_3pa')}</div>
    <div style="display: flex; gap: 18px; margin-top: 12px; font-size: 13px; color: #374151; flex-wrap: wrap;">{legend('attempts')}</div>
  </div>
  <div>
    <div style="display: flex; align-items: baseline; gap: 12px; margin-bottom: 14px;">
      <div style="font-family: 'Fraunces', Georgia, serif; font-size: 38px; font-weight: 600; color: #CE1141; line-height: 1;">{TRIO_TOTAL_3PM_PCT:.1f}%</div>
      <div style="font-family: 'JetBrains Mono', monospace; font-size: 11px; letter-spacing: 0.12em; color: #6B7280; text-transform: uppercase;">of Houston's 3-point makes</div>
    </div>
    <div style="display: flex; height: 32px; border-radius: 6px; overflow: hidden;">{segments('pct_3pm')}</div>
    <div style="display: flex; gap: 18px; margin-top: 12px; font-size: 13px; color: #374151; flex-wrap: wrap;">{legend('makes')}</div>
  </div>
</div>
"""



PROSE = {
    "Amen Thompson": "Amen Thompson's perimeter shooting was a known weakness coming into the season, and it was expected — just 21.7% from three. He produced almost entirely inside: 69.6% of his points came from the paint. Opposing defenses regularly stationed their centers on Amen, daring him to shoot from the perimeter while collapsing the paint and disrupting Houston's interior offense.",
    "Alperen Sengun": "Alperen Sengun isn't a perimeter threat by design. He's the team's primary playmaking hub, operating from the post and high elbow rather than spacing the floor himself. His 30.5% from three on low volume reflects his role, not a flaw. The structural issue isn't Sengun's shooting; it's that with him and Amen sharing the floor, two of Houston's five starters offered no perimeter threat at all.",
}

RELIABLE_PROSE = "These three players carried Houston's perimeter shooting in 2025–26. Each shot above the league average (36.0%) from three: Durant at 41.3%, Sheppard at 39.4%, and Smith Jr. at 36.3%. Together, they accounted for nearly two-thirds of Houston's 3-point production: 58.6% of the team's 3-point attempts and 62.7% of the team's makes. They are the only real threats Houston have from beyond the arc."

# compute player (3p%) label
def compute_player_3pt_pct(player_name):
    p_shots = shots[
        (shots["PLAYER_NAME"] == player_name) &
        (shots["SEASON_LABEL"] == "2025-26") &
        (shots["SEASON_TYPE"] == "Regular Season") &
        (shots["TEAM_ID"] == ROCKETS_ID) &
        (shots["SHOT_TYPE"] == "3PT Field Goal")
    ]
    if len(p_shots) == 0:
        return 0
    return p_shots["SHOT_MADE_FLAG"].sum() / len(p_shots) * 100


def player_subtitle(player_name):
    pct = compute_player_3pt_pct(player_name)
    return f"<b>{player_name}</b><br><span style='font-size:13px;color:#6B7280;font-weight:normal'><b style='color:#1A1A1A;font-size:15px'>{pct:.1f}%</b> from three</span>"

# Player sets (Eason and Okogie removed — 01e is built in its own cell)
PLAYER_SETS = [
    {
        "label": "Starters",
        "players": ["Amen Thompson", "Alperen Sengun"],
        "title": "Three-Point Shooting Woes: The Starters",
        "subtitle": "Defenses don't have to respect Houston's young stars from three.",
        "filename": "01d_heatmap_starters.html",
    },
    {
        "label": "Reliable Shooters",
        "players": ["Kevin Durant", "Jabari Smith Jr.", "Reed Sheppard"],
        "title": "Houston's Only Reliable Shooters",
        "subtitle": "Three players carried Houston's perimeter shooting in 2025–26.",
        "filename": "01f_heatmap_reliable_shooters.html",
    },
]


def draw_court_lines(fig, row, col, line_color="#1A1A1A"):
    lw = 1.3
    arc_t = np.linspace(np.pi * 0.08, np.pi * 0.92, 100)
    fig.add_trace(go.Scatter(
        x=237.5 * np.cos(arc_t), y=237.5 * np.sin(arc_t), mode='lines',
        line=dict(color=line_color, width=lw),
        showlegend=False, hoverinfo='skip'
    ), row=row, col=col)
    for x in [-220, 220]:
        fig.add_trace(go.Scatter(x=[x, x], y=[-47, 92], mode='lines',
                                 line=dict(color=line_color, width=lw),
                                 showlegend=False, hoverinfo='skip'), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=[-80, 80, 80, -80, -80], y=[-47, -47, 143, 143, -47],
        mode='lines', line=dict(color=line_color, width=lw),
        showlegend=False, hoverinfo='skip'
    ), row=row, col=col)
    ft_t = np.linspace(0, 2*np.pi, 60)
    fig.add_trace(go.Scatter(
        x=60*np.cos(ft_t), y=60*np.sin(ft_t)+143, mode='lines',
        line=dict(color=line_color, width=lw),
        showlegend=False, hoverinfo='skip'
    ), row=row, col=col)
    rim_t = np.linspace(0, 2*np.pi, 30)
    fig.add_trace(go.Scatter(
        x=7.5*np.cos(rim_t), y=7.5*np.sin(rim_t), mode='lines',
        line=dict(color="#CE1141", width=lw + 0.3),
        showlegend=False, hoverinfo='skip'
    ), row=row, col=col)
    fig.add_trace(go.Scatter(x=[-250, 250], y=[-47, -47], mode='lines',
                             line=dict(color=line_color, width=lw),
                             showlegend=False, hoverinfo='skip'), row=row, col=col)


def compute_zone_stats(player_shots):
    """Compute attempts, makes, FG% per zone."""
    stats = {}
    for (zone, area), group in player_shots.groupby(["SHOT_ZONE_BASIC", "SHOT_ZONE_AREA"]):
        if zone == "Backcourt":
            continue
        attempts = len(group)
        made = int(group["SHOT_MADE_FLAG"].sum())
        stats[(zone, area)] = {
            "attempts": attempts,
            "made": made,
            "fg_pct": made / attempts if attempts > 0 else 0,
            "x": group["LOC_X"].mean(),
            "y": group["LOC_Y"].mean(),
        }
    return stats


def build_heatmap_html(player_set):
    """Build one HTML file for a given set of players."""
    players = player_set["players"]
    
    # Filter shots for this set
    set_shots = shots[
        (shots["PLAYER_NAME"].isin(players)) &
        (shots["SEASON_LABEL"] == "2025-26") &
        (shots["SEASON_TYPE"] == "Regular Season") &
        (shots["TEAM_ID"] == ROCKETS_ID)
    ].copy()
    
    all_stats = {p: compute_zone_stats(set_shots[set_shots["PLAYER_NAME"] == p]) for p in players}
    
    max_attempts = max(
        (s["attempts"] for player_stats in all_stats.values() for s in player_stats.values()),
        default=1
    )
    
    fig = make_subplots(
        rows=1, cols=len(players),
        subplot_titles=[player_subtitle(p) for p in players],
        horizontal_spacing=0.04,
    )
    
    for idx, player in enumerate(players):
        col = idx + 1
        zone_stats = all_stats[player]
        draw_court_lines(fig, row=1, col=col)
        for (zone, area), s in zone_stats.items():
            size = 20 + (s["attempts"] / max_attempts) * 50
            intensity = s["attempts"] / max_attempts
            color = f"rgba({int(206 - intensity*40)}, {int(17 + (1-intensity)*100)}, {int(65 + (1-intensity)*70)}, 0.78)"
            fig.add_trace(go.Scatter(
                x=[s["x"]], y=[s["y"]],
                mode='markers+text',
                marker=dict(
                    size=size,
                    color=color,
                    line=dict(color='white', width=2),
                ),
                text=[f"{s['fg_pct']*100:.0f}%"],
                textfont=dict(size=10.5, color='white', family='Inter'),
                textposition='middle center',
                hovertemplate=(
                    f"<b>{zone} ({area})</b><br>"
                    f"Attempts: {s['attempts']}<br>"
                    f"Made: {s['made']}<br>"
                    f"FG%: {s['fg_pct']*100:.1f}%<extra></extra>"
                ),
                showlegend=False,
            ), row=1, col=col)
    
    fig.update_layout(
        height=520,
        paper_bgcolor='white',
        plot_bgcolor='white',
        font=dict(family="Inter, sans-serif", color="#1A1A1A"),
        margin=dict(l=10, r=10, t=50, b=10),
        showlegend=False,
    )
    
    for ann in fig['layout']['annotations']:
        ann['font'] = dict(color='#1A1A1A', family='Inter, sans-serif', size=14)
    
    for i in range(1, len(players) + 1):
        fig.update_xaxes(range=[-260, 260], visible=False, 
                         scaleanchor=f"y{i}", scaleratio=1, row=1, col=i)
        fig.update_yaxes(range=[-60, 290], visible=False, row=1, col=i)
    
    plotly_html = fig.to_html(include_plotlyjs='cdn', full_html=True,
                              config={'displayModeBar': False})
    
    wrapper = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>__TITLE__</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@500&display=swap" rel="stylesheet">
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    font-family: 'Inter', sans-serif;
    background: #FAFAF7;
    color: #1A1A1A;
    padding: 60px 40px;
  }
  .wrapper { max-width: 1100px; margin: 0 auto; }
  .table-label {
    font-family: 'JetBrains Mono', monospace;
    font-size: 11px;
    letter-spacing: 0.15em;
    text-transform: uppercase;
    color: #CE1141;
    margin-bottom: 8px;
  }
  h2 {
    font-family: 'Fraunces', Georgia, serif;
    font-size: 26px;
    font-weight: 600;
    letter-spacing: -0.015em;
    margin-bottom: 8px;
  }
  .subtitle {
    color: #6B7280;
    font-size: 15px;
    margin-bottom: 24px;
  }
  .chart-frame {
    background: white;
    border-radius: 8px;
    overflow: hidden;
    box-shadow: 0 1px 3px rgba(0,0,0,0.04), 0 1px 2px rgba(0,0,0,0.06);
    padding: 8px;
  }
  .player-prose { margin-top: 28px; padding: 20px 24px; background: white; border-left: 3px solid #CE1141; border-radius: 4px; }
  .player-prose .player-name { font-family: 'Fraunces', Georgia, serif; font-size: 18px; font-weight: 600; margin-bottom: 8px; color: #1A1A1A; }
  .player-prose p { font-size: 14px; line-height: 1.65; color: #374151; }
  .group-prose { margin-top: 28px; padding: 20px 24px; background: white; border-left: 3px solid #CE1141; border-radius: 4px; font-size: 14px; line-height: 1.65; color: #374151; }
</style>
</head>
<body>
<div class="wrapper">
  <h2>__TITLE__</h2>
  <p class="subtitle">__SUBTITLE__</p>
  <div class="chart-frame">
__CHART__
  </div>
__PROSE__
</div>
</body>
</html>"""
    
    import re
    chart_div = re.search(r'<body>(.*?)</body>', plotly_html, re.DOTALL).group(1)
    
    # Build prose blocks for this set
    if player_set["label"] == "Reliable Shooters":
        # Bars chart first, then prose
        prose_blocks = build_trio_breakdown_html()
        prose_blocks += f'<div class="player-prose"><div class="player-name">Kevin Durant, Jabari Smith Jr., Reed Sheppard</div><p>{RELIABLE_PROSE}</p></div>'
    else:
        prose_blocks = ""
        for p in player_set["players"]:
            if p in PROSE:
                prose_blocks += f'<div class="player-prose"><div class="player-name">{p}</div><p>{PROSE[p]}</p></div>'
    
    final = (
        wrapper
        .replace("__TITLE__", player_set["title"])
        .replace("__SUBTITLE__", player_set["subtitle"])
        .replace("__CHART__", chart_div)
        .replace("__PROSE__", prose_blocks)
    )
    
    output_path = ASSETS_DIR / player_set["filename"]
    with open(output_path, "w") as f:
        f.write(final)
    
    return output_path, fig


# Build all
for player_set in PLAYER_SETS:
    path, fig = build_heatmap_html(player_set)
    print(f"Saved: {path}")
    fig.show()

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/01d_heatmap_starters.html


Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/01f_heatmap_reliable_shooters.html


In [302]:
# Per-player 3PA and 3PM for the trio + team totals
KD_ID = 201142
JABARI_ID = 1631095
REED_ID = 1642263

trio = {
    "Kevin Durant": KD_ID,
    "Jabari Smith Jr.": JABARI_ID,
    "Reed Sheppard": REED_ID,
}

team_3pa = team_gl_2526["FG3A"].sum()
team_3pm = team_gl_2526["FG3M"].sum()

print(f"Team totals: {team_3pm}/{team_3pa} = {team_3pm/team_3pa*100:.1f}%\n")

trio_3pa_total = 0
trio_3pm_total = 0

for name, pid in trio.items():
    p = player_gamelogs_rockets[
        (player_gamelogs_rockets["Player_ID"] == pid) &
        (player_gamelogs_rockets["SEASON"] == "2025-26") &
        (player_gamelogs_rockets["SEASON_TYPE"] == "Regular Season")
    ]
    p_3pa = p["FG3A"].sum()
    p_3pm = p["FG3M"].sum()
    trio_3pa_total += p_3pa
    trio_3pm_total += p_3pm
    
    pct_of_team_3pa = p_3pa / team_3pa * 100
    pct_of_team_3pm = p_3pm / team_3pm * 100
    
    print(f"{name}:")
    print(f"  3PM/3PA: {p_3pm}/{p_3pa} = {p_3pm/p_3pa*100:.1f}%")
    print(f"  Share of team 3PA: {pct_of_team_3pa:.1f}%")
    print(f"  Share of team 3PM: {pct_of_team_3pm:.1f}%")
    print()

other_3pa = team_3pa - trio_3pa_total
other_3pm = team_3pm - trio_3pm_total

print(f"All other Rockets:")
print(f"  3PM/3PA: {other_3pm}/{other_3pa} = {other_3pm/max(other_3pa,1)*100:.1f}%")
print(f"  Share of team 3PA: {other_3pa/team_3pa*100:.1f}%")
print(f"  Share of team 3PM: {other_3pm/team_3pm*100:.1f}%")

print(f"\nTrio combined: {trio_3pa_total/team_3pa*100:.1f}% of attempts, {trio_3pm_total/team_3pm*100:.1f}% of makes")

Team totals: 940/2580 = 36.4%

Kevin Durant:
  3PM/3PA: 186/450 = 41.3%
  Share of team 3PA: 17.4%
  Share of team 3PM: 19.8%

Jabari Smith Jr.:
  3PM/3PA: 177/488 = 36.3%
  Share of team 3PA: 18.9%
  Share of team 3PM: 18.8%

Reed Sheppard:
  3PM/3PA: 227/576 = 39.4%
  Share of team 3PA: 22.3%
  Share of team 3PM: 24.1%

All other Rockets:
  3PM/3PA: 350/1066 = 32.8%
  Share of team 3PA: 41.3%
  Share of team 3PM: 37.2%

Trio combined: 58.7% of attempts, 62.8% of makes


In [318]:
# ============================================================
# Build two test versions of the trio bars for comparison
# ============================================================

# Data
trio_data = [
    {"name": "Sheppard",    "pct_3pa": 22.3, "pct_3pm": 24.1, "color": "#CE1141"},
    {"name": "Smith Jr.",   "pct_3pa": 18.9, "pct_3pm": 18.8, "color": "#E8869B"},
    {"name": "Durant",      "pct_3pa": 17.4, "pct_3pm": 19.8, "color": "#F2B5C2"}
]
other_3pa = 41.3
other_3pm = 37.2
trio_total_3pa = sum(p["pct_3pa"] for p in trio_data)
trio_total_3pm = sum(p["pct_3pm"] for p in trio_data)

# Numerical breakdowns for legend
trio_attempts = {"Sheppard": 576, "Smith Jr.": 488, "Durant": 450}
trio_makes = {"Sheppard": 227, "Smith Jr.": 177, "Durant": 186}
other_attempts = 1066
other_makes = 350

# Common segments definition
def build_segments(metric):
    """Build segment HTML for a given metric: '3pa' or '3pm'."""
    segments = ""
    for p in trio_data:
        pct = p["pct_3pa"] if metric == "3pa" else p["pct_3pm"]
        segments += f'<div style="width: {pct:.2f}%; background: {p["color"]}; display: flex; align-items: center; justify-content: center; color: white; font-size: 12px; font-weight: 600;">{pct:.1f}%</div>'
    other_pct = other_3pa if metric == "3pa" else other_3pm
    segments += f'<div style="width: {other_pct:.2f}%; background: #E5E2DA; display: flex; align-items: center; justify-content: center; color: #6B7280; font-size: 12px; font-weight: 500;">{other_pct:.1f}%</div>'
    return segments


def build_legend(metric):
    """Build legend HTML for a given metric."""
    legend_items = ""
    for p in trio_data:
        n = trio_attempts[p["name"]] if metric == "3pa" else trio_makes[p["name"]]
        legend_items += f'<div style="display: flex; align-items: center; gap: 6px;"><span style="width: 11px; height: 11px; background: {p["color"]}; border-radius: 2px;"></span><span>{p["name"]} ({n:,})</span></div>'
    other_n = other_attempts if metric == "3pa" else other_makes
    legend_items += f'<div style="display: flex; align-items: center; gap: 6px;"><span style="width: 11px; height: 11px; background: #E5E2DA; border-radius: 2px;"></span><span>All other Rockets ({other_n:,})</span></div>'
    return legend_items


# ============================================================
# LAYOUT 1: Single panel, both bars stacked vertically
# ============================================================
layout1_html = f"""
<div style="margin-top: 36px; padding: 24px; background: white; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.04);">
  
  <!-- Block 1: 3PA -->
  <div style="margin-bottom: 28px;">
    <div style="display: flex; align-items: baseline; gap: 12px; margin-bottom: 14px;">
      <div style="font-family: 'Fraunces', Georgia, serif; font-size: 38px; font-weight: 600; color: #CE1141; line-height: 1;">{trio_total_3pa:.1f}%</div>
      <div style="font-family: 'JetBrains Mono', monospace; font-size: 11px; letter-spacing: 0.12em; color: #6B7280; text-transform: uppercase;">of Houston's 3-point attempts</div>
    </div>
    <div style="display: flex; height: 32px; border-radius: 6px; overflow: hidden;">
      {build_segments('3pa')}
    </div>
    <div style="display: flex; gap: 18px; margin-top: 12px; font-size: 13px; color: #374151; flex-wrap: wrap;">
      {build_legend('3pa')}
    </div>
  </div>
  
  <!-- Block 2: 3PM -->
  <div>
    <div style="display: flex; align-items: baseline; gap: 12px; margin-bottom: 14px;">
      <div style="font-family: 'Fraunces', Georgia, serif; font-size: 38px; font-weight: 600; color: #CE1141; line-height: 1;">{trio_total_3pm:.1f}%</div>
      <div style="font-family: 'JetBrains Mono', monospace; font-size: 11px; letter-spacing: 0.12em; color: #6B7280; text-transform: uppercase;">of Houston's 3-point makes</div>
    </div>
    <div style="display: flex; height: 32px; border-radius: 6px; overflow: hidden;">
      {build_segments('3pm')}
    </div>
    <div style="display: flex; gap: 18px; margin-top: 12px; font-size: 13px; color: #374151; flex-wrap: wrap;">
      {build_legend('3pm')}
    </div>
  </div>
  
</div>
"""

# ============================================================
# LAYOUT 2: Two side-by-side cards
# ============================================================
layout2_html = f"""
<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-top: 36px;">
  
  <!-- Card 1: 3PA -->
  <div style="background: white; border-radius: 8px; padding: 24px; border-left: 4px solid #CE1141; box-shadow: 0 1px 3px rgba(0,0,0,0.04);">
    <div style="font-family: 'Inter', sans-serif; font-size: 13px; color: #6B7280; margin-bottom: 12px; font-weight: 500;">3-Point Attempts</div>
    <div style="font-family: 'JetBrains Mono', monospace; font-size: 42px; font-weight: 700; color: #1A1A1A; line-height: 1; margin-bottom: 16px;">{trio_total_3pa:.1f}%</div>
    <div style="display: flex; height: 28px; border-radius: 4px; overflow: hidden; margin-bottom: 12px;">
      {build_segments('3pa')}
    </div>
    <div style="display: flex; flex-direction: column; gap: 6px; font-size: 12px; color: #374151;">
      {build_legend('3pa')}
    </div>
  </div>
  
  <!-- Card 2: 3PM -->
  <div style="background: white; border-radius: 8px; padding: 24px; border-left: 4px solid #CE1141; box-shadow: 0 1px 3px rgba(0,0,0,0.04);">
    <div style="font-family: 'Inter', sans-serif; font-size: 13px; color: #6B7280; margin-bottom: 12px; font-weight: 500;">3-Point Makes</div>
    <div style="font-family: 'JetBrains Mono', monospace; font-size: 42px; font-weight: 700; color: #1A1A1A; line-height: 1; margin-bottom: 16px;">{trio_total_3pm:.1f}%</div>
    <div style="display: flex; height: 28px; border-radius: 4px; overflow: hidden; margin-bottom: 12px;">
      {build_segments('3pm')}
    </div>
    <div style="display: flex; flex-direction: column; gap: 6px; font-size: 12px; color: #374151;">
      {build_legend('3pm')}
    </div>
  </div>
  
</div>
"""


# Wrap both layouts in a preview page
wrapper_template = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>__TITLE__</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@500;700&display=swap" rel="stylesheet">
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: 'Inter', sans-serif; background: #FAFAF7; color: #1A1A1A; padding: 60px 40px; }
  .wrapper { max-width: 1100px; margin: 0 auto; }
  h2 { font-family: 'Fraunces', Georgia, serif; font-size: 26px; font-weight: 600; margin-bottom: 8px; }
  .subtitle { color: #6B7280; font-size: 15px; margin-bottom: 24px; }
</style>
</head>
<body>
<div class="wrapper">
  <h2>__TITLE__</h2>
  <p class="subtitle">Durant, Sheppard, Smith Jr. vs. all other Rockets.</p>
  __CONTENT__
</div>
</body>
</html>"""

# Save layout 1 preview
out1 = ASSETS_DIR / "preview_layout1_stacked.html"
out1.write_text(
    wrapper_template
    .replace("__TITLE__", "Three Point Shooting Breakdown")
    .replace("__CONTENT__", layout1_html)
)
print(f"Saved: {out1.name}")

# Save layout 2 preview
out2 = ASSETS_DIR / "preview_layout2_cards.html"
out2.write_text(
    wrapper_template
    .replace("__TITLE__", "Layout 2: Side-by-Side Cards")
    .replace("__CONTENT__", layout2_html)
)
print(f"Saved: {out2.name}")

Saved: preview_layout1_stacked.html
Saved: preview_layout2_cards.html


### Supporting cast heatmaps: DFS + Eason (pre/post All-Star break)

Eason's shooting cratered after the All-Star break (46.0% → 21.8% from three). This is rendered as a split chart inside `01e`.

In [ ]:
# ============================================================
# Build 01e — Supporting Cast: DFS + Eason pre/post-ASB split
# ============================================================
import pandas as pd

ASB_LAST = pd.Timestamp("2026-02-11")
ASB_FIRST = pd.Timestamp("2026-02-19")

DFS_PROSE = "Dorian Finney-Smith didn't debut until Christmas Day 2025, having missed the first two months recovering from an ongoing ankle issue. He played in just 37 games and shot only 27.0% from three on 100 attempts. Signed as a 3-and-D replacement for Brooks's defensive presence and perimeter shooting but ended up a huge disappointment."

EASON_PROSE = "Tari Eason started the season historically hot. Through his first 11 games, he led the NBA in 3-point shooting at 50.9%. He finished with 46.0% on 150 attempts pre-All star break. After the All star break, his 3-point percentage plummeted to just 21.8% on 110 attempts. He had an extremely cold shooting stretch in March where he missed 21 consecutive 3-point attempts across seven games. He finished the season at 35.8%."


def build_single_player_chart(player_name):
    """Build a 1-panel shot chart for a single player."""
    p_shots = shots[
        (shots["PLAYER_NAME"] == player_name) &
        (shots["SEASON_LABEL"] == "2025-26") &
        (shots["SEASON_TYPE"] == "Regular Season") &
        (shots["TEAM_ID"] == ROCKETS_ID)
    ].copy()
    
    zone_stats = compute_zone_stats(p_shots)
    max_attempts = max((s["attempts"] for s in zone_stats.values()), default=1)
    
    fig = make_subplots(
        rows=1, cols=1,
        subplot_titles=[player_subtitle(player_name)],
    )
    draw_court_lines(fig, row=1, col=1)
    
    for (zone, area), s in zone_stats.items():
        size = 20 + (s["attempts"] / max_attempts) * 50
        intensity = s["attempts"] / max_attempts
        color = f"rgba({int(206 - intensity*40)}, {int(17 + (1-intensity)*100)}, {int(65 + (1-intensity)*70)}, 0.78)"
        fig.add_trace(go.Scatter(
            x=[s["x"]], y=[s["y"]], mode='markers+text',
            marker=dict(size=size, color=color, line=dict(color='white', width=2)),
            text=[f"{s['fg_pct']*100:.0f}%"],
            textfont=dict(size=10.5, color='white', family='Inter'),
            textposition='middle center',
            hovertemplate=(
                f"<b>{zone} ({area})</b><br>"
                f"Attempts: {s['attempts']}<br>"
                f"Made: {s['made']}<br>"
                f"FG%: {s['fg_pct']*100:.1f}%<extra></extra>"
            ),
            showlegend=False,
        ), row=1, col=1)
    
    fig.update_layout(
        height=480, paper_bgcolor='white', plot_bgcolor='white',
        font=dict(family="Inter, sans-serif", color="#1A1A1A"),
        margin=dict(l=10, r=10, t=50, b=10),
        showlegend=False,
    )
    for ann in fig['layout']['annotations']:
        ann['font'] = dict(color='#1A1A1A', family='Inter, sans-serif', size=14)
    fig.update_xaxes(range=[-260, 260], visible=False, scaleanchor="y", scaleratio=1)
    fig.update_yaxes(range=[-60, 290], visible=False)
    
    plotly_html = fig.to_html(include_plotlyjs='cdn', full_html=True, config={'displayModeBar': False})
    return re.search(r'<body>(.*?)</body>', plotly_html, re.DOTALL).group(1)


def build_eason_split_chart():
    """Two-panel Eason chart: pre-ASB and post-ASB. Returns (chart_html, season_pct)."""
    shots_dated = shots.copy()
    shots_dated["GAME_DATE_parsed"] = pd.to_datetime(shots_dated["GAME_DATE"], format="%Y%m%d", errors="coerce")
    
    eason_all = shots_dated[
        (shots_dated["PLAYER_NAME"] == "Tari Eason") &
        (shots_dated["SEASON_LABEL"] == "2025-26") &
        (shots_dated["SEASON_TYPE"] == "Regular Season") &
        (shots_dated["TEAM_ID"] == ROCKETS_ID)
    ]
    pre_shots = eason_all[eason_all["GAME_DATE_parsed"] <= ASB_LAST]
    post_shots = eason_all[eason_all["GAME_DATE_parsed"] >= ASB_FIRST]
    
    pre_stats = compute_zone_stats(pre_shots)
    post_stats = compute_zone_stats(post_shots)
    max_attempts = max(
        (s["attempts"] for ps in [pre_stats, post_stats] for s in ps.values()),
        default=1,
    )
    
    def three_pt_pct(period_shots):
        threes = period_shots[period_shots["SHOT_TYPE"] == "3PT Field Goal"]
        if len(threes) == 0:
            return 0
        return threes["SHOT_MADE_FLAG"].sum() / len(threes) * 100
    
    pre_pct = three_pt_pct(pre_shots)
    post_pct = three_pt_pct(post_shots)
    
    all_threes = eason_all[eason_all["SHOT_TYPE"] == "3PT Field Goal"]
    season_pct = all_threes["SHOT_MADE_FLAG"].sum() / max(len(all_threes), 1) * 100
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            f"<b>Pre All-Star Break</b><br><span style='font-size:13px;color:#6B7280;font-weight:normal'><b style='color:#1A1A1A;font-size:15px'>{pre_pct:.1f}%</b> from three</span>",
            f"<b>Post All-Star Break</b><br><span style='font-size:13px;color:#6B7280;font-weight:normal'><b style='color:#1A1A1A;font-size:15px'>{post_pct:.1f}%</b> from three</span>",
        ],
        horizontal_spacing=0.04,
    )
    
    for idx, zone_stats in enumerate([pre_stats, post_stats]):
        col = idx + 1
        draw_court_lines(fig, row=1, col=col)
        for (zone, area), s in zone_stats.items():
            size = 20 + (s["attempts"] / max_attempts) * 50
            intensity = s["attempts"] / max_attempts
            color = f"rgba({int(206 - intensity*40)}, {int(17 + (1-intensity)*100)}, {int(65 + (1-intensity)*70)}, 0.78)"
            fig.add_trace(go.Scatter(
                x=[s["x"]], y=[s["y"]], mode='markers+text',
                marker=dict(size=size, color=color, line=dict(color='white', width=2)),
                text=[f"{s['fg_pct']*100:.0f}%"],
                textfont=dict(size=10.5, color='white', family='Inter'),
                textposition='middle center',
                hovertemplate=(
                    f"<b>{zone} ({area})</b><br>"
                    f"Attempts: {s['attempts']}<br>"
                    f"Made: {s['made']}<br>"
                    f"FG%: {s['fg_pct']*100:.1f}%<extra></extra>"
                ),
                showlegend=False,
            ), row=1, col=col)
    
    # Increase top margin to accommodate the chart-level header
    fig.update_layout(
        height=600,
        paper_bgcolor='white', plot_bgcolor='white',
        font=dict(family="Inter, sans-serif", color="#1A1A1A"),
        margin=dict(l=10, r=10, t=150, b=10),
        showlegend=False,
    )
    
    # Style subplot title annotations (Inter font, like DFS)
    for ann in fig['layout']['annotations']:
        ann['font'] = dict(color='#1A1A1A', family='Inter, sans-serif', size=14)
        # Move subplot titles a little lower so they don't crowd the header
        if ann.text and ("Pre All-Star" in str(ann.text) or "Post All-Star" in str(ann.text)):
            ann.y = 0.98
    
    # Add overall player header above the two subplots — uses Inter (same as DFS)
    fig.add_annotation(
        x=0.5, y=1.18,
        xref="paper", yref="paper",
        text=f"<b style='font-size:18px'>Tari Eason</b><br><span style='font-size:13px;color:#6B7280;font-weight:normal'><b style='color:#1A1A1A;font-size:15px'>{season_pct:.1f}%</b> from three</span>",
        showarrow=False,
        align="center",
        xanchor="center",
        yanchor="bottom",
        font=dict(family="Inter, sans-serif", color="#1A1A1A"),
    )
    
    for i in range(1, 3):
        fig.update_xaxes(range=[-260, 260], visible=False, scaleanchor=f"y{i}", scaleratio=1, row=1, col=i)
        fig.update_yaxes(range=[-60, 290], visible=False, row=1, col=i)
    
    plotly_html = fig.to_html(include_plotlyjs='cdn', full_html=True, config={'displayModeBar': False})
    return re.search(r'<body>(.*?)</body>', plotly_html, re.DOTALL).group(1), season_pct


# Build both charts
dfs_chart_html = build_single_player_chart("Dorian Finney-Smith")
eason_chart_html, eason_season_pct = build_eason_split_chart()

# Assemble 01e
page_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Three-Point Shooting Woes: The Supporting Cast</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@500&display=swap" rel="stylesheet">
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: 'Inter', sans-serif; background: #FAFAF7; color: #1A1A1A; padding: 60px 40px; }}
  .wrapper {{ max-width: 1100px; margin: 0 auto; }}
  .table-label {{ font-family: 'JetBrains Mono', monospace; font-size: 11px; letter-spacing: 0.15em; text-transform: uppercase; color: #CE1141; margin-bottom: 8px; }}
  h2 {{ font-family: 'Fraunces', Georgia, serif; font-size: 26px; font-weight: 600; letter-spacing: -0.015em; margin-bottom: 8px; }}
  .subtitle {{ color: #6B7280; font-size: 15px; margin-bottom: 24px; }}
  .chart-frame {{ background: white; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.04), 0 1px 2px rgba(0,0,0,0.06); padding: 8px; }}
  .player-prose {{ margin-top: 28px; margin-bottom: 36px; padding: 20px 24px; background: white; border-left: 3px solid #CE1141; border-radius: 4px; }}
  .player-prose .player-name {{ font-family: 'Fraunces', Georgia, serif; font-size: 18px; font-weight: 600; margin-bottom: 8px; color: #1A1A1A; }}
  .player-prose p {{ font-size: 14px; line-height: 1.65; color: #374151; }}
</style>
</head>
<body>
<div class="wrapper">
  <h2>Three-Point Shooting Woes: The Supporting Cast</h2>
  <p class="subtitle">Houston's signed veteran and returning wing didn't deliver the consistent perimeter shooting expected of them.</p>
  
  <div class="chart-frame">{dfs_chart_html}</div>
  <div class="player-prose">
    <div class="player-name">Dorian Finney-Smith</div>
    <p>{DFS_PROSE}</p>
  </div>

  <div class="chart-frame">{eason_chart_html}</div>
  <div class="player-prose">
    <div class="player-name">Tari Eason</div>
    <p>{EASON_PROSE}</p>
  </div>
</div>
</body>
</html>"""

(ASSETS_DIR / "01e_heatmap_supporting_cast.html").write_text(page_html)
print(f"Saved: 01e_heatmap_supporting_cast.html")

Saved: 01e_heatmap_supporting_cast.html


In [57]:
CHECK_PLAYERS = ["Kevin Durant", "Jabari Smith Jr.", "Reed Sheppard"]

check_shots = shots[
    (shots["PLAYER_NAME"].isin(CHECK_PLAYERS)) &
    (shots["SEASON_LABEL"] == "2025-26") &
    (shots["SEASON_TYPE"] == "Regular Season") &
    (shots["TEAM_ID"] == ROCKETS_ID)
].copy()

print("3PT shooting for the proposed 'reliable shooters' group:\n")
for p in CHECK_PLAYERS:
    ps = check_shots[check_shots["PLAYER_NAME"] == p]
    threes = ps[ps["SHOT_TYPE"] == "3PT Field Goal"]
    n_3pa = len(threes)
    n_3pm = int(threes["SHOT_MADE_FLAG"].sum())
    pct = (n_3pm / n_3pa * 100) if n_3pa > 0 else 0
    # Also get league context: what's "reliable" range for 3PT shooters? Usually 36%+
    print(f"  {p}:")
    print(f"    Total FGA: {len(ps)}")
    print(f"    3PA: {n_3pa}, 3PM: {n_3pm}, 3PT%: {pct:.1f}%")
    print()

# For comparison, league average 3PT% in 2025-26
league_3pa = league_gamelogs_2526["FG3A"].sum()
league_3pm = league_gamelogs_2526["FG3M"].sum()
print(f"League 3PT%: {league_3pm/league_3pa*100:.1f}%")

3PT shooting for the proposed 'reliable shooters' group:

  Kevin Durant:
    Total FGA: 1376
    3PA: 450, 3PM: 186, 3PT%: 41.3%

  Jabari Smith Jr.:
    Total FGA: 975
    3PA: 488, 3PM: 177, 3PT%: 36.3%

  Reed Sheppard:
    Total FGA: 946
    3PA: 576, 3PM: 227, 3PT%: 39.4%

League 3PT%: 36.0%


<a id="section-2"></a>
# Section 2: The Close-Out Problem

Builds the 5 assets for the Close-Out Problem tab:

| File | What it shows |
|---|---|
| `02a_clutch_comparison.html` | Year-over-year clutch stats (record, EFG%, ratings, TOV%) |
| `02b_overtime_problem.html` | OT record + Minnesota collapse (built later — see "Section 02b" below) |
| `02c_four_collapses.html` | Four Q4 collapses table |
| `02d_when_kd_sits.html` | Two-game side-by-side (POR + NYK) margin timeline |
| `02e_kd_off_scatter.html` | Margin change during KD-off stretches across all games |

## Load clutch data + Houston's clutch row

In [60]:
# Load clutch data
team_clutch_24_adv = pd.read_parquet(
    next(RAW_DIR.glob("league_team_clutch__measure_type=Advanced_season=2024-25*"))
)
team_clutch_25_adv = pd.read_parquet(
    next(RAW_DIR.glob("league_team_clutch__measure_type=Advanced_season=2025-26*"))
)
team_clutch_24_base = pd.read_parquet(
    next(RAW_DIR.glob("league_team_clutch__measure_type=Base_season=2024-25*"))
)
team_clutch_25_base = pd.read_parquet(
    next(RAW_DIR.glob("league_team_clutch__measure_type=Base_season=2025-26*"))
)

# Houston's clutch numbers
print("=== 2024-25 Clutch (Advanced) ===")
hou_24_adv = team_clutch_24_adv[team_clutch_24_adv["TEAM_ID"] == ROCKETS_ID].iloc[0]
print(hou_24_adv[["GP", "W", "L", "W_PCT", "OFF_RATING", "DEF_RATING", "NET_RATING", "TM_TOV_PCT", "EFG_PCT", "TS_PCT"]])

print("\n=== 2025-26 Clutch (Advanced) ===")
hou_25_adv = team_clutch_25_adv[team_clutch_25_adv["TEAM_ID"] == ROCKETS_ID].iloc[0]
print(hou_25_adv[["GP", "W", "L", "W_PCT", "OFF_RATING", "DEF_RATING", "NET_RATING", "TM_TOV_PCT", "EFG_PCT", "TS_PCT"]])

print("\n=== 2024-25 Clutch (Base) ===")
hou_24_base = team_clutch_24_base[team_clutch_24_base["TEAM_ID"] == ROCKETS_ID].iloc[0]
print(hou_24_base[["GP", "W", "L", "FG_PCT", "FG3_PCT", "FT_PCT", "TOV", "PTS"]])

print("\n=== 2025-26 Clutch (Base) ===")
hou_25_base = team_clutch_25_base[team_clutch_25_base["TEAM_ID"] == ROCKETS_ID].iloc[0]
print(hou_25_base[["GP", "W", "L", "FG_PCT", "FG3_PCT", "FT_PCT", "TOV", "PTS"]])

=== 2024-25 Clutch (Advanced) ===
GP               44
W                26
L                18
W_PCT         0.591
OFF_RATING    106.6
DEF_RATING    107.5
NET_RATING     -0.9
TM_TOV_PCT    0.127
EFG_PCT       0.438
TS_PCT        0.521
Name: 10, dtype: object

=== 2025-26 Clutch (Advanced) ===
GP               45
W                22
L                23
W_PCT         0.489
OFF_RATING    108.2
DEF_RATING    117.0
NET_RATING     -8.8
TM_TOV_PCT    0.169
EFG_PCT       0.493
TS_PCT        0.549
Name: 10, dtype: object

=== 2024-25 Clutch (Base) ===
GP            44
W             26
L             18
FG_PCT     0.381
FG3_PCT    0.274
FT_PCT     0.762
TOV         42.0
PTS          354
Name: 10, dtype: object

=== 2025-26 Clutch (Base) ===
GP            45
W             22
L             23
FG_PCT     0.441
FG3_PCT    0.318
FT_PCT     0.729
TOV         70.0
PTS          448
Name: 10, dtype: object


## Build `02a_clutch_comparison.html` — Year-over-year clutch stats table

In [360]:
# ============================================================
# Section 2: Clutch Stats Comparison Table
# ============================================================

# Pull Houston's clutch row from each season
hou_24_adv = team_clutch_24_adv[team_clutch_24_adv["TEAM_ID"] == ROCKETS_ID].iloc[0]
hou_25_adv = team_clutch_25_adv[team_clutch_25_adv["TEAM_ID"] == ROCKETS_ID].iloc[0]

# Build rows
def fmt_clutch(val, kind):
    if kind == "record":
        return val  # already a string like "26-18 (.591)"
    elif kind == "rating":
        sign = "+" if val > 0 else ""
        return f"{sign}{val:.1f}"
    elif kind == "pct":
        return f"{val*100:.1f}%"
    elif kind == "raw":
        return f"{val:.1f}"

# Reordered: Record, EFG%, Off Rtg, Def Rtg, Net Rtg, TOV%
clutch_rows = [
    ("Clutch Game Record",
     f"{int(hou_24_adv['W'])}–{int(hou_24_adv['L'])} (.{int(round(hou_24_adv['W_PCT']*1000)):03d})",
     f"{int(hou_25_adv['W'])}–{int(hou_25_adv['L'])} (.{int(round(hou_25_adv['W_PCT']*1000)):03d})"),
    ("Effective FG%",
     fmt_clutch(hou_24_adv['EFG_PCT'], 'pct'),
     fmt_clutch(hou_25_adv['EFG_PCT'], 'pct')),
    ("Offensive Rating",
     fmt_clutch(hou_24_adv['OFF_RATING'], 'raw'),
     fmt_clutch(hou_25_adv['OFF_RATING'], 'raw')),
    ("Defensive Rating",
     fmt_clutch(hou_24_adv['DEF_RATING'], 'raw'),
     fmt_clutch(hou_25_adv['DEF_RATING'], 'raw')),
    ("Net Rating",
     fmt_clutch(hou_24_adv['NET_RATING'], 'rating'),
     fmt_clutch(hou_25_adv['NET_RATING'], 'rating')),
    ("Turnover %",
     fmt_clutch(hou_24_adv['TM_TOV_PCT'], 'pct'),
     fmt_clutch(hou_25_adv['TM_TOV_PCT'], 'pct')),
]

clutch_rows_html = ""
for metric, v24, v25 in clutch_rows:
    clutch_rows_html += f"""
    <tr>
      <td class="metric">{metric}</td>
      <td class="value">{v24}</td>
      <td class="value">{v25}</td>
    </tr>"""

clutch_body = f"""
  <div class="table-label">Clutch Time, Section 02</div>
  <h2>Better Offense, Broken Defense</h2>
  <p class="table-subtitle">KD elevated Houston's clutch scoring, but defensive collapses and turnovers turned tight games into losses.</p>
  <table>
    <thead>
      <tr>
        <th>Metric</th>
        <th>2024–25</th>
        <th>2025–26</th>
      </tr>
    </thead>
    <tbody>{clutch_rows_html}
    </tbody>
  </table>
  <p class="table-footnote">Clutch games are defined as games where the score differential is 5 points or less with 5 minutes or less remaining in the fourth quarter or overtime.</p>
"""

with open(ASSETS_DIR / "02a_clutch_comparison.html", "w") as f:
    f.write(wrap_html("Clutch: Better Offense, Broken Defense", clutch_body))

print(f"Saved: {ASSETS_DIR / '02a_clutch_comparison.html'}")

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/02a_clutch_comparison.html


## Blown leads analysis

Walk play-by-play to find games where Houston led at any point and lost. Used to identify the four Q4 collapses for `02c`.

In [66]:
# Load play-by-play
pbp = pd.read_parquet(PROCESSED_DIR / "rockets_play_by_play.parquet")
print(f"Total PBP events: {len(pbp):,}")
print(f"Unique games: {pbp['gameId'].nunique()}")
print(f"\nColumns: {pbp.columns.tolist()}")
print(f"\nSample rows:")
pbp.head(3)

Total PBP events: 89,540
Unique games: 177

Columns: ['gameId', 'actionNumber', 'clock', 'period', 'teamId', 'teamTricode', 'personId', 'playerName', 'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult', 'isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal', 'location', 'description', 'actionType', 'subType', 'videoAvailable', 'shotValue', 'actionId']

Sample rows:


,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,...,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0022400016,2,PT12M00.00S,1,0,,0,,,0,...,0,0,0,,Start of 1st Period (8:12 PM EST),period,start,0,0,1
1,0022400016,4,PT12M00.00S,1,1610612745,HOU,1630578,Sengun,A. Sengun,0,...,,,0,h,Jump Ball Sengun vs. Zubac: Tip to Smith Jr.,Jump Ball,,1,0,2
2,0022400016,7,PT11M38.00S,1,1610612745,HOU,1628415,Brooks,D. Brooks,-11,...,,,0,h,MISS Brooks 6' Fadeaway Jumper,Missed Shot,Fadeaway Jump Shot,1,2,3


In [ ]:
# ============================================================
# Section 2: Blown Leads Analysis (2025-26 regular season only)
# ============================================================

team_gl = pd.read_parquet(PROCESSED_DIR / "rockets_team_gamelogs.parquet")
pbp = pd.read_parquet(PROCESSED_DIR / "rockets_play_by_play.parquet")

# Filter to 2025-26 regular season games only
team_gl_2526 = team_gl[
    (team_gl["SEASON"] == "2025-26") & 
    (team_gl["SEASON_TYPE"] == "Regular Season")
].copy()

team_gl_2526["GAME_ID_STR"] = team_gl_2526["Game_ID"].astype(str).str.zfill(10)
team_gl_2526["IS_HOME"] = team_gl_2526["MATCHUP"].str.contains("vs")

# Get the set of game IDs we care about
target_game_ids = set(team_gl_2526["GAME_ID_STR"])

# Filter PBP to those games, cast scores
pbp = pbp.copy()
pbp["gameId_str"] = pbp["gameId"].astype(str).str.zfill(10)
pbp = pbp[pbp["gameId_str"].isin(target_game_ids)]
pbp["scoreHome"] = pd.to_numeric(pbp["scoreHome"], errors="coerce").fillna(0).astype(int)
pbp["scoreAway"] = pd.to_numeric(pbp["scoreAway"], errors="coerce").fillna(0).astype(int)

# Game metadata
game_info = team_gl_2526.set_index("GAME_ID_STR")[
    ["GAME_DATE", "MATCHUP", "WL", "IS_HOME", "PTS"]
].to_dict("index")

# Walk through each game
lead_rows = []
for game_id, group in pbp.groupby("gameId_str"):
    info = game_info[game_id]
    is_home = info["IS_HOME"]
    margin = (group["scoreHome"] - group["scoreAway"]) if is_home else (group["scoreAway"] - group["scoreHome"])
    lead_rows.append({
        "game_id": game_id,
        "max_lead": int(margin.max()),
        "final_margin": int(margin.iloc[-1]),
        "result": info["WL"],
        "date": info["GAME_DATE"],
        "matchup": info["MATCHUP"],
    })

lead_summary = pd.DataFrame(lead_rows)
print(f"Computed lead summary for {len(lead_summary)} games (2025-26 regular season)")
lead_summary.head(50)

Computed lead summary for 82 games (2025-26 regular season)


,game_id,max_lead,final_margin,result,date,matchup
0,0022500001,12,-1,L,"OCT 21, 2025",HOU @ OKC
1,0022500012,24,23,W,"DEC 25, 2025",HOU @ LAL
2,0022500032,11,-11,L,"NOV 07, 2025",HOU @ SAS
3,0022500042,30,24,W,"NOV 14, 2025",HOU vs. POR
4,0022500054,5,-3,L,"NOV 21, 2025",HOU vs. DEN


In [81]:
lead_summary.head(50)

,game_id,max_lead,final_margin,result,date,matchup
0,0022500001,12,-1,L,"OCT 21, 2025",HOU @ OKC
1,0022500012,24,23,W,"DEC 25, 2025",HOU @ LAL
2,0022500032,11,-11,L,"NOV 07, 2025",HOU @ SAS
3,0022500042,30,24,W,"NOV 14, 2025",HOU vs. POR
4,0022500054,5,-3,L,"NOV 21, 2025",HOU vs. DEN
5,0022500066,8,4,W,"NOV 26, 2025",HOU @ GSW
6,0022500093,6,-4,L,"OCT 24, 2025",HOU vs. DET
7,0022500116,33,28,W,"OCT 27, 2025",HOU vs. BKN
8,0022500131,21,18,W,"OCT 29, 2025",HOU @ TOR
9,0022500146,36,27,W,"NOV 01, 2025",HOU @ BOS


In [80]:
# ============================================================
# Refined blown leads: end-of-Q3 lead and OT lead
# ============================================================

def analyze_game_leads(game_id, group, is_home):
    """Return dict with end-of-Q3 lead and max OT lead for this game."""
    # Compute margin at every event
    if is_home:
        margin = group["scoreHome"] - group["scoreAway"]
    else:
        margin = group["scoreAway"] - group["scoreHome"]
    
    # End of Q3: last event in period 3
    q3_events = group[group["period"] == 3]
    if len(q3_events) > 0:
        end_q3_idx = q3_events.index[-1]
        end_q3_lead = (
            (group.loc[end_q3_idx, "scoreHome"] - group.loc[end_q3_idx, "scoreAway"])
            if is_home else
            (group.loc[end_q3_idx, "scoreAway"] - group.loc[end_q3_idx, "scoreHome"])
        )
    else:
        end_q3_lead = None
    
    # OT: any period >= 5
    ot_events = group[group["period"] >= 5]
    if len(ot_events) > 0:
        ot_margin = (
            (ot_events["scoreHome"] - ot_events["scoreAway"])
            if is_home else
            (ot_events["scoreAway"] - ot_events["scoreHome"])
        )
        max_ot_lead = int(ot_margin.max())
        went_to_ot = True
    else:
        max_ot_lead = None
        went_to_ot = False
    
    return {
        "end_q3_lead": int(end_q3_lead) if end_q3_lead is not None else None,
        "max_ot_lead": max_ot_lead,
        "went_to_ot": went_to_ot,
    }


# Build detailed analysis
detailed_rows = []
for game_id, group in pbp.groupby("gameId_str"):
    info = game_info[game_id]
    analysis = analyze_game_leads(game_id, group, info["IS_HOME"])
    detailed_rows.append({
        "game_id": game_id,
        "date": info["GAME_DATE"],
        "matchup": info["MATCHUP"],
        "result": info["WL"],
        **analysis,
    })

detailed = pd.DataFrame(detailed_rows)

# === Show what we found ===
print("=" * 60)
print("Q4 COLLAPSES (Led by 10+ at end of Q3, lost the game)")
print("=" * 60)
q4_collapses = detailed[(detailed["end_q3_lead"] >= 10) & (detailed["result"] == "L")]
print(f"Count: {len(q4_collapses)}")
print(q4_collapses[["date", "matchup", "end_q3_lead", "result"]].to_string(index=False))

print()
print("=" * 60)
print("OT COLLAPSES (Led by 10+ in OT, lost the game)")
print("=" * 60)
ot_collapses = detailed[(detailed["went_to_ot"]) & (detailed["max_ot_lead"] >= 10) & (detailed["result"] == "L")]
print(f"Count: {len(ot_collapses)}")
if len(ot_collapses) > 0:
    print(ot_collapses[["date", "matchup", "max_ot_lead", "result"]].to_string(index=False))

# Also show with thresholds of 5 and 15 for context
print()
print("=" * 60)
print("Q4 LEAD COLLAPSES — different thresholds")
print("=" * 60)
for thresh in [5, 10, 15, 20]:
    n = len(detailed[(detailed["end_q3_lead"] >= thresh) & (detailed["result"] == "L")])
    total_with_lead = len(detailed[detailed["end_q3_lead"] >= thresh])
    print(f"Led by {thresh}+ entering Q4 — Record: {total_with_lead - n}-{n} ({(total_with_lead-n)/total_with_lead*100:.0f}% win rate)")

Q4 COLLAPSES (Led by 10+ at end of Q3, lost the game)
Count: 3
        date   matchup  end_q3_lead result
DEC 18, 2025 HOU @ NOP           16      L
JAN 09, 2026 HOU @ POR           13      L
FEB 21, 2026 HOU @ NYK           16      L

OT COLLAPSES (Led by 10+ in OT, lost the game)
Count: 1
        date   matchup  max_ot_lead result
MAR 25, 2026 HOU @ MIN         13.0      L

Q4 LEAD COLLAPSES — different thresholds
Led by 5+ entering Q4 — Record: 34-5 (87% win rate)
Led by 10+ entering Q4 — Record: 26-3 (90% win rate)
Led by 15+ entering Q4 — Record: 20-2 (91% win rate)
Led by 20+ entering Q4 — Record: 15-0 (100% win rate)


## KD substitution + Four Collapses table

Reconstruct KD's on/off timeline for each blown-lead game by walking PBP substitutions. Builds the data behind `02c_four_collapses.html`.

In [84]:
# Let's understand the substitution structure better
# Look at all substitution events for a single game
sample_game = pbp["gameId_str"].iloc[0]  # any game
sample_subs = pbp[
    (pbp["gameId_str"] == sample_game) & 
    (pbp["actionType"] == "Substitution")
].copy()

print(f"Sample game: {sample_game}")
print(f"Total subs in this game: {len(sample_subs)}")
print(sample_subs[["period", "clock", "playerName", "description"]].head(20))

Sample game: 0022500001
Total subs in this game: 67
       period        clock   playerName                         description
41476       1  PT06M40.00S    Smith Jr.            SUB: Eason FOR Smith Jr.
41484       1  PT06M15.00S  Hartenstein       SUB: Williams FOR Hartenstein
41489       1  PT05M31.00S       Sengun            SUB: Sheppard FOR Sengun
41500       1  PT04M08.00S        Adams               SUB: Capela FOR Adams
41503       1  PT04M03.00S      Wallace            SUB: Wiggins FOR Wallace
41512       1  PT03M26.00S     Holmgren            SUB: Caruso FOR Holmgren
41513       1  PT03M26.00S     Thompson            SUB: Sengun FOR Thompson
41520       1  PT02M21.00S         Dort             SUB: Barnhizer FOR Dort
41543       1  PT00M05.60S       Caruso            SUB: Mitchell FOR Caruso
41544       1  PT00M05.60S    Barnhizer       SUB: Youngblood FOR Barnhizer
41545       1  PT00M05.60S     Sheppard          SUB: Thompson FOR Sheppard
41546       1  PT00M05.60S       Cap

In [92]:
def reconstruct_kd_timeline(game_id):
    info = game_info[game_id]
    is_home = info["IS_HOME"]
    
    game_pbp = pbp[pbp["gameId_str"] == game_id].copy().reset_index(drop=True)
    
    # Running margin
    game_pbp["scoreHome_clean"] = game_pbp["scoreHome"].cummax()
    game_pbp["scoreAway_clean"] = game_pbp["scoreAway"].cummax()
    if is_home:
        game_pbp["margin"] = game_pbp["scoreHome_clean"] - game_pbp["scoreAway_clean"]
    else:
        game_pbp["margin"] = game_pbp["scoreAway_clean"] - game_pbp["scoreHome_clean"]
    
    # KD sub events
    kd_subs = game_pbp[
        (game_pbp["actionType"] == "Substitution") &
        ((game_pbp["playerName"] == "Durant") |
         (game_pbp["description"].str.contains("SUB: Durant", na=False)))
    ].copy()
    kd_subs["action"] = kd_subs.apply(
        lambda r: "OUT" if r["playerName"] == "Durant" else "IN", axis=1
    )
    
    # KD non-sub actions (made shot, foul, rebound, etc.) — confirms he was on the court
    kd_actions_nonsub = game_pbp[
        (game_pbp["actionType"] != "Substitution") &
        ((game_pbp["playerName"] == "Durant") |
         (game_pbp["description"].str.contains("Durant", na=False, case=False)))
    ]
    
    timeline = []
    kd_on = True  # KD starts the game (he's a starter)
    
    # Add the implicit Q1 12:00 IN
    timeline.append({
        "period": 1,
        "clock_sec": 12 * 60,
        "action": "IN",
        "margin": 0,
        "implicit": True,
    })
    
    for p in sorted(game_pbp["period"].unique()):
        quarter_subs = kd_subs[kd_subs["period"] == p].sort_values("actionNumber")
        quarter_nonsub_actions = kd_actions_nonsub[kd_actions_nonsub["period"] == p]
        
        # For quarters after Q1, infer the start-of-quarter state
        if p > 1:
            has_subs = len(quarter_subs) > 0
            has_nonsub = len(quarter_nonsub_actions) > 0
            
            if has_subs:
                first_sub_action = quarter_subs.iloc[0]["action"]
                desired_start_state = "ON" if first_sub_action == "OUT" else "OFF"
            elif has_nonsub:
                desired_start_state = "ON"
            else:
                desired_start_state = "ON" if kd_on else "OFF"
            
            period_start = game_pbp[game_pbp["period"] == p].iloc[0]
            
            if desired_start_state == "ON" and not kd_on:
                # State flip: was OFF, now ON
                timeline.append({
                    "period": p,
                    "clock_sec": 12 * 60,
                    "action": "IN",
                    "margin": int(period_start["margin"]),
                    "implicit": True,
                })
                kd_on = True
            elif desired_start_state == "OFF" and kd_on:
                # State flip: was ON, now OFF
                timeline.append({
                    "period": p,
                    "clock_sec": 12 * 60,
                    "action": "OUT",
                    "margin": int(period_start["margin"]),
                    "implicit": True,
                })
                kd_on = False
            else:
                # State carries over — add an explicit marker for clarity
                timeline.append({
                    "period": p,
                    "clock_sec": 12 * 60,
                    "action": "IN" if kd_on else "OUT",
                    "margin": int(period_start["margin"]),
                    "implicit": True,
                    "carryover": True,  # new flag to distinguish from state flips
                })
        
        # Now process all explicit subs in this quarter
        for _, sub in quarter_subs.iterrows():
            timeline.append({
                "period": p,
                "clock_sec": parse_clock_to_seconds(sub["clock"]),
                "action": sub["action"],
                "margin": int(sub["margin"]),
                "implicit": False,
            })
            kd_on = (sub["action"] == "IN")
    
    return timeline


for game_id in Q4_COLLAPSE_GAMES:
    info = game_info[game_id]
    print(f"\n{'='*70}")
    print(f"  {info['GAME_DATE']}  |  {info['MATCHUP']}  |  Result: {info['WL']}")
    print(f"{'='*70}")
    
    tl = reconstruct_kd_timeline(game_id)
    for evt in tl:
        clock_s = evt["clock_sec"]
        mins = int(clock_s // 60)
        secs = clock_s - mins * 60
        period_label = f"Q{evt['period']}" if evt['period'] <= 4 else f"OT{evt['period']-4}"
        if evt.get("carryover"):
            marker = " (continued from previous quarter)"
        elif evt["implicit"]:
            marker = " (inferred at quarter break)"
        else:
            marker = ""
        print(f"  {period_label}  {mins:02d}:{secs:05.2f}  KD {evt['action']:3s}  |  margin: {evt['margin']:+d}{marker}")


  DEC 18, 2025  |  HOU @ NOP  |  Result: L
  Q1  12:00.00  KD IN   |  margin: +0 (inferred at quarter break)
  Q1  03:04.00  KD OUT  |  margin: +9
  Q2  12:00.00  KD IN   |  margin: +9 (inferred at quarter break)
  Q2  05:58.00  KD OUT  |  margin: +18
  Q2  02:43.00  KD IN   |  margin: +23
  Q3  12:00.00  KD IN   |  margin: +22 (continued from previous quarter)
  Q3  03:25.00  KD OUT  |  margin: +17
  Q4  12:00.00  KD IN   |  margin: +16 (inferred at quarter break)
  Q4  06:03.00  KD OUT  |  margin: +6
  Q4  05:28.00  KD IN   |  margin: +5
  OT1  12:00.00  KD IN   |  margin: +0 (continued from previous quarter)

  JAN 09, 2026  |  HOU @ POR  |  Result: L
  Q1  12:00.00  KD IN   |  margin: +0 (inferred at quarter break)
  Q1  02:06.00  KD OUT  |  margin: -3
  Q2  12:00.00  KD OUT  |  margin: +1 (continued from previous quarter)
  Q2  08:45.00  KD IN   |  margin: -3
  Q3  12:00.00  KD IN   |  margin: +1 (continued from previous quarter)
  Q4  12:00.00  KD OUT  |  margin: +13 (inferred a

### Build `02c_four_collapses.html` — Four Q4 collapses table

In [94]:
# ============================================================
# Visualization A: The Four Collapses (table)
# ============================================================

# Build the 4-game list with full context (3 Q4 + 1 OT)
all_collapses = [
    {
        "date": "DEC 18, 2025", "opponent": "@ Pelicans",
        "lead": "+16 entering Q4", "result": "L (OT)",
        "note": "Lost in overtime",
    },
    {
        "date": "JAN 09, 2026", "opponent": "@ Trail Blazers",
        "lead": "+13 entering Q4", "result": "L",
        "note": "Q4 collapse",
    },
    {
        "date": "FEB 21, 2026", "opponent": "@ Knicks",
        "lead": "+16 entering Q4", "result": "L",
        "note": "Q4 collapse",
    },
    {
        "date": "MAR 25, 2026", "opponent": "@ Timberwolves",
        "lead": "+13 in OT", "result": "L (OT)",
        "note": "OT collapse",
    },
]

table_rows = ""
for g in all_collapses:
    table_rows += f"""
    <tr>
      <td class="metric">{g['date']}</td>
      <td class="value" style="text-align:left">{g['opponent']}</td>
      <td class="value">{g['lead']}</td>
      <td class="value">{g['result']}</td>
      <td class="value" style="text-align:left;color:#6B7280;font-style:italic">{g['note']}</td>
    </tr>"""

four_collapses_body = f"""
  <div class="table-label">Clutch Time, Section 02</div>
  <h2>The Four Collapses</h2>
  <p class="table-subtitle">Houston was 26–3 when leading by 10+ entering the fourth quarter. The three games they didn't close out, plus one stunning overtime collapse, are the ones that still sting.</p>
  <table>
    <thead>
      <tr>
        <th style="text-align:left">Date</th>
        <th style="text-align:left">Opponent</th>
        <th>Lead Held</th>
        <th>Result</th>
        <th style="text-align:left">Notes</th>
      </tr>
    </thead>
    <tbody>{table_rows}
    </tbody>
  </table>
"""

with open(ASSETS_DIR / "02b_four_collapses.html", "w") as f:
    f.write(wrap_html("The Four Collapses", four_collapses_body))

print(f"Saved: {ASSETS_DIR / '02b_four_collapses.html'}")

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/02b_four_collapses.html


### Build `02d_when_kd_sits.html` — POR + NYK margin timeline

In [278]:
# ============================================================
# Visualization B: When KD Sits (POR + NYK timeline)
# ============================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Find POR and NYK game IDs
POR_GAME = q4_collapses[q4_collapses["matchup"].str.contains("POR")]["game_id"].iloc[0]
NYK_GAME = q4_collapses[q4_collapses["matchup"].str.contains("NYK")]["game_id"].iloc[0]

KD_SITS_GAMES = [
    {"game_id": POR_GAME, "label": "vs Portland (Jan 9)", "short": "POR", "win_prob": 96.0},
    {"game_id": NYK_GAME, "label": "vs New York (Feb 21)", "short": "NYK", "win_prob": 96.8},
]


def build_margin_timeline(game_id):
    """Return a DataFrame with elapsed_seconds and margin for every event in the game."""
    info = game_info[game_id]
    is_home = info["IS_HOME"]
    
    game_pbp = pbp[pbp["gameId_str"] == game_id].copy().reset_index(drop=True)
    game_pbp["scoreHome_clean"] = game_pbp["scoreHome"].cummax()
    game_pbp["scoreAway_clean"] = game_pbp["scoreAway"].cummax()
    if is_home:
        game_pbp["margin"] = game_pbp["scoreHome_clean"] - game_pbp["scoreAway_clean"]
    else:
        game_pbp["margin"] = game_pbp["scoreAway_clean"] - game_pbp["scoreHome_clean"]
    
    # Convert clock + period to elapsed seconds from start of game
    # Quarter is 12 mins = 720 sec. So elapsed = (period-1)*720 + (720 - clock_seconds_remaining)
    def elapsed_seconds(row):
        clock_s = parse_clock_to_seconds(row["clock"])
        if clock_s is None:
            return None
        return (row["period"] - 1) * 720 + (720 - clock_s)
    
    game_pbp["elapsed_sec"] = game_pbp.apply(elapsed_seconds, axis=1)
    return game_pbp[["period", "clock", "elapsed_sec", "margin"]].dropna()


def build_off_court_intervals(timeline):
    """From the reconstructed KD timeline, return list of (start_sec, end_sec) for off-court intervals."""
    intervals = []
    off_start = None
    
    for evt in timeline:
        clock_s = evt["clock_sec"]
        elapsed = (evt["period"] - 1) * 720 + (720 - clock_s)
        
        if evt["action"] == "OUT":
            off_start = elapsed
        elif evt["action"] == "IN" and off_start is not None:
            intervals.append((off_start, elapsed))
            off_start = None
    
    # If still off when timeline ends, close at end of game
    if off_start is not None:
        intervals.append((off_start, off_start + 30))  # cap with a short tail
    
    return intervals


# Build the chart — 2 panels side-by-side
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[g["label"] for g in KD_SITS_GAMES],
    horizontal_spacing=0.08,
)

ROCKETS_RED = "#CE1141"
BENCH_GRAY = "rgba(150, 150, 150, 0.2)"

for idx, game in enumerate(KD_SITS_GAMES):
    col = idx + 1
    game_id = game["game_id"]
    
    # Margin timeline
    tl_df = build_margin_timeline(game_id)
    # Reconstruct KD on/off timeline
    kd_tl = reconstruct_kd_timeline(game_id)
    off_intervals = build_off_court_intervals(kd_tl)
    
    # Determine x-axis: focus on Q3 start through end of game
    q3_start = 2 * 720  # start of Q3
    game_end = tl_df["elapsed_sec"].max()
    
    # Filter to Q3 + Q4 + OT
    tl_focus = tl_df[tl_df["elapsed_sec"] >= q3_start].copy()
    
    # Plot margin line with smoothed spline
    fig.add_trace(go.Scatter(
        x=tl_focus["elapsed_sec"],
        y=tl_focus["margin"],
        mode="lines",
        line=dict(color=ROCKETS_RED, width=2.5, shape="spline", smoothing=0.6),
        name="HOU margin",
        hovertemplate="Margin: %{y:+d}<extra></extra>",
        showlegend=False,
    ), row=1, col=col)
    
    # Add zero line
    fig.add_hline(y=0, line=dict(color="#9CA3AF", width=1, dash="dot"),
                  row=1, col=col)
    
    # Shade KD off-court intervals (only those that overlap with Q3+)
    for start, end in off_intervals:
        if end < q3_start:
            continue  # skip earlier-game subs
        shade_start = max(start, q3_start)
        fig.add_vrect(
            x0=shade_start, x1=end,
            fillcolor=BENCH_GRAY,
            line_width=0,
            layer="below",
            row=1, col=col,
        )
    
    # Add quarter break markers
    for q_boundary in [3*720, 4*720]:  # end of Q3, end of Q4
        if q_boundary <= game_end:
            fig.add_vline(
                x=q_boundary,
                line=dict(color="#1A1A1A", width=1),
                row=1, col=col,
            )
    
    # Annotations for key moments
    # Lead and win probability entering Q4
    q4_start = 3 * 720
    lead_at_q4 = tl_focus[tl_focus["elapsed_sec"] <= q4_start]["margin"].iloc[-1]
    fig.add_annotation(
        x=q4_start, y=lead_at_q4,
        text=f"<b>+{int(lead_at_q4)} entering Q4</b><br><span style='font-size:10px'>{game['win_prob']}% win probability per ESPN</span>",
        showarrow=True, arrowhead=2, ax=-70, ay=-45,
        font=dict(size=11, color="#1A1A1A"),
        bgcolor="white", bordercolor="#1A1A1A", borderwidth=1, borderpad=6,
        align="left",
        row=1, col=col,
    )
    
    # Lead when KD came back
    kd_returns = [e for e in kd_tl if e["action"] == "IN" and not e.get("implicit") 
                  and (e["period"] - 1) * 720 + (720 - e["clock_sec"]) > q4_start]
    if kd_returns:
        first_return = kd_returns[0]
        ret_elapsed = (first_return["period"] - 1) * 720 + (720 - first_return["clock_sec"])
        ret_margin = first_return["margin"]
        fig.add_annotation(
            x=ret_elapsed, y=ret_margin,
            text=f"+{int(ret_margin)} when KD returns",
            showarrow=True, arrowhead=2, ax=-30, ay=50,
            font=dict(size=11, color="#CE1141"),
            bgcolor="white", bordercolor="#CE1141", borderwidth=1, borderpad=4,
            row=1, col=col,
        )

# X-axis tick labels (show quarter labels at boundaries)
quarter_ticks = [2*720, 3*720, 4*720, 5*720]
quarter_labels = ["Start Q3", "Start Q4", "End Q4", "End OT"]

for col in [1, 2]:
    fig.update_xaxes(
        tickmode="array",
        tickvals=quarter_ticks,
        ticktext=quarter_labels,
        showgrid=False,
        zeroline=False,
        row=1, col=col,
    )
    fig.update_yaxes(
        title_text="Rockets Margin" if col == 1 else None,
        zeroline=False,
        gridcolor="#F0EDE5",
        row=1, col=col,
    )

fig.update_layout(
    height=420,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Inter, sans-serif", color="#1A1A1A"),
    margin=dict(l=50, r=20, t=60, b=40),
    showlegend=False,
)

labels = [g["label"] for g in KD_SITS_GAMES]
for ann in fig['layout']['annotations']:
    if ann.text in labels:
        ann.font = dict(size=14, color="#1A1A1A", family="Inter, sans-serif")

# Save as HTML
plotly_html = fig.to_html(include_plotlyjs='cdn', full_html=True,
                          config={'displayModeBar': False})

wrapper = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>When KD Sits</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@500&display=swap" rel="stylesheet">
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    font-family: 'Inter', sans-serif;
    background: #FAFAF7;
    color: #1A1A1A;
    padding: 60px 40px;
  }
  .wrapper { max-width: 1100px; margin: 0 auto; }
  .table-label {
    font-family: 'JetBrains Mono', monospace;
    font-size: 11px;
    letter-spacing: 0.15em;
    text-transform: uppercase;
    color: #CE1141;
    margin-bottom: 8px;
  }
  h2 {
    font-family: 'Fraunces', Georgia, serif;
    font-size: 26px;
    font-weight: 600;
    letter-spacing: -0.015em;
    margin-bottom: 8px;
  }
  .subtitle {
    color: #6B7280;
    font-size: 15px;
    margin-bottom: 24px;
  }
  .legend-note {
    font-size: 13px;
    color: #6B7280;
    margin-top: 14px;
    font-style: italic;
  }
  .legend-swatch {
    display: inline-block;
    width: 14px;
    height: 14px;
    background: rgba(150, 150, 150, 0.4);
    vertical-align: middle;
    margin-right: 4px;
    border-radius: 2px;
  }
  .chart-frame {
    background: white;
    border-radius: 8px;
    padding: 8px;
    box-shadow: 0 1px 3px rgba(0,0,0,0.04), 0 1px 2px rgba(0,0,0,0.06);
  }
</style>
</head>
<body>
<div class="wrapper">
  <div class="table-label">Clutch Time, Section 02</div>
  <h2>When KD Sits</h2>
  <p class="subtitle">In two of Houston's most painful Q4 collapses, Kevin Durant played the entire 3rd quarter and started the 4th quarter on the bench. By the time he returned, the lead was nearly gone.</p>
  <div class="chart-frame">
__CHART__
  </div>
  <p class="legend-note"><span class="legend-swatch"></span> Shaded regions indicate when Kevin Durant was off the court.</p>
</div>
</body>
</html>"""

import re
chart_div = re.search(r'<body>(.*?)</body>', plotly_html, re.DOTALL).group(1)
final = wrapper.replace("__CHART__", chart_div)

with open(ASSETS_DIR / "02d_when_kd_sits.html", "w") as f:
    f.write(final)
print(f"Saved: {ASSETS_DIR / '02d_when_kd_sits.html'}")
fig.show()

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/02d_when_kd_sits.html


## Expanded KD-off analysis

Walk every game for stretches where KD subbed off, measure margin change while he was on the bench. Used for `02e` scatterplot.

In [104]:
# ============================================================
# Expanded analysis: KD-off stretches across all games
# ============================================================

def find_kd_off_stretches(game_id):
    """Return list of dicts with each KD-off stretch in the game."""
    timeline = reconstruct_kd_timeline(game_id)
    
    stretches = []
    out_event = None  # The "OUT" event that started the current off-stretch
    
    for evt in timeline:
        if evt["action"] == "OUT":
            out_event = evt
        elif evt["action"] == "IN" and out_event is not None:
            # End of an off-stretch
            duration_sec = (
                ((evt["period"] - 1) * 720 + (720 - evt["clock_sec"]))
                - ((out_event["period"] - 1) * 720 + (720 - out_event["clock_sec"]))
            )
            margin_change = evt["margin"] - out_event["margin"]
            stretches.append({
                "game_id": game_id,
                "out_period": out_event["period"],
                "out_clock_sec": out_event["clock_sec"],
                "out_margin": out_event["margin"],
                "in_period": evt["period"],
                "in_clock_sec": evt["clock_sec"],
                "in_margin": evt["margin"],
                "duration_sec": duration_sec,
                "margin_change": margin_change,
                "out_implicit": out_event["implicit"],
                "in_implicit": evt["implicit"],
            })
            out_event = None
    return stretches


# Filter to games where Houston led at some point
games_with_leads = []
for game_id in team_gl_2526["GAME_ID_STR"]:
    game_pbp = pbp[pbp["gameId_str"] == game_id]
    if len(game_pbp) == 0:
        continue
    info = game_info[game_id]
    is_home = info["IS_HOME"]
    home = game_pbp["scoreHome"].cummax()
    away = game_pbp["scoreAway"].cummax()
    margin = (home - away) if is_home else (away - home)
    if margin.max() > 0:
        games_with_leads.append(game_id)

print(f"Games where Houston led at some point: {len(games_with_leads)}")

# Run KD off-stretch analysis on all qualifying games
all_stretches = []
for game_id in games_with_leads:
    all_stretches.extend(find_kd_off_stretches(game_id))

stretches_df = pd.DataFrame(all_stretches)
stretches_df["duration_min"] = stretches_df["duration_sec"] / 60

# Filter out tiny stretches (< 30 seconds) — these are mostly free throw subs
real_stretches = stretches_df[stretches_df["duration_sec"] >= 30].copy()

print(f"\nTotal KD-off stretches: {len(stretches_df)}")
print(f"Stretches ≥ 30 seconds: {len(real_stretches)}")
print(f"\n--- Aggregate stats across all stretches ---")
print(f"Total KD-off time: {real_stretches['duration_sec'].sum() / 60:.0f} minutes")
print(f"Average stretch duration: {real_stretches['duration_min'].mean():.1f} minutes")
print(f"Net margin change across all stretches: {real_stretches['margin_change'].sum():+d} points")
print(f"Average margin change per stretch: {real_stretches['margin_change'].mean():+.2f} points")
print(f"Margin change per minute: {real_stretches['margin_change'].sum() / (real_stretches['duration_sec'].sum() / 60):+.2f} points per minute KD sits")

# How often does the lead shrink during KD-off stretches?
shrunk = (real_stretches["margin_change"] < 0).sum()
grew = (real_stretches["margin_change"] > 0).sum()
even = (real_stretches["margin_change"] == 0).sum()
print(f"\nStretches where margin SHRUNK while KD sat: {shrunk} ({shrunk/len(real_stretches)*100:.0f}%)")
print(f"Stretches where margin GREW while KD sat:   {grew} ({grew/len(real_stretches)*100:.0f}%)")
print(f"Stretches where margin was EVEN:            {even}")

Games where Houston led at some point: 82

Total KD-off stretches: 198
Stretches ≥ 30 seconds: 190

--- Aggregate stats across all stretches ---
Total KD-off time: 655 minutes
Average stretch duration: 3.4 minutes
Net margin change across all stretches: +33 points
Average margin change per stretch: +0.17 points
Margin change per minute: +0.05 points per minute KD sits

Stretches where margin SHRUNK while KD sat: 78 (41%)
Stretches where margin GREW while KD sat:   96 (51%)
Stretches where margin was EVEN:            16


In [105]:
# ============================================================
# Per-game KD-off summary
# ============================================================

# Aggregate stretches by game
per_game = (
    real_stretches.groupby("game_id")
    .agg(
        n_stretches=("duration_sec", "count"),
        total_off_minutes=("duration_sec", lambda s: s.sum() / 60),
        net_margin_change=("margin_change", "sum"),
    )
    .reset_index()
)

# Add game info (date, opponent, result)
per_game["date"] = per_game["game_id"].map(lambda g: game_info[g]["GAME_DATE"])
per_game["matchup"] = per_game["game_id"].map(lambda g: game_info[g]["MATCHUP"])
per_game["result"] = per_game["game_id"].map(lambda g: game_info[g]["WL"])

# Sort by worst margin change (most damaging KD-off games first)
per_game = per_game.sort_values("net_margin_change").reset_index(drop=True)

# Show the 15 worst
print("=" * 80)
print("WORST 15 GAMES (Houston lost the most ground while KD sat)")
print("=" * 80)
print(per_game.head(15).to_string(index=False))

print("\n" + "=" * 80)
print("BEST 15 GAMES (Houston gained the most ground while KD sat)")
print("=" * 80)
print(per_game.tail(15).to_string(index=False))

print("\n" + "=" * 80)
print("DISTRIBUTION")
print("=" * 80)
print(f"Total games: {len(per_game)}")
print(f"Games where Houston LOST ground while KD sat: {(per_game['net_margin_change'] < 0).sum()}")
print(f"Games where Houston GAINED ground while KD sat: {(per_game['net_margin_change'] > 0).sum()}")
print(f"\nGames where Houston lost 10+ points while KD sat: {(per_game['net_margin_change'] <= -10).sum()}")
print(f"Games where Houston lost 15+ points while KD sat: {(per_game['net_margin_change'] <= -15).sum()}")
print(f"\nGames where the result was a loss AND Houston lost ground while KD sat: " + 
      f"{((per_game['result'] == 'L') & (per_game['net_margin_change'] < 0)).sum()}")

WORST 15 GAMES (Houston lost the most ground while KD sat)
   game_id  n_stretches  total_off_minutes  net_margin_change         date     matchup result
0022500491            3           9.768333                -21 JAN 03, 2026   HOU @ DAL      L
0022500540            2           6.550000                -15 JAN 09, 2026   HOU @ POR      L
0022500816            2           9.633333                -15 FEB 21, 2026   HOU @ NYK      L
0022500864            3          10.141667                -15 FEB 28, 2026   HOU @ MIA      L
0022500042            2           7.696667                -13 NOV 14, 2025 HOU vs. POR      W
0022500782            3           9.101667                -12 FEB 11, 2026 HOU vs. LAC      L
0022500032            3           8.450000                -10 NOV 07, 2025   HOU @ SAS      L
0022500830            3           7.971667                 -9 FEB 23, 2026 HOU vs. UTA      W
0022500927            2          10.333333                 -9 MAR 08, 2026   HOU @ SAS      L
0

In [106]:
# Examine the MAR 29 NOP game in detail
TARGET = "0022501088"
print(f"Game: {TARGET}")
print(f"Info: {game_info[TARGET]}\n")

# Show KD's reconstructed timeline
print("KD timeline (reconstructed):")
tl = reconstruct_kd_timeline(TARGET)
for evt in tl:
    clock_s = evt["clock_sec"]
    mins = int(clock_s // 60)
    secs = clock_s - mins * 60
    period_label = f"Q{evt['period']}" if evt['period'] <= 4 else f"OT{evt['period']-4}"
    if evt.get("carryover"):
        marker = " (continued from previous quarter)"
    elif evt["implicit"]:
        marker = " (inferred at quarter break)"
    else:
        marker = ""
    print(f"  {period_label}  {mins:02d}:{secs:05.2f}  KD {evt['action']:3s}  |  margin: {evt['margin']:+d}{marker}")

print("\n--- Raw KD events in this game (any event mentioning Durant) ---")
game_pbp_this = pbp[pbp["gameId_str"] == TARGET]
durant_events = game_pbp_this[
    (game_pbp_this["playerName"] == "Durant") |
    (game_pbp_this["description"].str.contains("Durant", na=False, case=False))
]
print(durant_events[["period", "clock", "actionType", "playerName", "description"]].to_string(index=False))

Game: 0022501088
Info: {'GAME_DATE': 'MAR 29, 2026', 'MATCHUP': 'HOU @ NOP', 'WL': 'W', 'IS_HOME': False, 'PTS': 134}

KD timeline (reconstructed):
  Q1  12:00.00  KD IN   |  margin: +0 (inferred at quarter break)
  Q2  12:00.00  KD OUT  |  margin: +0 (inferred at quarter break)
  Q2  04:51.00  KD IN   |  margin: +20
  Q3  12:00.00  KD IN   |  margin: +21 (continued from previous quarter)
  Q3  00:04.70  KD OUT  |  margin: +20
  Q4  12:00.00  KD OUT  |  margin: +21 (continued from previous quarter)

--- Raw KD events in this game (any event mentioning Durant) ---
 period       clock   actionType playerName                                               description
      1 PT11M19.00S    Made Shot     Sengun               Sengun 16' Jump Shot (2 PTS) (Durant 1 AST)
      1 PT10M33.00S    Made Shot     Durant Durant 26' 3PT Running Jump Shot (3 PTS) (Thompson 1 AST)
      1 PT09M13.00S  Missed Shot     Durant                      MISS Durant 25' 3PT Pullup Jump Shot
      1 PT08M27.00S   

### Build `02e_kd_off_scatter.html` — Margin change across all KD-off stretches

In [403]:
# ============================================================
# Visualization: KD-Off Margin Change Across All Games (scatterplot)
# ============================================================
import plotly.graph_objects as go
from datetime import datetime

# Make sure per_game is sorted chronologically (not by margin_change anymore)
per_game_chron = per_game.copy()
# Parse dates so we can sort and use them as x-axis
per_game_chron["date_parsed"] = pd.to_datetime(per_game_chron["date"], format="%b %d, %Y")
per_game_chron = per_game_chron.sort_values("date_parsed").reset_index(drop=True)

# Clean opponent string for hover
def short_matchup(m):
    return m.replace("HOU vs. ", "vs ").replace("HOU @ ", "@ ")

per_game_chron["opp_short"] = per_game_chron["matchup"].apply(short_matchup)

# Colors
WIN_COLOR = "#4A8B5C"   # muted green
LOSS_COLOR = "#CE1141"  # Rockets red
ZERO_GRAY = "#9CA3AF"

# Build figure
fig = go.Figure()

# Add zero reference line
fig.add_hline(y=0, line=dict(color=ZERO_GRAY, width=1, dash="dot"))

# Wins
wins = per_game_chron[per_game_chron["result"] == "W"]
fig.add_trace(go.Scatter(
    x=wins["date_parsed"], y=wins["net_margin_change"],
    mode="markers",
    name="Win",
    marker=dict(
        size=wins["total_off_minutes"].clip(lower=4, upper=15),
        color=WIN_COLOR,
        opacity=0.65,
        line=dict(color="white", width=1),
    ),
    customdata=wins[["opp_short", "total_off_minutes", "n_stretches"]],
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "%{x|%b %d, %Y}<br>"
        "Margin change: %{y:+d}<br>"
        "KD off: %{customdata[1]:.1f} min (%{customdata[2]} stretches)<br>"
        "Result: Win<extra></extra>"
    ),
))

# Losses
losses = per_game_chron[per_game_chron["result"] == "L"]
fig.add_trace(go.Scatter(
    x=losses["date_parsed"], y=losses["net_margin_change"],
    mode="markers",
    name="Loss",
    marker=dict(
        size=losses["total_off_minutes"].clip(lower=4, upper=15),
        color=LOSS_COLOR,
        opacity=0.75,
        line=dict(color="white", width=1),
    ),
    customdata=losses[["opp_short", "total_off_minutes", "n_stretches"]],
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "%{x|%b %d, %Y}<br>"
        "Margin change: %{y:+d}<br>"
        "KD off: %{customdata[1]:.1f} min (%{customdata[2]} stretches)<br>"
        "Result: Loss<extra></extra>"
    ),
))

# Annotate the worst games (margin change <= -12)
worst_games = per_game_chron[per_game_chron["net_margin_change"] <= -12].copy()
for _, row in worst_games.iterrows():
    # Custom offset for specific games to avoid overlap
    if row["opp_short"] == "@ MIA":
        ax_offset = 60  # push right
    elif row["opp_short"] == "vs LAC":
        ax_offset = -50  # push left,
    else:
        ax_offset = 0
    
    fig.add_annotation(
        x=row["date_parsed"], y=row["net_margin_change"],
        text=f"<b>{row['opp_short']}</b><br><span style='font-size:10px'>{row['net_margin_change']:+d}</span>",
        showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1,
        ax=ax_offset, ay=30,
        font=dict(size=11, color="#1A1A1A"),
        bgcolor="white", bordercolor="#1A1A1A", borderwidth=1, borderpad=4,
        align="center",
    )

# Layout
fig.update_layout(
    height=450,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Inter, sans-serif", color="#1A1A1A"),
    margin=dict(l=60, r=30, t=30, b=50),
    legend=dict(
        orientation="h",
        yanchor="top", y=1.05,
        xanchor="right", x=1,
        bgcolor="rgba(255,255,255,0)",
    ),
    xaxis=dict(
        title=None,
        showgrid=False,
        zeroline=False,
        tickformat="%b",
        dtick="M1",
    ),
    yaxis=dict(
        title="Margin Change While KD Sat",
        zeroline=False,
        gridcolor="#F0EDE5",
        ticksuffix="",
    ),
)

# Save to HTML
plotly_html = fig.to_html(include_plotlyjs='cdn', full_html=True,
                          config={'displayModeBar': False})

wrapper = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>KD-Off Margin Change</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@500&display=swap" rel="stylesheet">
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    font-family: 'Inter', sans-serif;
    background: #FAFAF7;
    color: #1A1A1A;
    padding: 60px 40px;
  }
  .wrapper { max-width: 1100px; margin: 0 auto; }
  .table-label {
    font-family: 'JetBrains Mono', monospace;
    font-size: 11px;
    letter-spacing: 0.15em;
    text-transform: uppercase;
    color: #CE1141;
    margin-bottom: 8px;
  }
  h2 {
    font-family: 'Fraunces', Georgia, serif;
    font-size: 26px;
    font-weight: 600;
    letter-spacing: -0.015em;
    margin-bottom: 8px;
  }
  .subtitle {
    color: #6B7280;
    font-size: 15px;
    margin-bottom: 24px;
  }
  .chart-frame {
    background: white;
    border-radius: 8px;
    padding: 8px;
    box-shadow: 0 1px 3px rgba(0,0,0,0.04), 0 1px 2px rgba(0,0,0,0.06);
  }
  .legend-note {
    margin-top: 14px;
    font-size: 13px;
    color: #6B7280;
    font-style: italic;
  }
</style>
</head>
<body>
<div class="wrapper">
  <div class="table-label">Clutch Time, Section 02</div>
  <h2>When KD Sits, Across the Season</h2>
  <p class="subtitle">Across 78 games, Houston's margin while Kevin Durant was on the bench was neutral on average. But in a handful of games, the lead vanished, and those games were almost always losses.</p>
  <div class="chart-frame">
__CHART__
  </div>
  <p class="legend-note">Each dot represents one game. Dot size reflects total minutes Durant spent on the bench. Annotated games had a swing of 12 points or more. A "stretch" refers to one continuous on-off period within a game.</p>
</div>
</body>
</html>"""

import re
chart_div = re.search(r'<body>(.*?)</body>', plotly_html, re.DOTALL).group(1)
final = wrapper.replace("__CHART__", chart_div)

with open(ASSETS_DIR / "02d_kd_off_scatter.html", "w") as f:
    f.write(final)

print(f"Saved: {ASSETS_DIR / '02d_kd_off_scatter.html'}")
fig.show()

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/02d_kd_off_scatter.html


<a id="lineup-reconstruction"></a>
# Lineup Reconstruction (foundational work)

The output of this section (a per-game history of 5-man lineups across all 82 games) is used by Section 3 ("Wheels Came Off" — KD-off lineup performance), Section 4 (starting lineup + lineup ratings), and the team gravity comparison.

**Two phases:**
- **Phase 1 (abandoned)**: Try NBA Stats PBP. Discovers that NBA PBP omits quarter-break substitutions, making lineup reconstruction unreliable.
- **Phase 2 (production)**: Switch to ESPN's PBP feed, which includes quarter-break subs.

## Phase 1: NBA Stats PBP approach (exploratory, abandoned)

Tested on one game (NYK Feb 21) to validate the approach before scaling to all 82 games. Discovered that NBA PBP doesn't emit substitution events that happen during quarter breaks — so the reconstructed lineup drifts after Q1. Kept here for reference; moved to ESPN below.

In [110]:
# ============================================================
# Lineup reconstruction — single game test (NYK)
# ============================================================

NYK_GAME = "0022500816"  # Feb 21 NYK game

game_pbp = pbp[pbp["gameId_str"] == NYK_GAME].copy().reset_index(drop=True)
print(f"Total events: {len(game_pbp)}")

# Filter to Rockets events only (need this to find Rockets players)
# Rockets team_id is ROCKETS_ID
rockets_events = game_pbp[game_pbp["teamId"] == ROCKETS_ID]
print(f"Rockets events: {len(rockets_events)}")

# Find Rockets starters: players who took actions in Q1 BEFORE the first Rockets substitution
q1_rockets = rockets_events[rockets_events["period"] == 1].sort_values("actionNumber")

# First Rockets substitution in Q1
first_sub_idx = q1_rockets[q1_rockets["actionType"] == "Substitution"].index.min()
print(f"First Rockets substitution at index: {first_sub_idx}")

# Anyone with an action before that is a starter
before_first_sub = q1_rockets[q1_rockets.index < first_sub_idx]
starters_from_actions = before_first_sub[
    before_first_sub["playerName"].notna() & (before_first_sub["playerName"] != "")
]["playerName"].unique().tolist()
print(f"Players with actions before first sub: {starters_from_actions}")

# Also check the first substitution itself — the OUT player must have been a starter
first_sub_event = game_pbp.loc[first_sub_idx]
print(f"\nFirst sub event: {first_sub_event['description']}")
print(f"  Player going OUT (= starter): {first_sub_event['playerName']}")

Total events: 481
Rockets events: 227
First Rockets substitution at index: 51
Players with actions before first sub: ['Sengun', 'Thompson', 'Durant', 'Eason', 'Smith Jr.']

First sub event: SUB: Capela FOR Sengun
  Player going OUT (= starter): Sengun


In [111]:
# ============================================================
# Full lineup reconstruction for NYK game
# ============================================================

# Step 1: Identify the 5 starters
# Combine: players with actions before first sub + the OUT player on the first sub
starters = set(starters_from_actions)
starters.add(first_sub_event['playerName'])

# But we might have MORE than 5 if some player came in via the first sub event AND took an action right after.
# To be safe: take only the first 5 unique action-takers
# Actually we already filter to "before_first_sub" so this should be fine.
# But starters_from_actions could include 4 or fewer if some starter had no action

# Show what we found
print(f"Identified Rockets starters: {sorted(starters)}")
print(f"Count: {len(starters)}")

if len(starters) != 5:
    print(f"\n⚠️ Warning: expected 5 starters, got {len(starters)}")
    print("Need to investigate this game")

Identified Rockets starters: ['Durant', 'Eason', 'Sengun', 'Smith Jr.', 'Thompson']
Count: 5


In [112]:
# Step 2: Walk through every Rockets substitution and update lineup
# Parse sub events: description is "SUB: <IN> FOR <OUT>"
import re

def parse_sub(description):
    """Extract (player_in, player_out) from a substitution description."""
    m = re.match(r"SUB:\s+(.+?)\s+FOR\s+(.+)", description)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return None, None

# Initialize lineup at start of game
current_lineup = set(starters)
lineup_log = []  # records: (event_idx, period, clock, lineup_set)

# Snapshot at game start
lineup_log.append({
    "event_idx": -1,
    "period": 1,
    "clock": "PT12M00.00S",
    "action": "GAME_START",
    "lineup": frozenset(current_lineup),
})

# Walk through Rockets substitutions in chronological order
rockets_subs = rockets_events[rockets_events["actionType"] == "Substitution"].sort_values("actionNumber")
print(f"Total Rockets substitutions: {len(rockets_subs)}")

issues = []
for idx, sub in rockets_subs.iterrows():
    player_in, player_out = parse_sub(sub["description"])
    if player_in is None:
        issues.append(f"Couldn't parse: {sub['description']}")
        continue
    
    # Apply the substitution
    if player_out not in current_lineup:
        issues.append(f"Period {sub['period']} {sub['clock']}: {player_out} subbed OUT but not in lineup. Current: {sorted(current_lineup)}")
    if player_in in current_lineup:
        issues.append(f"Period {sub['period']} {sub['clock']}: {player_in} subbed IN but already in lineup")
    
    current_lineup.discard(player_out)
    current_lineup.add(player_in)
    
    if len(current_lineup) != 5:
        issues.append(f"Period {sub['period']} {sub['clock']}: lineup size = {len(current_lineup)}, expected 5. {sorted(current_lineup)}")
    
    lineup_log.append({
        "event_idx": idx,
        "period": sub["period"],
        "clock": sub["clock"],
        "action": sub["description"],
        "lineup": frozenset(current_lineup),
    })

print(f"\nIssues found: {len(issues)}")
for i in issues[:10]:
    print(f"  {i}")

print(f"\nFirst 5 lineup states:")
for log in lineup_log[:5]:
    print(f"  P{log['period']} {log['clock']}: {sorted(log['lineup'])}")

print(f"\nLast 3 lineup states:")
for log in lineup_log[-3:]:
    print(f"  P{log['period']} {log['clock']}: {sorted(log['lineup'])}")

Total Rockets substitutions: 28

Issues found: 19
  Period 2 PT08M54.00S: Smith Jr. subbed IN but already in lineup
  Period 2 PT08M54.00S: lineup size = 4, expected 5. ['Durant', 'Eason', 'Sheppard', 'Smith Jr.']
  Period 2 PT08M54.00S: Finney-Smith subbed OUT but not in lineup. Current: ['Durant', 'Eason', 'Sheppard', 'Smith Jr.']
  Period 2 PT08M54.00S: Eason subbed IN but already in lineup
  Period 2 PT08M54.00S: lineup size = 4, expected 5. ['Durant', 'Eason', 'Sheppard', 'Smith Jr.']
  Period 2 PT07M17.00S: Durant subbed IN but already in lineup
  Period 2 PT07M17.00S: lineup size = 3, expected 5. ['Durant', 'Eason', 'Smith Jr.']
  Period 2 PT07M17.00S: Tate subbed OUT but not in lineup. Current: ['Durant', 'Eason', 'Smith Jr.']
  Period 2 PT07M17.00S: lineup size = 4, expected 5. ['Durant', 'Eason', 'Smith Jr.', 'Thompson']
  Period 2 PT00M14.70S: Sengun subbed OUT but not in lineup. Current: ['Durant', 'Eason', 'Smith Jr.', 'Thompson']

First 5 lineup states:
  P1 PT12M00.00S: 

In [113]:
# Dump all Rockets subs in the NYK game with context
NYK_GAME = "0022500816"
game_pbp = pbp[pbp["gameId_str"] == NYK_GAME].copy()
rockets_subs_all = game_pbp[
    (game_pbp["actionType"] == "Substitution") &
    (game_pbp["teamId"] == ROCKETS_ID)
].sort_values(["period", "actionNumber"])

print("All Rockets substitutions in NYK game:")
print(rockets_subs_all[["actionNumber", "period", "clock", "playerName", "description"]].to_string(index=False))

All Rockets substitutions in NYK game:
 actionNumber  period       clock   playerName                     description
           71       1 PT06M07.00S       Sengun          SUB: Capela FOR Sengun
           94       1 PT05M03.00S     Thompson      SUB: Sheppard FOR Thompson
          109       1 PT03M50.00S    Smith Jr. SUB: Finney-Smith FOR Smith Jr.
          110       1 PT03M50.00S        Eason         SUB: Thompson FOR Eason
          154       1 PT00M06.50S       Durant            SUB: Tate FOR Durant
          155       1 PT00M06.50S     Sheppard     SUB: Smith Jr. FOR Sheppard
          160       1 PT00M06.50S       Capela           SUB: Eason FOR Capela
          168       1 PT00M04.60S Finney-Smith  SUB: Sheppard FOR Finney-Smith
          172       1 PT00M04.60S         Tate            SUB: Durant FOR Tate
          222       2 PT08M54.00S     Thompson     SUB: Smith Jr. FOR Thompson
          223       2 PT08M54.00S Finney-Smith     SUB: Eason FOR Finney-Smith
          241

## Phase 2: ESPN API (production)

ESPN's PBP feed records every substitution including those that happen during quarter breaks. This lets us reconstruct accurate 5-man lineups across the entire game.

### Constructing lineup at the start and end of each quarter (NYK test game)

In [129]:
# ============================================================
# ESPN API for PBP data
# 
# ESPN's PBP feed includes substitutions made during quarter breaks,
# which NBA Stats' PBP omits. This lets us reconstruct 5-man lineups
# accurately across the game.
# ============================================================

import requests
import json
import re

# Test game: HOU @ NYK, Feb 21 2026
ESPN_GAME_ID = "401810671"
url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/summary?event={ESPN_GAME_ID}"

response = requests.get(url)
print(f"Status: {response.status_code}")

data = response.json()
plays = data["plays"]
boxscore = data["boxscore"]

print(f"Total plays: {len(plays)}")
print(f"Top-level keys: {list(data.keys())}")

Status: 200
Total plays: 470
Top-level keys: ['boxscore', 'format', 'gameInfo', 'leaders', 'seasonseries', 'injuries', 'broadcasts', 'pickcenter', 'odds', 'againstTheSpread', 'news', 'article', 'videos', 'header', 'winprobability', 'plays', 'wallclockAvailable', 'meta', 'standings']


In [137]:
# Look for substitution events in ESPN's PBP
plays = data["plays"]

# How many plays have "Substitution" or "Subs" in the type or text?
print("=" * 70)
print("Searching for substitution-related plays")
print("=" * 70)

# Check all unique play types first
play_types = set()
for p in plays:
    if isinstance(p.get("type"), dict):
        play_types.add(p["type"].get("text", ""))
print(f"\nAll unique play types found:")
for pt in sorted(play_types):
    print(f"  - {pt}")

# Now specifically search for subs in text
print("\n" + "=" * 70)
print("Plays mentioning 'sub' or 'enters' in text:")
print("=" * 70)
for p in plays:
    text = p.get("text", "").lower()
    if "sub" in text or "enters" in text or "for" in text and "fouls" not in text:
        # Show first ones
        print(f"  Q{p['period']['number']} {p['clock']['displayValue']}: {p.get('text', '')}")

Searching for substitution-related plays

All unique play types found:
  - Alley Oop Dunk Shot
  - Back Court Turnover
  - Bad Pass
Turnover
  - Coach's Challenge (Overturned)
  - Coach's Challenge (Supported)
  - Cutting Dunk Shot
  - Cutting Finger Roll Layup Shot
  - Defensive Rebound
  - Driving Dunk Shot
  - Driving Finger Roll Layup
  - Driving Floating Bank Jump Shot
  - Driving Floating Jump Shot
  - Driving Hook Shot
  - Driving Layup Shot
  - Dunk Shot
  - End Game
  - End Period
  - Fade Away Jump Shot
  - Floating Jump Shot
  - Free Throw - 1 of 1
  - Free Throw - 1 of 2
  - Free Throw - 2 of 2
  - Free Throw - Technical
  - Full Timeout
  - Heave Jump Shot
  - Hook Shot
  - Jump Shot
  - Jump Shot Bank
  - Jumpball
  - Kicked Ball
  - Layup Driving Reverse
  - Layup Shot
  - Layup Shot Putback
  - Loose Ball Foul
  - Lost Ball Turnover
  - Offensive Charge
  - Offensive Foul
  - Offensive Foul Turnover
  - Offensive Rebound
  - Out of Bounds - Bad Pass Turnover
  - Out of 

In [130]:
# ============================================================
# Build maps: athlete_id -> name, and per-team rosters
# ============================================================

HOU_ESPN_ID = "10"  # Houston Rockets team_id in ESPN data

id_to_name = {}
hou_player_ids = set()
opp_player_ids = set()

for team_block in boxscore["players"]:
    team_id = str(team_block.get("team", {}).get("id", ""))
    for stat_group in team_block.get("statistics", []):
        for player in stat_group.get("athletes", []):
            athlete = player.get("athlete", {})
            pid = str(athlete["id"])
            id_to_name[pid] = athlete.get("displayName", "Unknown")
            if team_id == HOU_ESPN_ID:
                hou_player_ids.add(pid)
            else:
                opp_player_ids.add(pid)

print(f"Total players mapped: {len(id_to_name)}")
print(f"Houston players: {len(hou_player_ids)}")
print(f"Opponent players: {len(opp_player_ids)}")

Total players mapped: 28
Houston players: 13
Opponent players: 15


In [164]:
# ============================================================
# Extract all Rockets substitution events in chronological order
# ============================================================

def parse_sub_text(text):
    """Parse 'X enters the game for Y' -> (X, Y)."""
    m = re.match(r"(.+?) enters the game for (.+)", text)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return None, None


hou_subs = []
for p in plays:
    if not isinstance(p.get("type"), dict):
        continue
    if p["type"].get("text") != "Substitution":
        continue
    if str(p.get("team", {}).get("id")) != HOU_ESPN_ID:
        continue
    
    player_in, player_out = parse_sub_text(p.get("text", ""))
    if player_in is None:
        continue
    
    hou_subs.append({
        "period": p["period"]["number"],
        "clock": p["clock"]["displayValue"],
        "sequence": int(p["sequenceNumber"]),
        "player_in": player_in,
        "player_out": player_out,
    })

def sub_elapsed(s):
    """Compute elapsed seconds for a sub for chronological sorting."""
    period = s["period"]
    clock_remaining = parse_clock_to_sec(s["clock"])
    period_length = 720 if period <= 4 else 300
    in_period = period_length - clock_remaining
    prev = sum(720 if p <= 4 else 300 for p in range(1, period))
    return prev + in_period

hou_subs.sort(key=lambda s: (sub_elapsed(s), s["sequence"]))
print(f"Total Houston substitutions: {len(hou_subs)}")

Total Houston substitutions: 30


In [165]:
print(f"Total Houston substitutions: {len(hou_subs)}")
print(f"\nFirst 10 subs:")
for s in hou_subs[:20]:
    print(f"  Q{s['period']} {s['clock']:>5}: {s['player_in']:<22} IN for {s['player_out']:<22} OUT")

Total Houston substitutions: 30

First 10 subs:
  Q1  7:27: Tari Eason             IN for Amen Thompson          OUT
  Q1  6:16: Steven Adams           IN for Alperen Sengun         OUT
  Q1  3:35: Reed Sheppard          IN for Jabari Smith Jr.       OUT
  Q1  2:33: Alperen Sengun         IN for Kevin Durant           OUT
  Q1  2:33: Amen Thompson          IN for Josh Okogie            OUT
  Q2 12:00: Josh Okogie            IN for Steven Adams           OUT
  Q2 12:00: Clint Capela           IN for Amen Thompson          OUT
  Q2  9:03: Kevin Durant           IN for Alperen Sengun         OUT
  Q2  7:05: Amen Thompson          IN for Reed Sheppard          OUT
  Q2  7:05: Jabari Smith Jr.       IN for Tari Eason             OUT
  Q2  6:08: Alperen Sengun         IN for Josh Okogie            OUT
  Q2  6:08: Steven Adams           IN for Clint Capela           OUT
  Q2  2:11: Josh Okogie            IN for Kevin Durant           OUT
  Q2  2:11: Tari Eason             IN for Steven Adams 

In [132]:
# ============================================================
# Identify the 5 Houston starters from early-Q1 actions
# ============================================================

first_sub_seq = hou_subs[0]["sequence"]
q1_houston_actors = set()

for p in plays:
    if int(p["sequenceNumber"]) >= first_sub_seq:
        break
    if p["period"]["number"] != 1:
        continue
    for participant in p.get("participants", []):
        athlete_id = str(participant.get("athlete", {}).get("id", ""))
        if athlete_id in hou_player_ids and athlete_id in id_to_name:
            q1_houston_actors.add(id_to_name[athlete_id])

# Include the OUT player of the first sub (catches starters with no early action)
q1_houston_actors.add(hou_subs[0]["player_out"])

starters = q1_houston_actors
print(f"Starters ({len(starters)}): {sorted(starters)}")
assert len(starters) == 5, f"Expected 5 starters, got {len(starters)}"

Starters (5): ['Alperen Sengun', 'Amen Thompson', 'Jabari Smith Jr.', 'Kevin Durant', 'Tari Eason']


In [133]:
# ============================================================
# Walk through subs maintaining 5-man lineup,
# inferring missing quarter-break subs from early-period actions
# ============================================================

def get_actors_in_early_period(plays, period, threshold_seq, hou_player_ids, id_to_name):
    """Return set of Houston player names with any action in `period` before `threshold_seq`."""
    actors = set()
    for p in plays:
        if int(p["sequenceNumber"]) >= threshold_seq:
            continue
        if p["period"]["number"] != period:
            continue
        for participant in p.get("participants", []):
            athlete_id = str(participant.get("athlete", {}).get("id", ""))
            if athlete_id in hou_player_ids and athlete_id in id_to_name:
                actors.add(id_to_name[athlete_id])
    return actors


def reconstruct_lineup(starters, hou_subs, plays, hou_player_ids, id_to_name):
    """Walk through subs in order, inferring any missing quarter-break subs."""
    current = set(starters)
    history = [{
        "period": 1, "clock": "12:00", "event": "GAME START",
        "lineup": frozenset(current), "inferred": False,
    }]
    
    last_period = 1
    for s in hou_subs:
        # Crossed into a new period? Check for missing quarter-break subs
        if s["period"] > last_period:
            for missed_period in range(last_period + 1, s["period"] + 1):
                first_sub = next((sub for sub in hou_subs if sub["period"] == missed_period), None)
                threshold = first_sub["sequence"] if first_sub else float("inf")
                
                actors = get_actors_in_early_period(plays, missed_period, threshold, hou_player_ids, id_to_name)
                if first_sub:
                    actors.add(first_sub["player_out"])
                
                inferred_in = sorted(actors - current)
                inferred_out = sorted(current - actors)
                
                for player_in, player_out in zip(inferred_in, inferred_out):
                    current.discard(player_out)
                    current.add(player_in)
                    history.append({
                        "period": missed_period, "clock": "12:00",
                        "event": f"{player_in} IN for {player_out} (inferred)",
                        "lineup": frozenset(current), "inferred": True,
                    })
        
        last_period = s["period"]
        current.discard(s["player_out"])
        current.add(s["player_in"])
        history.append({
            "period": s["period"], "clock": s["clock"],
            "event": f"{s['player_in']} IN for {s['player_out']}",
            "lineup": frozenset(current), "inferred": False,
        })
    
    return history


lineup_history = reconstruct_lineup(starters, hou_subs, plays, hou_player_ids, id_to_name)
print(f"Total lineup events: {len(lineup_history)}")

Total lineup events: 34


In [135]:
# ============================================================
# Verify: show lineup at start and end of each quarter
# ============================================================

print("Lineup transitions across the game:\n")

prev_end_lineup = None
for period in sorted(set(h["period"] for h in lineup_history)):
    in_period = [h for h in lineup_history if h["period"] == period]
    if not in_period:
        continue
    
    first = in_period[0]
    if first["event"] == "GAME START":
        start_lineup = first["lineup"]
    elif first["clock"] == "12:00":
        cluster = [h for h in in_period if h["clock"] == "12:00"]
        start_lineup = cluster[-1]["lineup"]
    else:
        start_lineup = prev_end_lineup
    
    end_lineup = in_period[-1]["lineup"]
    prev_end_lineup = end_lineup
    
    print(f"Q{period}:")
    print(f"  Start: {sorted(start_lineup)}")
    print(f"  End  : {sorted(end_lineup)}")
    print()

Lineup transitions across the game:

Q1:
  Start: ['Alperen Sengun', 'Amen Thompson', 'Jabari Smith Jr.', 'Kevin Durant', 'Tari Eason']
  End  : ['Amen Thompson', 'Jabari Smith Jr.', 'Kevin Durant', 'Reed Sheppard', 'Tari Eason']

Q2:
  Start: ['Alperen Sengun', 'Amen Thompson', 'Dorian Finney-Smith', "Jae'Sean Tate", 'Reed Sheppard']
  End  : ['Alperen Sengun', 'Amen Thompson', 'Jabari Smith Jr.', 'Kevin Durant', 'Tari Eason']

Q3:
  Start: ['Alperen Sengun', 'Amen Thompson', 'Jabari Smith Jr.', 'Kevin Durant', 'Tari Eason']
  End  : ['Amen Thompson', 'Clint Capela', 'Dorian Finney-Smith', 'Kevin Durant', 'Reed Sheppard']

Q4:
  Start: ['Alperen Sengun', 'Amen Thompson', 'Dorian Finney-Smith', 'Jabari Smith Jr.', 'Reed Sheppard']
  End  : ['Alperen Sengun', 'Amen Thompson', 'Jabari Smith Jr.', 'Kevin Durant', 'Reed Sheppard']



### Pull ESPN PBP for all 82 Rockets games

Build a mapping from NBA game_id to ESPN game_id, then fetch and cache the ESPN PBP for every Rockets game.

In [138]:
# Test: fetch ESPN's scoreboard for one date
test_date = "20251024"  # Approximate NBA opening night
url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard?dates={test_date}"

response = requests.get(url)
data = response.json()

print(f"Top-level keys: {list(data.keys())}")
print(f"\nEvents for {test_date}: {len(data.get('events', []))}")

# Show structure of one event
if data.get("events"):
    sample = data["events"][0]
    print(f"\nFirst event keys: {list(sample.keys())}")
    print(f"\nID: {sample.get('id')}")
    print(f"Date: {sample.get('date')}")
    print(f"Name: {sample.get('name')}")
    # Look at competitions for team info
    comp = sample.get("competitions", [{}])[0]
    teams_in_game = [c["team"]["abbreviation"] for c in comp.get("competitors", [])]
    print(f"Teams: {teams_in_game}")

Top-level keys: ['leagues', 'events', 'provider']

Events for 20251024: 12

First event keys: ['id', 'uid', 'date', 'name', 'shortName', 'season', 'competitions', 'links', 'status']

ID: 401809946
Date: 2025-10-24T22:30Z
Name: Milwaukee Bucks at Toronto Raptors
Teams: ['TOR', 'MIL']


In [139]:
# ============================================================
# Build mapping: NBA game_id -> ESPN game_id for all Rockets games
# ============================================================
from datetime import datetime
from tqdm.auto import tqdm

# Get all Rockets game dates from our gamelogs
rockets_games = team_gl_2526[["GAME_ID_STR", "GAME_DATE", "MATCHUP"]].copy()

# Convert MMM DD, YYYY -> YYYYMMDD
rockets_games["espn_date"] = pd.to_datetime(rockets_games["GAME_DATE"], format="%b %d, %Y").dt.strftime("%Y%m%d")

print(f"Dates to look up: {rockets_games['espn_date'].nunique()} unique dates")

# Cache scoreboard pulls
scoreboard_cache = {}

def fetch_scoreboard(date_str):
    if date_str in scoreboard_cache:
        return scoreboard_cache[date_str]
    url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard?dates={date_str}"
    r = requests.get(url)
    if r.status_code != 200:
        return None
    scoreboard_cache[date_str] = r.json()
    return scoreboard_cache[date_str]


# Build the mapping
nba_to_espn = {}
unmatched = []

for _, row in tqdm(rockets_games.iterrows(), total=len(rockets_games), desc="Matching games"):
    nba_gid = row["GAME_ID_STR"]
    espn_date = row["espn_date"]
    
    sb = fetch_scoreboard(espn_date)
    if sb is None:
        unmatched.append((nba_gid, espn_date, "fetch failed"))
        continue
    
    # Find the event where Houston is one of the teams
    for event in sb.get("events", []):
        comp = event.get("competitions", [{}])[0]
        teams = [c.get("team", {}).get("abbreviation", "") for c in comp.get("competitors", [])]
        if "HOU" in teams:
            nba_to_espn[nba_gid] = event["id"]
            break
    else:
        unmatched.append((nba_gid, espn_date, "no HOU game found"))

print(f"\nMapped: {len(nba_to_espn)} games")
print(f"Unmatched: {len(unmatched)}")
if unmatched:
    print("First few unmatched:")
    for u in unmatched[:5]:
        print(f"  {u}")

Dates to look up: 82 unique dates


Matching games:   0%|          | 0/82 [00:00<?, ?it/s]


Mapped: 82 games
Unmatched: 0


In [140]:
# ============================================================
# Fetch ESPN PBP for all 82 Rockets games (cached)
# ============================================================
import time

ESPN_CACHE_DIR = PROJECT_ROOT / "data" / "raw" / "espn"
ESPN_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def fetch_espn_game(espn_game_id):
    """Fetch one game's full ESPN PBP, with disk caching."""
    cache_path = ESPN_CACHE_DIR / f"{espn_game_id}.json"
    if cache_path.exists():
        with open(cache_path) as f:
            return json.load(f)
    
    url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/summary?event={espn_game_id}"
    r = requests.get(url)
    if r.status_code != 200:
        return None
    
    data = r.json()
    with open(cache_path, "w") as f:
        json.dump(data, f)
    return data


# Pull all 82 games
espn_data = {}
failures = []

for nba_gid, espn_gid in tqdm(nba_to_espn.items(), desc="Fetching ESPN PBP"):
    data = fetch_espn_game(espn_gid)
    if data is None:
        failures.append((nba_gid, espn_gid))
        continue
    espn_data[nba_gid] = data
    time.sleep(0.3)  # polite rate limit on first pull, instant on cached re-runs

print(f"\nLoaded {len(espn_data)} games from ESPN")
print(f"Failures: {len(failures)}")

Fetching ESPN PBP:   0%|          | 0/82 [00:00<?, ?it/s]


Loaded 82 games from ESPN
Failures: 0


### Generalize the lineup reconstruction across all 82 games

In [166]:
# ============================================================
# Generalize lineup reconstruction across all 82 games
# ============================================================

def reconstruct_lineups_for_game(espn_data):
    """
    Given one game's ESPN data, return a list of lineup events for Houston.
    Returns None if reconstruction fails (e.g., wrong number of starters).
    """
    plays = espn_data["plays"]
    boxscore = espn_data["boxscore"]
    
    # Build player ID maps
    id_to_name = {}
    hou_player_ids = set()
    for team_block in boxscore["players"]:
        team_id = str(team_block.get("team", {}).get("id", ""))
        for stat_group in team_block.get("statistics", []):
            for player in stat_group.get("athletes", []):
                athlete = player.get("athlete", {})
                pid = str(athlete["id"])
                id_to_name[pid] = athlete.get("displayName", "Unknown")
                if team_id == HOU_ESPN_ID:
                    hou_player_ids.add(pid)
    
    # Extract Houston subs
    hou_subs = []
    for p in plays:
        if not isinstance(p.get("type"), dict):
            continue
        if p["type"].get("text") != "Substitution":
            continue
        if str(p.get("team", {}).get("id")) != HOU_ESPN_ID:
            continue
        pi, po = parse_sub_text(p.get("text", ""))
        if pi is None:
            continue
        hou_subs.append({
            "period": p["period"]["number"],
            "clock": p["clock"]["displayValue"],
            "sequence": int(p["sequenceNumber"]),
            "player_in": pi,
            "player_out": po,
        })
    hou_subs.sort(key=lambda s: (sub_elapsed(s), s["sequence"]))
    
    if not hou_subs:
        return None  # no subs, can't determine lineup beyond starters
    
    # Identify starters: walk through Q1, collect Houston players in order of first appearance.
    # A player is a starter if their first event isn't a sub-IN.
    starters = set()
    subbed_in = set()  # players we've seen sub IN (NOT starters)
    
    for p in plays:
        if p["period"]["number"] != 1:
            continue
        
        # Is this a substitution?
        is_sub = isinstance(p.get("type"), dict) and p["type"].get("text") == "Substitution"
        is_hou_sub = is_sub and str(p.get("team", {}).get("id")) == HOU_ESPN_ID
        
        if is_hou_sub:
            pi, po = parse_sub_text(p.get("text", ""))
            if pi is not None:
                # Player coming IN is NOT a starter (unless we already identified them as one)
                if pi not in starters:
                    subbed_in.add(pi)
                # Player going OUT IS a starter (if not already identified as a substitute)
                if po not in subbed_in:
                    starters.add(po)
        else:
            # Non-sub event: any Houston player participant is on the court
            for participant in p.get("participants", []):
                aid = str(participant.get("athlete", {}).get("id", ""))
                if aid in hou_player_ids and aid in id_to_name:
                    name = id_to_name[aid]
                    if name not in subbed_in:
                        starters.add(name)
        
        if len(starters) >= 5:
            break
    
    if len(starters) != 5:
        return None
    
    # Run the walker
    history = reconstruct_lineup(starters, hou_subs, plays, hou_player_ids, id_to_name)
    return history


# Run across all 82 games
all_lineups = {}
recon_failures = []

for nba_gid, data in tqdm(espn_data.items(), desc="Reconstructing lineups"):
    history = reconstruct_lineups_for_game(data)
    if history is None:
        recon_failures.append(nba_gid)
        continue
    all_lineups[nba_gid] = history

print(f"\nSuccessfully reconstructed lineups for {len(all_lineups)} games")
print(f"Failures: {len(recon_failures)}")
if recon_failures:
    print("Failed game IDs (need manual investigation):")
    for gid in recon_failures[:10]:
        info = game_info.get(gid, {})
        print(f"  {gid}: {info.get('GAME_DATE')} {info.get('MATCHUP')}")

Reconstructing lineups:   0%|          | 0/82 [00:00<?, ?it/s]


Successfully reconstructed lineups for 82 games
Failures: 0


### Compute 5-man lineup stints from the reconstructed lineups

(Note: the `Phase 1` label inside the code below refers to a sub-step of this stints computation, not Phase 1 vs. Phase 2 above.)

In [179]:
# ============================================================
# Phase 1: Compute 5-man lineup stints from ESPN PBP
# ============================================================

def parse_clock_to_sec(clock_str):
    """Parse '12:00', '5:23', '48.4', '6.5' etc into seconds remaining in period."""
    if ":" in clock_str:
        m, s = clock_str.split(":")
        return int(m) * 60 + float(s)
    else:
        # Just seconds (e.g., '48.4')
        return float(clock_str)


def event_elapsed_seconds(period, clock_str):
    """Total elapsed seconds since start of game."""
    period_length = 720 if period <= 4 else 300  # OT = 5 minutes
    secs_remaining = parse_clock_to_sec(clock_str)
    # Elapsed in this period
    in_period = period_length - secs_remaining
    # Plus all previous periods
    prev_periods_total = 0
    for p in range(1, period):
        prev_periods_total += 720 if p <= 4 else 300
    return prev_periods_total + in_period


def compute_lineup_stints(espn_data, lineup_history):
    plays = espn_data["plays"]
    
    snapshots = []
    for idx, h in enumerate(lineup_history):
        if h["event"] == "GAME START":
            elapsed = 0
        else:
            elapsed = event_elapsed_seconds(h["period"], h["clock"])
        snapshots.append({
            "elapsed": elapsed,
            "lineup": h["lineup"],
            "history_idx": idx,         # NEW: track source index
            "period": h["period"],      # NEW
            "clock": h["clock"],        # NEW
        })
    
    teams = espn_data["boxscore"]["teams"]
    hou_is_home = None
    for t in teams:
        if str(t.get("team", {}).get("id")) == HOU_ESPN_ID:
            hou_is_home = t.get("homeAway") == "home"
            break
    
    score_events = []
    for p in plays:
        try:
            elapsed = event_elapsed_seconds(p["period"]["number"], p["clock"]["displayValue"])
            score_events.append({
                "elapsed": elapsed,
                "home": int(p.get("homeScore", 0)),
                "away": int(p.get("awayScore", 0)),
            })
        except Exception:
            continue
    score_events.sort(key=lambda e: e["elapsed"])
    if not score_events:
        return []
    game_end = score_events[-1]["elapsed"]
    
    def score_at(elapsed):
        last = {"home": 0, "away": 0}
        for e in score_events:
            if e["elapsed"] > elapsed:
                break
            last = {"home": e["home"], "away": e["away"]}
        return last
    
    stints = []
    for i, snap in enumerate(snapshots):
        start = snap["elapsed"]
        end = snapshots[i + 1]["elapsed"] if i + 1 < len(snapshots) else game_end
        if end <= start:
            continue
        
        score_start = score_at(start)
        score_end = score_at(end)
        hou_pts = (score_end["home"] - score_start["home"]) if hou_is_home else (score_end["away"] - score_start["away"])
        opp_pts = (score_end["away"] - score_start["away"]) if hou_is_home else (score_end["home"] - score_start["home"])
        
        stints.append({
            "lineup": snap["lineup"],
            "duration_sec": end - start,
            "hou_pts": hou_pts,
            "opp_pts": opp_pts,
            "plus_minus": hou_pts - opp_pts,
            "period": snap["period"],       # NEW: when stint started
            "clock": snap["clock"],         # NEW
        })
    
    return stints


# Test on one game first
TEST_GAME = "0022500816"  # NYK
test_stints = compute_lineup_stints(espn_data[TEST_GAME], all_lineups[TEST_GAME])

print(f"NYK game: {len(test_stints)} lineup stints")
print(f"Total duration: {sum(s['duration_sec'] for s in test_stints) / 60:.1f} minutes")
print(f"Total Hou points: {sum(s['hou_pts'] for s in test_stints)}")
print(f"Total Opp points: {sum(s['opp_pts'] for s in test_stints)}")
print(f"Final plus/minus: {sum(s['plus_minus'] for s in test_stints)}")

NYK game: 21 lineup stints
Total duration: 48.0 minutes
Total Hou points: 106
Total Opp points: 108
Final plus/minus: -2


### Aggregate stints across all games + per-lineup totals

In [180]:
# ============================================================
# Compute stints across all games + aggregate
# ============================================================

all_stints = []
for nba_gid, history in tqdm(all_lineups.items(), desc="Computing stints"):
    espn = espn_data[nba_gid]
    stints = compute_lineup_stints(espn, history)
    for s in stints:
        s["game_id"] = nba_gid
        all_stints.append(s)

print(f"Total stints: {len(all_stints)}")

print(f"Total stints across all games: {len(all_stints)}")
print(f"Total duration: {sum(s['duration_sec'] for s in all_stints) / 60:.0f} minutes")
print(f"Expected (~82 games × 48 min): {82 * 48} minutes")

Computing stints:   0%|          | 0/82 [00:00<?, ?it/s]

Total stints: 1921
Total stints across all games: 1921
Total duration: 3981 minutes
Expected (~82 games × 48 min): 3936 minutes


In [147]:
# ============================================================
# Aggregate stints by unique 5-man lineup
# ============================================================
from collections import defaultdict

# Aggregator
lineup_agg = defaultdict(lambda: {
    "duration_sec": 0,
    "hou_pts": 0,
    "opp_pts": 0,
    "plus_minus": 0,
    "stints": 0,
})

for s in all_stints:
    key = s["lineup"]  # frozenset of 5 player names
    lineup_agg[key]["duration_sec"] += s["duration_sec"]
    lineup_agg[key]["hou_pts"] += s["hou_pts"]
    lineup_agg[key]["opp_pts"] += s["opp_pts"]
    lineup_agg[key]["plus_minus"] += s["plus_minus"]
    lineup_agg[key]["stints"] += 1

# Convert to dataframe
rows = []
for lineup, agg in lineup_agg.items():
    minutes = agg["duration_sec"] / 60
    rows.append({
        "lineup": tuple(sorted(lineup)),
        "kd_in": "Kevin Durant" in lineup,
        "minutes": round(minutes, 1),
        "stints": agg["stints"],
        "pts_for": agg["hou_pts"],
        "pts_against": agg["opp_pts"],
        "plus_minus": agg["plus_minus"],
        "pts_for_per_min": round(agg["hou_pts"] / minutes, 2) if minutes > 0 else 0,
        "pts_against_per_min": round(agg["opp_pts"] / minutes, 2) if minutes > 0 else 0,
        "plus_minus_per_36": round(agg["plus_minus"] / minutes * 36, 1) if minutes > 0 else 0,
    })

lineup_df = pd.DataFrame(rows)

# Filter: minimum 5 minutes
lineup_df_filtered = lineup_df[lineup_df["minutes"] >= 5].copy()

print(f"Total unique 5-man lineups: {len(lineup_df)}")
print(f"Lineups with 5+ minutes: {len(lineup_df_filtered)}")

# Split KD in vs KD out
kd_in = lineup_df_filtered[lineup_df_filtered["kd_in"]].sort_values("minutes", ascending=False)
kd_out = lineup_df_filtered[~lineup_df_filtered["kd_in"]].sort_values("minutes", ascending=False)

print(f"\n--- KD-IN lineups (5+ min): {len(kd_in)} ---")
print(f"Total KD-on minutes: {kd_in['minutes'].sum():.0f}")
print(f"Net plus/minus: {kd_in['plus_minus'].sum():+d}")

print(f"\n--- KD-OUT lineups (5+ min): {len(kd_out)} ---")
print(f"Total KD-off minutes: {kd_out['minutes'].sum():.0f}")
print(f"Net plus/minus: {kd_out['plus_minus'].sum():+d}")

print("\n=== TOP 10 KD-IN LINEUPS (by minutes) ===")
for _, row in kd_in.head(10).iterrows():
    others = [p for p in row["lineup"] if p != "Kevin Durant"]
    print(f"  KD + {' / '.join(others)}")
    print(f"    {row['minutes']:.1f} min · {row['plus_minus']:+d} +/- · {row['plus_minus_per_36']:+.1f} per 36")

print("\n=== TOP 10 KD-OUT LINEUPS (by minutes) ===")
for _, row in kd_out.head(10).iterrows():
    print(f"  {' / '.join(row['lineup'])}")
    print(f"    {row['minutes']:.1f} min · {row['plus_minus']:+d} +/- · {row['plus_minus_per_36']:+.1f} per 36")

Total unique 5-man lineups: 415
Lineups with 5+ minutes: 149

--- KD-IN lineups (5+ min): 87 ---
Total KD-on minutes: 2611
Net plus/minus: +388

--- KD-OUT lineups (5+ min): 62 ---
Total KD-off minutes: 887
Net plus/minus: +67

=== TOP 10 KD-IN LINEUPS (by minutes) ===
  KD + Alperen Sengun / Amen Thompson / Jabari Smith Jr. / Josh Okogie
    378.8 min · +76 +/- · +7.2 per 36
  KD + Alperen Sengun / Amen Thompson / Jabari Smith Jr. / Tari Eason
    305.3 min · +51 +/- · +6.0 per 36
  KD + Alperen Sengun / Amen Thompson / Jabari Smith Jr. / Reed Sheppard
    267.3 min · -3 +/- · -0.4 per 36
  KD + Alperen Sengun / Amen Thompson / Jabari Smith Jr. / Steven Adams
    96.9 min · +2 +/- · +0.7 per 36
  KD + Amen Thompson / Clint Capela / Dorian Finney-Smith / Reed Sheppard
    83.5 min · +34 +/- · +14.7 per 36
  KD + Alperen Sengun / Amen Thompson / Reed Sheppard / Tari Eason
    76.7 min · -13 +/- · -6.1 per 36
  KD + Clint Capela / Jabari Smith Jr. / Reed Sheppard / Tari Eason
    66.2 mi

## Exploratory: KD-off lineups during the 4 worst games

Investigate which 5-man lineups were on the floor during the four worst KD-off stretches identified in Section 2. The output here informed early drafts of "When the Wheels Came Off" but the cells below also include test versions (v1, v2) that didn't ship — kept here for reference.

In [169]:
# ============================================================
# What KD-off lineups played during the 4 worst games?
# ============================================================

BAD_GAMES = {
    "0022500491": "JAN 03 @ DAL",
    "0022500540": "JAN 09 @ POR",
    "0022500816": "FEB 21 @ NYK",
    "0022500864": "FEB 28 @ MIA",
}

for nba_gid, label in BAD_GAMES.items():
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"{'='*70}")
    
    # Get this game's stints
    game_stints = [s for s in all_stints if s["game_id"] == nba_gid]
    
    # Filter to KD-out stints with meaningful duration
    kd_out_stints = [
        s for s in game_stints 
        if "Kevin Durant" not in s["lineup"]
    ]
    
    if not kd_out_stints:
        print("  No KD-out stints found")
        continue
    
    print(f"  Total KD-off stints: {len(kd_out_stints)}")
    print(f"  Total KD-off time: {sum(s['duration_sec'] for s in kd_out_stints) / 60:.1f} min")
    
    # Combine identical lineups within the game
    game_agg = defaultdict(lambda: {"dur": 0, "pf": 0, "pa": 0})
    for s in kd_out_stints:
        game_agg[s["lineup"]]["dur"] += s["duration_sec"]
        game_agg[s["lineup"]]["pf"] += s["hou_pts"]
        game_agg[s["lineup"]]["pa"] += s["opp_pts"]
    
    # Sort by duration in this game
    sorted_lineups = sorted(game_agg.items(), key=lambda x: -x[1]["dur"])
    
    for lineup, agg in sorted_lineups:
        minutes = agg["dur"] / 60
        pm = agg["pf"] - agg["pa"]
        players = sorted(lineup)
        print(f"\n    {' / '.join(players)}")
        print(f"    {minutes:.1f} min · {pm:+d} (HOU {agg['pf']}, OPP {agg['pa']})")


  JAN 03 @ DAL
  Total KD-off stints: 6
  Total KD-off time: 10.3 min

    Amen Thompson / Clint Capela / Jabari Smith Jr. / Reed Sheppard / Tari Eason
    2.6 min · -5 (HOU 2, OPP 7)

    Aaron Holiday / Amen Thompson / Jabari Smith Jr. / Jae'Sean Tate / Tari Eason
    2.2 min · +1 (HOU 3, OPP 2)

    Aaron Holiday / Amen Thompson / Jabari Smith Jr. / Jae'Sean Tate / Reed Sheppard
    2.0 min · -8 (HOU 0, OPP 8)

    Amen Thompson / Dorian Finney-Smith / Jabari Smith Jr. / Reed Sheppard / Tari Eason
    1.7 min · -4 (HOU 2, OPP 6)

    Aaron Holiday / Jabari Smith Jr. / Jae'Sean Tate / Josh Okogie / Tari Eason
    1.3 min · -3 (HOU 2, OPP 5)

    Amen Thompson / Dorian Finney-Smith / Jabari Smith Jr. / Josh Okogie / Tari Eason
    0.5 min · -2 (HOU 0, OPP 2)

  JAN 09 @ POR
  Total KD-off stints: 5
  Total KD-off time: 8.7 min

    Amen Thompson / Dorian Finney-Smith / Jabari Smith Jr. / Reed Sheppard / Steven Adams
    3.3 min · -11 (HOU 0, OPP 11)

    Amen Thompson / Jabari Smith 

In [150]:
# How many games did each of the 4 "worst" lineups play in?
WORST_LINEUPS = [
    ("DAL", frozenset(["Aaron Holiday", "Amen Thompson", "Jabari Smith Jr.", "Jae'Sean Tate", "Reed Sheppard"])),
    ("POR", frozenset(["Amen Thompson", "Dorian Finney-Smith", "Jabari Smith Jr.", "Reed Sheppard", "Steven Adams"])),
    ("NYK", frozenset(["Alperen Sengun", "Dorian Finney-Smith", "Jabari Smith Jr.", "Reed Sheppard", "Tari Eason"])),
    ("MIA", frozenset(["Aaron Holiday", "Alperen Sengun", "Amen Thompson", "Dorian Finney-Smith", "Tari Eason"])),
]

for label, target_lineup in WORST_LINEUPS:
    games_used = set()
    total_min = 0
    total_pm = 0
    for s in all_stints:
        if s["lineup"] == target_lineup:
            games_used.add(s["game_id"])
            total_min += s["duration_sec"] / 60
            total_pm += s["plus_minus"]
    
    game_dates = sorted([game_info[g]["GAME_DATE"] + " " + game_info[g]["MATCHUP"] for g in games_used])
    print(f"\n{label} bad lineup: {sorted(target_lineup)}")
    print(f"  Games used in: {len(games_used)} · Total minutes: {total_min:.1f} · Total +/-: {total_pm:+d}")
    for gd in game_dates:
        print(f"    {gd}")


DAL bad lineup: ['Aaron Holiday', 'Amen Thompson', 'Jabari Smith Jr.', "Jae'Sean Tate", 'Reed Sheppard']
  Games used in: 3 · Total minutes: 4.5 · Total +/-: -14
    DEC 06, 2025 HOU @ DAL
    DEC 23, 2025 HOU @ LAC
    JAN 03, 2026 HOU @ DAL

POR bad lineup: ['Amen Thompson', 'Dorian Finney-Smith', 'Jabari Smith Jr.', 'Reed Sheppard', 'Steven Adams']
  Games used in: 3 · Total minutes: 5.5 · Total +/-: -15
    JAN 07, 2026 HOU @ POR
    JAN 09, 2026 HOU @ POR
    JAN 11, 2026 HOU @ SAC

NYK bad lineup: ['Alperen Sengun', 'Dorian Finney-Smith', 'Jabari Smith Jr.', 'Reed Sheppard', 'Tari Eason']
  Games used in: 2 · Total minutes: 7.3 · Total +/-: -12
    FEB 07, 2026 HOU @ OKC
    FEB 21, 2026 HOU @ NYK

MIA bad lineup: ['Aaron Holiday', 'Alperen Sengun', 'Amen Thompson', 'Dorian Finney-Smith', 'Tari Eason']
  Games used in: 2 · Total minutes: 5.1 · Total +/-: -5
    FEB 02, 2026 HOU @ IND
    FEB 28, 2026 HOU @ MIA


In [154]:
LABEL_TO_GAME = {
    "DAL": "0022500491",
    "POR": "0022500540",
    "NYK": "0022500816",
    "MIA": "0022500864",
}

for label, target_lineup in WORST_LINEUPS:
    target_game = LABEL_TO_GAME[label]
    
    game_stints = [s for s in all_stints if s["game_id"] == target_game]
    
    running_pm = 0
    for s in game_stints:
        if s["lineup"] == target_lineup:
            print(f"\n{label}: lineup entered at HOU margin {running_pm:+d}")
            print(f"   Then went HOU {s['hou_pts']}, OPP {s['opp_pts']} in {s['duration_sec']/60:.1f} min")
            print(f"   Lineup left at HOU margin {running_pm + s['plus_minus']:+d}")
            break
        running_pm += s["plus_minus"]


DAL: lineup entered at HOU margin +6
   Then went HOU 0, OPP 8 in 2.0 min
   Lineup left at HOU margin -2

POR: lineup entered at HOU margin +13
   Then went HOU 0, OPP 11 in 3.3 min
   Lineup left at HOU margin +2

NYK: lineup entered at HOU margin +15
   Then went HOU 4, OPP 13 in 3.1 min
   Lineup left at HOU margin +6

MIA: lineup entered at HOU margin -4
   Then went HOU 0, OPP 9 in 3.5 min
   Lineup left at HOU margin -13


### Preview: 'Wheels Came Off' versions A/B

Early test versions of the Wheels Came Off table (the chosen version is built later in Section 3).

In [155]:
# ============================================================
# When the Wheels Came Off — two versions
# ============================================================

# Build a unified data structure for the 4 bad-stint games
bad_stints_data = [
    {
        "label": "@ Dallas",
        "date": "JAN 03, 2026",
        "entered_margin": +6,
        "left_margin": -2,
        "hou_pts": 0,
        "opp_pts": 8,
        "duration_min": 2.0,
        "lineup": ["Aaron Holiday", "Amen Thompson", "Jabari Smith Jr.", "Jae'Sean Tate", "Reed Sheppard"],
        "games_used_total": 3,
    },
    {
        "label": "@ Portland",
        "date": "JAN 09, 2026",
        "entered_margin": +13,
        "left_margin": +2,
        "hou_pts": 0,
        "opp_pts": 11,
        "duration_min": 3.3,
        "lineup": ["Amen Thompson", "Dorian Finney-Smith", "Jabari Smith Jr.", "Reed Sheppard", "Steven Adams"],
        "games_used_total": 3,
    },
    {
        "label": "@ New York",
        "date": "FEB 21, 2026",
        "entered_margin": +15,
        "left_margin": +6,
        "hou_pts": 4,
        "opp_pts": 13,
        "duration_min": 3.1,
        "lineup": ["Alperen Sengun", "Dorian Finney-Smith", "Jabari Smith Jr.", "Reed Sheppard", "Tari Eason"],
        "games_used_total": 2,
    },
    {
        "label": "@ Miami",
        "date": "FEB 28, 2026",
        "entered_margin": -4,
        "left_margin": -13,
        "hou_pts": 0,
        "opp_pts": 9,
        "duration_min": 3.5,
        "lineup": ["Aaron Holiday", "Alperen Sengun", "Amen Thompson", "Dorian Finney-Smith", "Tari Eason"],
        "games_used_total": 2,
    },
]

zero_pt_count = sum(1 for d in bad_stints_data if d["hou_pts"] == 0)


# ============================================================
# Helper: format a margin nicely
# ============================================================
def fmt_margin(m):
    if m > 0:
        return f"+{m}"
    return str(m)


# ============================================================
# VERSION 1: Minimal — just the 4 stints + 0-point callout
# ============================================================

v1_table_rows = ""
for d in bad_stints_data:
    v1_table_rows += f"""
    <tr>
      <td class="metric">{d['date']}<br><span style="color:#6B7280;font-size:13px;font-weight:400">{d['label']}</span></td>
      <td class="value">{fmt_margin(d['entered_margin'])} → {fmt_margin(d['left_margin'])}</td>
      <td class="value">{d['hou_pts']}–{d['opp_pts']}</td>
      <td class="value">{d['duration_min']:.1f}</td>
    </tr>"""

v1_body = f"""
  <div class="table-label">Clutch Time, Section 02</div>
  <h2>When the Wheels Came Off</h2>
  <p class="table-subtitle">In each of Houston's four worst KD-rest stretches, the lineup that took the floor produced a damaging run before Durant returned.</p>
  <table>
    <thead>
      <tr>
        <th style="text-align:left">Game</th>
        <th>Margin In → Out</th>
        <th>HOU–OPP</th>
        <th>Minutes</th>
      </tr>
    </thead>
    <tbody>{v1_table_rows}
    </tbody>
  </table>
  <div class="callout-grid" style="grid-template-columns: 1fr; max-width: 320px; margin-top: 28px;">
    <div class="card">
      <div class="label">Stretches with zero Houston points</div>
      <div class="big">{zero_pt_count} of 4</div>
      <div class="sub">Houston scored 0 in three of these four stretches.</div>
    </div>
  </div>
"""

with open(ASSETS_DIR / "02e_wheels_came_off_v1.html", "w") as f:
    f.write(wrap_html("When the Wheels Came Off (v1)", v1_body))


# ============================================================
# VERSION 2: Fuller — adds rare-lineup column + reframed subtitle
# ============================================================

v2_table_rows = ""
for d in bad_stints_data:
    # Format the lineup as a compact string
    lineup_str = ", ".join([p.split()[-1] for p in d["lineup"]])  # last names only
    v2_table_rows += f"""
    <tr>
      <td class="metric">{d['date']}<br><span style="color:#6B7280;font-size:13px;font-weight:400">{d['label']}</span></td>
      <td class="value" style="text-align:left;font-size:13px;color:#374151">{lineup_str}</td>
      <td class="value">{fmt_margin(d['entered_margin'])} → {fmt_margin(d['left_margin'])}</td>
      <td class="value">{d['hou_pts']}–{d['opp_pts']}</td>
      <td class="value">{d['games_used_total']}</td>
    </tr>"""

v2_body = f"""
  <div class="table-label">Clutch Time, Section 02</div>
  <h2>When the Wheels Came Off</h2>
  <p class="table-subtitle">In each of Houston's four worst KD-rest stretches, the lineup on the floor was one that Udoka had deployed in only 2 or 3 games all season.</p>
  <table>
    <thead>
      <tr>
        <th style="text-align:left">Game</th>
        <th style="text-align:left">Lineup</th>
        <th>Margin In → Out</th>
        <th>HOU–OPP</th>
        <th>Games Used</th>
      </tr>
    </thead>
    <tbody>{v2_table_rows}
    </tbody>
  </table>
  <div class="callout-grid" style="grid-template-columns: 1fr; max-width: 320px; margin-top: 28px;">
    <div class="card">
      <div class="label">Stretches with zero Houston points</div>
      <div class="big">{zero_pt_count} of 4</div>
      <div class="sub">Houston scored 0 in three of these four stretches.</div>
    </div>
  </div>
  <p class="table-footnote">"Games Used" shows how many games this exact 5-man combination played together all season.</p>
"""

with open(ASSETS_DIR / "02e_wheels_came_off_v2.html", "w") as f:
    f.write(wrap_html("When the Wheels Came Off (v2)", v2_body))


print(f"Saved:")
print(f"  {ASSETS_DIR / '02e_wheels_came_off_v1.html'}")
print(f"  {ASSETS_DIR / '02e_wheels_came_off_v2.html'}")

Saved:
  /Users/stevenyan/Desktop/rockets_season_review/report_assets/02e_wheels_came_off_v1.html
  /Users/stevenyan/Desktop/rockets_season_review/report_assets/02e_wheels_came_off_v2.html


## Refined exploratory: KD-off stints with broader context

Recompute the "concerning" KD-off stretches with full per-stint context. The next cells settle on the LOSS_STINTS dataset that ends up in the final `03a_wheels_came_off.html`.

In [181]:
print("=" * 130)
print("CONCERNING KD-OFF STINTS — FULL CONTEXT")
print("=" * 130)

# We need to rebuild `concerning` since `all_stints` was recomputed
concerning = []
for s in all_stints:
    if "Kevin Durant" in s["lineup"]:
        continue
    if s["duration_sec"] / 60 < 2.0:
        continue
    if s["plus_minus"] > -8:
        continue
    concerning.append(s)
concerning.sort(key=lambda s: s["plus_minus"])

def kd_played_in_game(game_id):
    return any("Kevin Durant" in s["lineup"] for s in all_stints if s["game_id"] == game_id)


for stint in concerning:
    game_id = stint["game_id"]
    info = game_info[game_id]
    
    if not kd_played_in_game(game_id):
        continue
    
    game_stints_local = [s for s in all_stints if s["game_id"] == game_id]
    target_idx = None
    for i, s in enumerate(game_stints_local):
        if (s["lineup"] == stint["lineup"] and
            abs(s["duration_sec"] - stint["duration_sec"]) < 1 and
            s["plus_minus"] == stint["plus_minus"] and
            s["period"] == stint["period"] and
            s["clock"] == stint["clock"]):
            target_idx = i
            break
    
    if target_idx is None:
        continue
    
    period = stint["period"]
    clock = stint["clock"]
    period_label = f"Q{period}" if period <= 4 else f"OT{period-4}"
    
    margin_at_start = sum(s["plus_minus"] for s in game_stints_local[:target_idx])
    margin_at_end = margin_at_start + stint["plus_minus"]
    hou_at_start = sum(s["hou_pts"] for s in game_stints_local[:target_idx])
    opp_at_start = sum(s["opp_pts"] for s in game_stints_local[:target_idx])
    hou_at_end = hou_at_start + stint["hou_pts"]
    opp_at_end = opp_at_start + stint["opp_pts"]
    
    prev_kd_lineup = None
    for j in range(target_idx - 1, -1, -1):
        if "Kevin Durant" in game_stints_local[j]["lineup"] and len(game_stints_local[j]["lineup"]) == 5:
            prev_kd_lineup = game_stints_local[j]["lineup"]
            break
    
    print(f"\n{info['GAME_DATE']} {info['MATCHUP']} ({info['WL']})")
    print(f"  When KD subbed off: {period_label} {clock}  ·  Stint lasted {stint['duration_sec']/60:.1f} min")
    print(f"  Score at sub-off:   HOU {hou_at_start} - OPP {opp_at_start}  (HOU {margin_at_start:+d})")
    print(f"  Score at sub-in:    HOU {hou_at_end} - OPP {opp_at_end}  (HOU {margin_at_end:+d})")
    print(f"  Stint result:       HOU {stint['hou_pts']} - OPP {stint['opp_pts']}  ({stint['plus_minus']:+d})")
    if prev_kd_lineup:
        print(f"  Lineup with KD:        {sorted(prev_kd_lineup)}")
    else:
        print(f"  Lineup with KD:        \u26a0 no prior KD-in lineup")
    print(f"  Lineup after KD sits:  {sorted(stint['lineup'])}")

CONCERNING KD-OFF STINTS — FULL CONTEXT

JAN 09, 2026 HOU @ POR (L)
  When KD subbed off: Q4 12:00  ·  Stint lasted 3.3 min
  Score at sub-off:   HOU 90 - OPP 77  (HOU +13)
  Score at sub-in:    HOU 90 - OPP 88  (HOU +2)
  Stint result:       HOU 0 - OPP 11  (-11)
  Lineup with KD:        ['Amen Thompson', 'Clint Capela', 'Jabari Smith Jr.', 'Kevin Durant', 'Reed Sheppard']
  Lineup after KD sits:  ['Amen Thompson', 'Dorian Finney-Smith', 'Jabari Smith Jr.', 'Reed Sheppard', 'Steven Adams']

MAR 31, 2026 HOU vs. NYK (W)
  When KD subbed off: Q2 12:00  ·  Stint lasted 4.6 min
  Score at sub-off:   HOU 37 - OPP 21  (HOU +16)
  Score at sub-in:    HOU 44 - OPP 37  (HOU +7)
  Stint result:       HOU 7 - OPP 16  (-9)
  Lineup with KD:        ['Alperen Sengun', 'Amen Thompson', 'Jabari Smith Jr.', 'Kevin Durant', 'Reed Sheppard']
  Lineup after KD sits:  ['Aaron Holiday', 'Alperen Sengun', 'Amen Thompson', 'Jabari Smith Jr.', "Jae'Sean Tate"]

MAR 11, 2026 HOU @ DEN (L)
  When KD subbed off:

In [189]:
# ============================================================
# When the Wheels Came Off — v1 and v2 (with Net Margin column)
# ============================================================

LOSS_STINTS = [
    {
        "label": "@ Dallas",
        "date": "JAN 03, 2026",
        "period": "Q2 12:00",
        "duration_min": 2.0,
        "entered_margin": +6,
        "left_margin": -2,
        "hou_pts": 0,
        "opp_pts": 8,
        "result": "L",
    },
    {
        "label": "@ Portland",
        "date": "JAN 09, 2026",
        "period": "Q4 12:00",
        "duration_min": 3.3,
        "entered_margin": +13,
        "left_margin": +2,
        "hou_pts": 0,
        "opp_pts": 11,
        "result": "L",
    },
    {
        "label": "vs LA Clippers",
        "date": "FEB 11, 2026",
        "period": "Q4 12:00",
        "duration_min": 3.0,
        "entered_margin": +6,
        "left_margin": -3,
        "hou_pts": 4,
        "opp_pts": 13,
        "result": "L",
    },
    {
        "label": "@ New York",
        "date": "FEB 21, 2026",
        "period": "Q4 12:00",       # FIXED: was Q4 10:10
        "duration_min": 3.1,
        "entered_margin": +16,       # FIXED: was +15
        "left_margin": +6,
        "hou_pts": 4,
        "opp_pts": 13,
        "result": "L",
    },
    {
        "label": "@ Miami",
        "date": "FEB 28, 2026",
        "period": "Q2 12:00",
        "duration_min": 3.5,
        "entered_margin": -4,
        "left_margin": -13,
        "hou_pts": 0,
        "opp_pts": 9,
        "result": "L",
    },
]

SURVIVED_STINT = {
    "label": "vs New York",
    "date": "MAR 31, 2026",
    "period": "Q1 0:02",            # FIXED: was Q2 12:00
    "duration_min": 4.6,
    "entered_margin": +16,
    "left_margin": +7,
    "hou_pts": 7,
    "opp_pts": 16,
    "result": "W",
}


def fmt_margin(m):
    return f"+{m}" if m > 0 else str(m)


def build_table_rows(stints, include_result=False):
    rows = ""
    for d in stints:
        net = d["hou_pts"] - d["opp_pts"]
        result_cell = f'<td class="value">{d["result"]}</td>' if include_result else ""
        rows += f"""
    <tr>
      <td class="metric">{d['date']}<br><span style="color:#6B7280;font-size:13px;font-weight:400">{d['label']}</span></td>
      <td class="value">{d['period']}</td>
      <td class="value">{fmt_margin(d['entered_margin'])} → {fmt_margin(d['left_margin'])}</td>
      <td class="value">{d['hou_pts']}–{d['opp_pts']}</td>
      <td class="value">{fmt_margin(net)}</td>
      <td class="value">{d['duration_min']:.1f}</td>{result_cell}
    </tr>"""
    return rows


# ============================================================
# VERSION 1: Just the 5 loss stints
# ============================================================

v1_rows = build_table_rows(LOSS_STINTS, include_result=False)
zero_pt_v1 = sum(1 for s in LOSS_STINTS if s["hou_pts"] == 0)

v1_body = f"""
  <div class="table-label">Clutch Time, Section 02</div>
  <h2>When the Wheels Came Off</h2>
  <p class="table-subtitle">Five games Houston lost where the unraveling started the moment Kevin Durant subbed out. In each, the bench unit gave up at least 8 points before he returned.</p>
  <table>
    <thead>
      <tr>
        <th style="text-align:left">Game</th>
        <th>KD Subbed Off</th>
        <th>Margin In → Out</th>
        <th>HOU–OPP</th>
        <th>Net Margin</th>
        <th>KD Rest (Min)</th>
      </tr>
    </thead>
    <tbody>{v1_rows}
    </tbody>
  </table>
  <div class="callout-grid" style="grid-template-columns: 1fr; max-width: 360px; margin-top: 28px;">
    <div class="card">
      <div class="label">Stretches with zero Houston points</div>
      <div class="big">{zero_pt_v1} of 5</div>
      <div class="sub">Houston scored 0 in three of these five stretches.</div>
    </div>
  </div>
  <p class="table-footnote">Criteria: KD-off stints lasting 2+ minutes where Houston was outscored by 8+ points, filtered to losses.</p>
"""

with open(ASSETS_DIR / "02e_wheels_came_off_v1.html", "w") as f:
    f.write(wrap_html("When the Wheels Came Off (v1)", v1_body))


# ============================================================
# VERSION 2: 5 losses + 1 survived
# ============================================================

all_stints_v2 = LOSS_STINTS + [SURVIVED_STINT]
v2_rows = build_table_rows(all_stints_v2, include_result=True)
zero_pt_v2 = sum(1 for s in all_stints_v2 if s["hou_pts"] == 0)

v2_body = f"""
  <div class="table-label">Clutch Time, Section 02</div>
  <h2>When the Wheels Came Off</h2>
  <p class="table-subtitle">Six games where the unraveling started the moment Kevin Durant subbed out. Houston lost five of them, with the bench giving up 8 or more points in every stretch before he returned.</p>
  <table>
    <thead>
      <tr>
        <th style="text-align:left">Game</th>
        <th>KD Subbed Off</th>
        <th>Margin In → Out</th>
        <th>HOU–OPP</th>
        <th>Net Margin</th>
        <th>KD Rest (Min)</th>
        <th>Result</th>
      </tr>
    </thead>
    <tbody>{v2_rows}
    </tbody>
  </table>
  <div class="callout-grid" style="grid-template-columns: 1fr; max-width: 360px; margin-top: 28px;">
    <div class="card">
      <div class="label">Stretches with zero Houston points</div>
      <div class="big">{zero_pt_v2} of 6</div>
      <div class="sub">Houston scored 0 in three of these stretches.</div>
    </div>
  </div>
  <p class="table-footnote">Criteria: KD-off stints lasting 2+ minutes where Houston was outscored by 8+ points. Garbage-time stretches were excluded by inspection.</p>
"""

with open(ASSETS_DIR / "02e_wheels_came_off_v2.html", "w") as f:
    f.write(wrap_html("When the Wheels Came Off (v2)", v2_body))


print(f"Saved:")
print(f"  {ASSETS_DIR / '02e_wheels_came_off_v1.html'}")
print(f"  {ASSETS_DIR / '02e_wheels_came_off_v2.html'}")

Saved:
  /Users/stevenyan/Desktop/rockets_season_review/report_assets/02e_wheels_came_off_v1.html
  /Users/stevenyan/Desktop/rockets_season_review/report_assets/02e_wheels_came_off_v2.html


### File restructure: rename Section 2 files, move Wheels Came Off to Section 3

Moved the Wheels Came Off asset out of Section 2 and into Section 3 (KD Reliance) at this point. This cell handles the file renames.

In [449]:
# ============================================================
# Restructure: rename Section 2, move Wheels Came Off to Section 3
# ============================================================
import shutil

# Step 1: Update section labels in existing Section 2 files
SECTION_2_FILES = [
    "02a_clutch_comparison.html",
    "02b_four_collapses.html",
    "02c_when_kd_sits.html",
    "02d_kd_off_scatter.html",
]

for fname in SECTION_2_FILES:
    fpath = ASSETS_DIR / fname
    if not fpath.exists():
        print(f"⚠ {fname} not found, skipping")
        continue
    
    content = fpath.read_text()
    # Replace the section label
    content = content.replace("CLUTCH TIME, SECTION 02", "THE CLOSE-OUT PROBLEM, SECTION 02")
    content = content.replace("Clutch Time, Section 02", "The Close-Out Problem, Section 02")
    fpath.write_text(content)
    print(f"Updated: {fname}")


# Step 2: Rebuild Wheels Came Off as 03a with new section label
LOSS_STINTS = [
    {"label": "@ Dallas", "date": "JAN 03, 2026", "period": "Q2 12:00",
     "duration_min": 2.0, "entered_margin": +6, "left_margin": -2,
     "hou_pts": 0, "opp_pts": 8, "result": "L"},
    {"label": "@ Portland", "date": "JAN 09, 2026", "period": "Q4 12:00",
     "duration_min": 3.3, "entered_margin": +13, "left_margin": +2,
     "hou_pts": 0, "opp_pts": 11, "result": "L"},
    {"label": "vs LA Clippers", "date": "FEB 11, 2026", "period": "Q4 12:00",
     "duration_min": 3.0, "entered_margin": +6, "left_margin": -3,
     "hou_pts": 4, "opp_pts": 13, "result": "L"},
    {"label": "@ New York", "date": "FEB 21, 2026", "period": "Q4 12:00",
     "duration_min": 3.1, "entered_margin": +16, "left_margin": +6,
     "hou_pts": 4, "opp_pts": 13, "result": "L"},
    {"label": "@ Miami", "date": "FEB 28, 2026", "period": "Q2 12:00",
     "duration_min": 3.5, "entered_margin": -4, "left_margin": -13,
     "hou_pts": 0, "opp_pts": 9, "result": "L"},
]


def fmt_margin(m):
    return f"+{m}" if m > 0 else str(m)


rows = ""
for d in LOSS_STINTS:
    net = d["hou_pts"] - d["opp_pts"]
    rows += f"""
    <tr>
      <td class="metric">{d['date']}<br><span style="color:#6B7280;font-size:13px;font-weight:400">{d['label']}</span></td>
      <td class="value">{d['period']}</td>
      <td class="value">{fmt_margin(d['entered_margin'])} → {fmt_margin(d['left_margin'])}</td>
      <td class="value">{d['hou_pts']}–{d['opp_pts']}</td>
      <td class="value">{fmt_margin(net)}</td>
      <td class="value">{d['duration_min']:.1f}</td>
    </tr>"""

zero_pt = sum(1 for s in LOSS_STINTS if s["hou_pts"] == 0)

body = f"""
  <div class="table-label">KD Reliance, Section 03</div>
  <h2>When the Wheels Came Off</h2>
  <p class="table-subtitle">Five losses where the unraveling started the moment Kevin Durant subbed out. In each, the bench unit gave up at least 8 points before he returned.</p>
  <table>
    <thead>
      <tr>
        <th style="text-align:left">Game</th>
        <th>KD Subbed Off</th>
        <th>Margin In → Out</th>
        <th>HOU–OPP</th>
        <th>Net Margin</th>
        <th>Stretch Length (Min)</th>
      </tr>
    </thead>
    <tbody>{rows}
    </tbody>
  </table>
  <p class="table-footnote">Criteria: KD-off stints lasting 2+ minutes where Houston was outscored by 8+ points, filtered to losses.<br>Houston scored 0 points in {zero_pt} of these 5 stretches.</p>
"""

with open(ASSETS_DIR / "03a_wheels_came_off.html", "w") as f:
    f.write(wrap_html("When the Wheels Came Off", body))
print(f"Created: 03a_wheels_came_off.html")


# Step 3: Clean up old v1 and v2 files
for old_file in ["02e_wheels_came_off_v1.html", "02e_wheels_came_off_v2.html"]:
    old_path = ASSETS_DIR / old_file
    if old_path.exists():
        old_path.unlink()
        print(f"Deleted: {old_file}")

Updated: 02a_clutch_comparison.html
⚠ 02b_four_collapses.html not found, skipping
Updated: 02c_when_kd_sits.html
Updated: 02d_kd_off_scatter.html
Created: 03a_wheels_came_off.html


<a id="section-3"></a>
# Section 3: KD Reliance

Builds 5 assets for the KD Reliance tab:

| File | What it shows |
|---|---|
| `03a_wheels_came_off.html` | KD-off loss stints (the 5 worst Q4/Q2 collapses) |
| `03b_kd_headline.html` | KD's headline stats card ("Carrying the Load") |
| `03c_kd_vs_stars.html` | KD vs. LeBron/Curry/Harden minutes comparison |
| `03d_kd_involvement.html` | KD's % of Houston's total scoring (31.9%) |
| `03e_kd_gravity.html` | KD's gravity numbers (1st in NBA) |

## Build `03b_kd_headline.html` — KD headline stats card ('Carrying the Load')

In [410]:
# ============================================================
# Section 03: KD Headline Card
# ============================================================

# KD was born September 29, 1988. During the 2025-26 season he was 37.
KD_AGE = 37

# Compute averages from kd_2526
min_g = kd_2526["MIN"].mean()
pts_g = kd_2526["PTS"].mean()
reb_g = kd_2526["REB"].mean()
ast_g = kd_2526["AST"].mean()
tov_g = kd_2526["TOV"].mean()
pm_g = kd_2526["PLUS_MINUS"].mean()

# Format
body = f"""
  <div class="table-label">KD Reliance, Section 03</div>
  <h2>Carrying the Load</h2>
  <p class="table-subtitle">Houston's title hopes ran through a 37-year-old playing extremely heavy minutes every night.</p>
  
  <div class="callout-grid" style="grid-template-columns: repeat(3, 1fr); gap: 24px; margin-top: 32px;">
    <div class="card">
      <div class="label">Age</div>
      <div class="big">{KD_AGE}</div>
      <div class="sub">2nd-oldest player to average 25+ ppg</div>
    </div>
    <div class="card">
      <div class="label">Minutes / Game</div>
      <div class="big">{min_g:.1f}</div>
      <div class="sub">3rd in NBA among all players</div>
    </div>
    <div class="card">
      <div class="label">Points / Game</div>
      <div class="big">{pts_g:.1f}</div>
      <div class="sub">Houston's leading scorer</div>
    </div>
  </div>
  
  <div class="callout-grid" style="grid-template-columns: 1fr; margin-top: 24px;">
    <div class="card" style="padding: 8px 20px; text-align: center;">
      <div class="label">Per Game</div>
      <div style="display: flex; gap: 28px; margin-top: 2px; align-items: baseline; justify-content: center;">
        <span><span style="font-size: 22px; font-weight: 600;">{reb_g:.1f}</span> <span style="font-size: 11px; color: #6B7280; text-transform: uppercase; letter-spacing: 0.08em;">REB</span></span>
        <span><span style="font-size: 22px; font-weight: 600;">{ast_g:.1f}</span> <span style="font-size: 11px; color: #6B7280; text-transform: uppercase; letter-spacing: 0.08em;">AST</span></span>
        <span><span style="font-size: 22px; font-weight: 600;">{tov_g:.1f}</span> <span style="font-size: 11px; color: #6B7280; text-transform: uppercase; letter-spacing: 0.08em;">TOV</span></span>
        <span><span style="font-size: 22px; font-weight: 600;">{fmt_pm(pm_g)}</span> <span style="font-size: 11px; color: #6B7280; text-transform: uppercase; letter-spacing: 0.08em;">+/-</span></span>
      </div>
    </div>
  </div>
"""

with open(ASSETS_DIR / "03b_kd_headline.html", "w") as f:
    f.write(wrap_html("Carrying the Load", body))

print(f"Updated.")

Updated.


### Cleanup: delete the abandoned workload chart (early draft of 03c)

In [217]:
# Delete the workload chart we abandoned
workload_path = ASSETS_DIR / "03c_kd_workload.html"
if workload_path.exists():
    workload_path.unlink()
    print(f"Deleted: 03c_kd_workload.html")

Deleted: 03c_kd_workload.html


## Build `03c_kd_vs_stars.html` — KD vs. aging stars

In [218]:
# ============================================================
# Pull 2025-26 stats for KD, LeBron, Curry, Harden
# ============================================================
from nba_api.stats.endpoints import playergamelog

COMPARISON_PLAYERS = [
    {"name": "LeBron James",   "player_id": 2544,    "age": 41},
    {"name": "Stephen Curry",  "player_id": 201939,  "age": 38},
    {"name": "Kevin Durant",   "player_id": 201142,  "age": 37},
    {"name": "James Harden",   "player_id": 201935,  "age": 36},
]

print("Fetching 2025-26 game logs...")
for p in COMPARISON_PLAYERS:
    log = playergamelog.PlayerGameLog(
        player_id=p["player_id"],
        season="2025-26",
        season_type_all_star="Regular Season"
    ).get_data_frames()[0]
    
    if len(log) == 0:
        print(f"  ⚠ {p['name']}: no games found")
        continue
    
    p["games"] = len(log)
    p["min_per_game"] = log["MIN"].mean()
    p["pts_per_game"] = log["PTS"].mean()
    print(f"  {p['name']:<20} (age {p['age']}): {p['games']} games, {p['min_per_game']:.1f} min/g, {p['pts_per_game']:.1f} ppg")

Fetching 2025-26 game logs...
  LeBron James         (age 41): 60 games, 33.1 min/g, 20.9 ppg
  Stephen Curry        (age 38): 43 games, 31.0 min/g, 26.6 ppg
  Kevin Durant         (age 37): 78 games, 36.4 min/g, 26.0 ppg
  James Harden         (age 36): 70 games, 34.9 min/g, 23.6 ppg


In [333]:
# ============================================================
# Section 03c: KD vs aging stars — minutes comparison
# ============================================================
import plotly.graph_objects as go

# Sort by minutes per game, ascending so highest is on top
players_sorted = sorted(COMPARISON_PLAYERS, key=lambda p: p["min_per_game"])

names = [p["name"] for p in players_sorted]
mins = [p["min_per_game"] for p in players_sorted]
ages = [p["age"] for p in players_sorted]
games = [p["games"] for p in players_sorted]
ppgs = [p["pts_per_game"] for p in players_sorted]

# Color KD's bar red, others gray
colors = [
    "#CE1141" if p["name"] == "Kevin Durant" else "#D1D5DB"
    for p in players_sorted
]

# Labels: "[Name], age X, Y games, Z ppg"
labels = [
    f"<b>{p['name']}</b><br><span style='font-size:11px;color:#6B7280'>Age {p['age']} · {p['games']} games · {p['pts_per_game']:.1f} PPG</span>"
    for p in players_sorted
]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=mins,
    y=labels,
    orientation="h",
    marker=dict(color=colors, line=dict(width=0)),
    text=[f"{m:.1f}" for m in mins],
    textposition="outside",
    textfont=dict(size=14, family="Inter, sans-serif", color="#1A1A1A"),
    hoverinfo="skip",
    cliponaxis=False,
))

fig.update_layout(
    height=360,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Inter, sans-serif", color="#1A1A1A", size=13),
    margin=dict(l=20, r=70, t=30, b=50),
    showlegend=False,
    xaxis=dict(
        title="Minutes per Game",
        zeroline=False,
        gridcolor="#F0EDE5",
        range=[0, max(mins) * 1.15],
    ),
    yaxis=dict(
        showgrid=False,
        zeroline=False,
        tickfont=dict(size=13),
    ),
    bargap=0.35,
)

plotly_html = fig.to_html(include_plotlyjs='cdn', full_html=True, config={'displayModeBar': False})

wrapper = """<!DOCTYPE html>
<html lang="en"><head>
<meta charset="UTF-8"><title>KD vs Aging Stars</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@500&display=swap" rel="stylesheet">
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: 'Inter', sans-serif; background: #FAFAF7; color: #1A1A1A; padding: 60px 40px; }
  .wrapper { max-width: 1100px; margin: 0 auto; }
  .table-label { font-family: 'JetBrains Mono', monospace; font-size: 11px; letter-spacing: 0.15em; text-transform: uppercase; color: #CE1141; margin-bottom: 8px; }
  h2 { font-family: 'Fraunces', Georgia, serif; font-size: 26px; font-weight: 600; letter-spacing: -0.015em; margin-bottom: 8px; }
  .subtitle { color: #6B7280; font-size: 15px; margin-bottom: 24px; }
  .chart-frame { background: white; border-radius: 8px; padding: 16px 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.04), 0 1px 2px rgba(0,0,0,0.06); }
  .footnote { color: #6B7280; font-size: 12px; margin-top: 14px; font-style: italic; }
</style></head>
<body>
<div class="wrapper">
  <div class="table-label">KD Reliance, Section 03</div>
  <h2>The Aging Star, Doing the Most</h2>
  <p class="subtitle">Among the league's other aging superstars, none carried a heavier load than Kevin Durant.</p>
  <div class="chart-frame">__CHART__</div>
  <p class="footnote">2025-26 regular season averages. Minutes per game, with age and total games played for context.</p>
</div>
</body></html>"""

import re
chart_div = re.search(r'<body>(.*?)</body>', plotly_html, re.DOTALL).group(1)
final = wrapper.replace("__CHART__", chart_div)

with open(ASSETS_DIR / "03c_kd_vs_stars.html", "w") as f:
    f.write(final)

print(f"Saved: {ASSETS_DIR / '03c_kd_vs_stars.html'}")
fig.show()

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/03c_kd_vs_stars.html


## Build `03d_kd_involvement.html` — KD scoring involvement

KD's points + assisted points as a share of Houston's total scoring (only counts games KD played).

In [221]:
# Find a few assist events in ESPN data to see the actual text format
print("Looking for Houston scoring events with assists...")
sample_count = 0
for game_id, espn in espn_data.items():
    plays = espn["plays"]
    for p in plays:
        if str(p.get("team", {}).get("id")) != HOU_ESPN_ID:
            continue
        text = p.get("text", "")
        if "AST" in text or "assist" in text.lower():
            print(f"  {text}")
            sample_count += 1
            if sample_count >= 10:
                break
    if sample_count >= 10:
        break

Looking for Houston scoring events with assists...
  Jae'Sean Tate makes 24-foot three point jumper (Reed Sheppard assists)
  Jae'Sean Tate makes 1-foot running dunk (Tari Eason assists)
  Reed Sheppard makes 24-foot running jump shot (Tari Eason assists)
  Aaron Holiday makes 26-foot three point jumper (Reed Sheppard assists)
  Tari Eason makes 2-foot alley oop dunk (JD Davison assists)
  Jae'Sean Tate makes driving layup (JD Davison assists)
  Aaron Holiday makes 23-foot running jump shot (JD Davison assists)
  Dorian Finney-Smith makes 26-foot three point jumper (Tari Eason assists)
  Clint Capela makes 6-foot hook shot (Josh Okogie assists)
  Josh Okogie makes 27-foot three point jumper (Jae'Sean Tate assists)


In [222]:
# ============================================================
# Compute KD Scoring Involvement
# ============================================================
import re

kd_pts = kd_2526["PTS"].sum()
hou_pts = team_gl_2526["PTS"].sum()

# Walk all Houston scoring plays, find ones where KD assists
kd_assist_pts = 0
kd_assist_events = 0

for game_id, espn in espn_data.items():
    plays = espn["plays"]
    for p in plays:
        if str(p.get("team", {}).get("id")) != HOU_ESPN_ID:
            continue
        text = p.get("text", "")
        # Format: "<Shooter> makes <description> (Durant assists)"
        if "(Durant assists)" not in text and "(Kevin Durant assists)" not in text:
            continue
        
        # Determine point value from shot type in the description
        if "three point" in text or "3-point" in text or "3pt" in text.lower():
            shot_pts = 3
        elif "free throw" in text.lower():
            shot_pts = 1  # rare for assists on FTs, but covering the case
        else:
            shot_pts = 2
        
        kd_assist_pts += shot_pts
        kd_assist_events += 1

print(f"KD assist events found: {kd_assist_events}")
print(f"KD assists from box score: {kd_2526['AST'].sum()}")
print(f"Points scored off KD assists: {kd_assist_pts}")

# Implied points-per-assist sanity check
if kd_assist_events > 0:
    ppa = kd_assist_pts / kd_assist_events
    print(f"Implied points per assist: {ppa:.2f} (typical: 2.0-2.5)")

# Summary
kd_total_impact = kd_pts + kd_assist_pts
involvement_rate = kd_total_impact / hou_pts * 100

print(f"\n=== KD SCORING INVOLVEMENT ===")
print(f"KD points scored:    {kd_pts}")
print(f"KD assist points:    {kd_assist_pts}")
print(f"KD total impact:     {kd_total_impact}")
print(f"HOU total points:    {hou_pts}")
print(f"Scoring involvement: {involvement_rate:.1f}%")

KD assist events found: 372
KD assists from box score: 372
Points scored off KD assists: 838
Implied points per assist: 2.25 (typical: 2.0-2.5)

=== KD SCORING INVOLVEMENT ===
KD points scored:    2026
KD assist points:    838
KD total impact:     2864
HOU total points:    9449
Scoring involvement: 30.3%


In [224]:
# Recompute using only games KD played
games_kd_played = set(kd_2526["Game_ID"].astype(str))
hou_pts_in_kd_games = team_gl_2526[
    team_gl_2526["Game_ID"].astype(str).isin(games_kd_played)
]["PTS"].sum()

print(f"HOU points in games KD played:  {hou_pts_in_kd_games}")
print(f"HOU points in all 82 games:     {hou_pts}")
print(f"HOU points without KD:          {hou_pts - hou_pts_in_kd_games}")
print(f"\nKD impact: {kd_total_impact}")
print(f"\nInvolvement (against 78-game total): {kd_total_impact / hou_pts_in_kd_games * 100:.1f}%")
print(f"Involvement (against 82-game total): {kd_total_impact / hou_pts * 100:.1f}%")

HOU points in games KD played:  8981
HOU points in all 82 games:     9449
HOU points without KD:          468

KD impact: 2864

Involvement (against 78-game total): 31.9%
Involvement (against 82-game total): 30.3%


In [225]:
# Use the games KD played as the denominator (apples-to-apples)
games_kd_played = set(kd_2526["Game_ID"].astype(str))
hou_pts_in_kd_games = team_gl_2526[
    team_gl_2526["Game_ID"].astype(str).isin(games_kd_played)
]["PTS"].sum()

# Recompute percentages
kd_pct = kd_pts / hou_pts_in_kd_games * 100
assist_pct = kd_assist_pts / hou_pts_in_kd_games * 100
other_pct = 100 - kd_pct - assist_pct
total_pct = kd_pct + assist_pct
other_pts = hou_pts_in_kd_games - kd_pts - kd_assist_pts

body = f"""
  <div class="table-label">KD Reliance, Section 03</div>
  <h2>Scoring Involvement</h2>
  <p class="subtitle">Houston points either scored by Kevin Durant or assisted by him, as a share of the team's total scoring.</p>

  <div style="text-align: center; margin: 48px 0 36px;">
    <div style="font-family: 'Fraunces', Georgia, serif; font-size: 96px; font-weight: 600; letter-spacing: -0.03em; color: #CE1141; line-height: 1;">
      {total_pct:.1f}%
    </div>
    <div style="font-family: 'JetBrains Mono', monospace; font-size: 12px; letter-spacing: 0.12em; text-transform: uppercase; color: #6B7280; margin-top: 12px;">
      of Houston's total points
    </div>
  </div>

  <div style="max-width: 800px; margin: 0 auto 16px;">
    <div style="display: flex; height: 36px; border-radius: 6px; overflow: hidden; box-shadow: 0 1px 2px rgba(0,0,0,0.05);">
      <div style="width: {kd_pct:.2f}%; background: #CE1141; display: flex; align-items: center; justify-content: center; color: white; font-size: 13px; font-weight: 600;">
        {kd_pct:.1f}%
      </div>
      <div style="width: {assist_pct:.2f}%; background: #E8869B; display: flex; align-items: center; justify-content: center; color: white; font-size: 13px; font-weight: 600;">
        {assist_pct:.1f}%
      </div>
      <div style="width: {other_pct:.2f}%; background: #E5E2DA; display: flex; align-items: center; justify-content: center; color: #6B7280; font-size: 13px; font-weight: 500;">
        {other_pct:.1f}%
      </div>
    </div>
    <div style="display: flex; justify-content: space-between; margin-top: 12px; font-size: 13px; color: #374151;">
      <div style="display: flex; align-items: center; gap: 8px;">
        <span style="width: 12px; height: 12px; background: #CE1141; border-radius: 2px;"></span>
        <span>KD points scored ({kd_pts:,})</span>
      </div>
      <div style="display: flex; align-items: center; gap: 8px;">
        <span style="width: 12px; height: 12px; background: #E8869B; border-radius: 2px;"></span>
        <span>Points off KD assists ({kd_assist_pts:,})</span>
      </div>
      <div style="display: flex; align-items: center; gap: 8px;">
        <span style="width: 12px; height: 12px; background: #E5E2DA; border-radius: 2px;"></span>
        <span>All other ({other_pts:,})</span>
      </div>
    </div>
  </div>

  <p class="table-footnote" style="text-align: center; margin-top: 32px;">
    In the {len(kd_2526)} regular season games Durant played in, he scored {kd_pts:,} points directly and created {kd_assist_pts:,} more on {int(kd_2526['AST'].sum())} assists.
  </p>
"""

with open(ASSETS_DIR / "03d_kd_involvement.html", "w") as f:
    f.write(wrap_html("Scoring Involvement", body))

print(f"Updated. New numbers:")
print(f"  Big number:       {total_pct:.1f}%")
print(f"  KD points share:  {kd_pct:.1f}%")
print(f"  Assist points:    {assist_pct:.1f}%")

Updated. New numbers:
  Big number:       31.9%
  KD points share:  22.6%
  Assist points:    9.3%


## Build `03e_kd_gravity.html` — KD gravity sidebar callout

In [ ]:
# ============================================================
# Section 03e: KD Gravity sidebar callout
# ============================================================

body = f"""
  <div class="table-label">KD Reliance, Section 03</div>
  <h2>Drawing All the Attention</h2>
  <p class="subtitle">Among all NBA players this season, Kevin Durant ranks 1st in average gravity and 2nd in total gravity minutes.</p>
  
  <div class="callout-grid" style="grid-template-columns: repeat(2, 1fr); gap: 24px; margin-top: 32px;">
    <div class="card">
      <div class="label">Average Gravity</div>
      <div class="big">+16.2</div>
      <div class="sub">1st in the NBA</div>
    </div>
    <div class="card">
      <div class="label">Gravity Minutes</div>
      <div class="big">694.8</div>
      <div class="sub">2nd in the NBA</div>
    </div>
  </div>
  
  <p class="table-footnote" style="margin-top: 28px;">
    Source: NBA.com <a href="https://www.nba.com/inside-the-game/player/gravity" style="color: #CE1141; text-decoration: none;">Inside the Game</a> tracking data.
  </p>
"""

with open(ASSETS_DIR / "03e_kd_gravity.html", "w") as f:
    f.write(wrap_html("Drawing All the Attention", body))

print(f"Saved: {ASSETS_DIR / '03e_kd_gravity.html'}")

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/03e_kd_gravity.html


<a id="section-4"></a>
# Section 4: Lineup Construction

Builds 3 assets for the Lineup Construction tab:

| File | What it shows |
|---|---|
| `04a_starting_lineup.html` | Fixed Four + Rotating Fifth table |
| `04b_team_gravity.html` | Houston player gravity comparison (7 players) |
| `04c_lineup_ratings.html` | Net rating cards for the 3 starting-five variants |

## Starting lineup analysis (exploratory — informs `04a`)

In [227]:
# ============================================================
# Section 04a: Starting lineup analysis
# ============================================================

# We have starters per game stored in all_lineups[game_id][0]["lineup"]
# (the GAME START entry)

starter_counts = {}
total_games_analyzed = 0

for game_id, history in all_lineups.items():
    if not history:
        continue
    game_start = history[0]
    if game_start.get("event") != "GAME START":
        continue
    starters = game_start["lineup"]
    if len(starters) != 5:
        continue
    
    total_games_analyzed += 1
    for player in starters:
        starter_counts[player] = starter_counts.get(player, 0) + 1

# Sort by games started
starter_df = pd.DataFrame([
    {"player": p, "games_started": c, "pct": c / total_games_analyzed * 100}
    for p, c in starter_counts.items()
]).sort_values("games_started", ascending=False).reset_index(drop=True)

print(f"Total games analyzed: {total_games_analyzed}")
print(f"\nStarter frequency:")
print(starter_df.to_string(index=False))

Total games analyzed: 82

Starter frequency:
             player  games_started       pct
      Amen Thompson             79 96.341463
       Kevin Durant             78 95.121951
   Jabari Smith Jr.             77 93.902439
     Alperen Sengun             72 87.804878
         Tari Eason             34 41.463415
        Josh Okogie             32 39.024390
      Reed Sheppard             21 25.609756
       Steven Adams             11 13.414634
       Clint Capela              3  3.658537
      Jae'Sean Tate              1  1.219512
Dorian Finney-Smith              1  1.219512
      Aaron Holiday              1  1.219512


## Build `04a_starting_lineup.html` — Fixed Four + Rotating Fifth

In [331]:
# ============================================================
# Section 04a: Starting Lineup
# ============================================================

# Group the data into "fixed core" + "rotating 5th slot"
# Plus note VanVleet (0 starts due to injury)

fixed_core = [
    ("Amen Thompson",    79),
    ("Kevin Durant",     78),
    ("Jabari Smith Jr.", 77),
    ("Alperen Sengun",   72),
]

rotating_fifth = [
    ("Tari Eason",          34),
    ("Josh Okogie",         32),
    ("Reed Sheppard",       21),
    ("Steven Adams",        11),
]


def build_lineup_row(player, starts):
    pct = starts / 82 * 100
    return f"""
    <tr>
      <td class="metric" style="text-align:left">{player}</td>
      <td class="value">{starts}</td>
      <td class="value">{pct:.0f}%</td>
    </tr>"""


fixed_rows = "".join(build_lineup_row(p, s) for p, s in fixed_core)
rotating_rows = "".join(build_lineup_row(p, s) for p, s in rotating_fifth)

body = f"""
  <div class="table-label">The Point Guard Hole, Section 04</div>
  <h2>The Lineup Houston Had</h2>
  <p class="table-subtitle">Houston's starting five was four locked-in players and one rotating slot. None of the four were a true point guard.</p>

  <div style="margin-top: 32px;">
    <div style="font-family: 'JetBrains Mono', monospace; font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; color: #6B7280; margin-bottom: 12px;">
      The Fixed Four
    </div>
    <table>
      <thead>
        <tr>
          <th style="text-align:left">Player</th>
          <th>Games Started</th>
          <th>% of Season</th>
        </tr>
      </thead>
      <tbody>{fixed_rows}
      </tbody>
    </table>
  </div>

  <div style="margin-top: 36px;">
    <div style="font-family: 'JetBrains Mono', monospace; font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; color: #6B7280; margin-bottom: 12px;">
      The Rotating Fifth
    </div>
    <table>
      <thead>
        <tr>
          <th style="text-align:left">Player</th>
          <th>Games Started</th>
          <th>% of Season</th>
        </tr>
      </thead>
      <tbody>{rotating_rows}
      </tbody>
    </table>
  </div>

  
"""

with open(ASSETS_DIR / "04a_starting_lineup.html", "w") as f:
    f.write(wrap_html("The Lineup Houston Had", body))

print(f"Saved: {ASSETS_DIR / '04a_starting_lineup.html'}")

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/04a_starting_lineup.html


## Background analysis: Reed Sheppard as starter

These cells were used to understand Reed's contribution as a starter vs. off the bench. The findings informed the lineup ratings cards in `04c`, but the detailed tables below don't appear in the final report.

In [231]:
# Reed's actual player ID
REED_PLAYER_ID = 1642263

# First, find which games Reed started
reed_started_game_ids = set()
for game_id, history in all_lineups.items():
    if not history:
        continue
    game_start = history[0]
    if game_start.get("event") == "GAME START" and "Reed Sheppard" in game_start["lineup"]:
        reed_started_game_ids.add(game_id)

print(f"Reed started: {len(reed_started_game_ids)} games\n")

# Pull Reed's per-game stats
reed_logs = player_gamelogs_rockets[
    (player_gamelogs_rockets["Player_ID"] == REED_PLAYER_ID) &
    (player_gamelogs_rockets["SEASON"] == "2025-26") &
    (player_gamelogs_rockets["SEASON_TYPE"] == "Regular Season")
].copy()

reed_logs["Game_ID_str"] = reed_logs["Game_ID"].astype(str).str.zfill(10)
reed_logs["started"] = reed_logs["Game_ID_str"].isin(reed_started_game_ids)

print(f"Reed total games played: {len(reed_logs)}")
print(f"Reed games as starter: {reed_logs['started'].sum()}")
print(f"Reed games off bench:   {(~reed_logs['started']).sum()}\n")

print("=" * 70)
print("Reed's stats: STARTER vs BENCH")
print("=" * 70)

for is_starter in [True, False]:
    label = "STARTED" if is_starter else "OFF BENCH"
    subset = reed_logs[reed_logs["started"] == is_starter]
    print(f"\n{label} ({len(subset)} games):")
    print(f"  MIN/G:        {subset['MIN'].mean():.1f}")
    print(f"  PTS/G:        {subset['PTS'].mean():.1f}")
    print(f"  AST/G:        {subset['AST'].mean():.1f}")
    print(f"  REB/G:        {subset['REB'].mean():.1f}")
    print(f"  STL/G:        {subset['STL'].mean():.1f}")
    print(f"  TOV/G:        {subset['TOV'].mean():.1f}")
    print(f"  +/- per game: {subset['PLUS_MINUS'].mean():+.1f}")
    fg3m, fg3a = subset['FG3M'].sum(), subset['FG3A'].sum()
    if fg3a > 0:
        print(f"  3PT%:         {fg3m / fg3a * 100:.1f}% ({fg3m}/{fg3a})")
    print(f"  Team W-L:     {(subset['WL'] == 'W').sum()}-{(subset['WL'] == 'L').sum()}")

Reed started: 21 games

Reed total games played: 82
Reed games as starter: 21
Reed games off bench:   61

Reed's stats: STARTER vs BENCH

STARTED (21 games):
  MIN/G:        30.2
  PTS/G:        15.4
  AST/G:        4.9
  REB/G:        3.9
  STL/G:        2.0
  TOV/G:        1.6
  +/- per game: +8.5
  3PT%:         38.5% (69/179)
  Team W-L:     17-4

OFF BENCH (61 games):
  MIN/G:        24.8
  PTS/G:        12.9
  AST/G:        2.9
  REB/G:        2.6
  STL/G:        1.3
  TOV/G:        1.5
  +/- per game: +1.7
  3PT%:         39.8% (158/397)
  Team W-L:     35-26


In [236]:
# Combined: Reed's starts with stats + the starting 5 lineup + game score
print("=" * 145)
print("Reed Sheppard's 21 STARTS — with starting lineup and final score")
print("=" * 145)
print(f"{'Date':<14} {'Opp':<10} {'WL':<3} {'Score':<10} {'MIN':>4} {'PTS':>4} {'AST':>4} {'TOV':>4} {'+/-':>5}   {'Starting 5'}")
print("-" * 145)

# Build a lookup for opponent points per game
# team_gl_2526 has HOU's PTS but not opponent's directly
# Use ESPN data we have for final scores

def get_final_score(game_id, hou_pts):
    """Return (hou_pts, opp_pts) using last play in ESPN data."""
    espn = espn_data.get(game_id)
    if not espn:
        return hou_pts, None
    plays = espn["plays"]
    if not plays:
        return hou_pts, None
    last = plays[-1]
    home_score = last.get("homeScore", 0)
    away_score = last.get("awayScore", 0)
    # Determine which is HOU
    teams = espn["boxscore"]["teams"]
    hou_is_home = None
    for t in teams:
        if str(t.get("team", {}).get("id")) == HOU_ESPN_ID:
            hou_is_home = t.get("homeAway") == "home"
            break
    if hou_is_home:
        return home_score, away_score
    else:
        return away_score, home_score


for _, row in reed_starts.iterrows():
    game_id = row["Game_ID_str"]
    history = all_lineups.get(game_id, [])
    starters = sorted(history[0]["lineup"]) if history else []
    short_names = [s.split()[-1] if s != "Jabari Smith Jr." else "Smith Jr." for s in starters]
    lineup_str = " / ".join(short_names)
    
    matchup_short = row["MATCHUP"].replace("HOU vs. ", "vs ").replace("HOU @ ", "@ ")
    
    # Get score
    hou_pts_team = team_gl_2526[team_gl_2526["Game_ID"] == row["Game_ID"]]["PTS"].iloc[0] if len(team_gl_2526[team_gl_2526["Game_ID"] == row["Game_ID"]]) else None
    hou_pts, opp_pts = get_final_score(game_id, hou_pts_team)
    score_str = f"{hou_pts}-{opp_pts}" if opp_pts is not None else "?"
    
    print(f"{row['GAME_DATE']:<14} {matchup_short:<10} {row['WL']:<3} {score_str:<10} "
          f"{row['MIN']:>4.0f} {row['PTS']:>4.0f} {row['AST']:>4.0f} {row['TOV']:>4.0f} "
          f"{row['PLUS_MINUS']:>+5.0f}   {lineup_str}")

Reed Sheppard's 21 STARTS — with starting lineup and final score
Date           Opp        WL  Score       MIN  PTS  AST  TOV   +/-   Starting 5
-------------------------------------------------------------------------------------------------------------------------------------------------
Nov 24, 2025   @ PHX      W   114-92       27    7    5    2    +6   Sengun / Thompson / Smith Jr. / Okogie / Sheppard
Nov 26, 2025   @ GSW      W   104-100      37   31    5    2    +7   Sengun / Thompson / Smith Jr. / Okogie / Sheppard
Nov 30, 2025   @ UTA      W   129-101      31    9    4    2   +33   Sengun / Thompson / Smith Jr. / Durant / Sheppard
Dec 01, 2025   @ UTA      L   125-133      23    9    2    0    +2   Sengun / Thompson / Smith Jr. / Durant / Sheppard
Feb 02, 2026   @ IND      W   118-114      25   11    5    2    +2   Sengun / Thompson / Smith Jr. / Sheppard / Eason
Feb 07, 2026   @ OKC      W   112-106      32   16    6    4    -3   Sengun / Smith Jr. / Durant / Sheppard / Eason

In [237]:
# Recompute Reed's starter stats EXCLUDING blowouts (final margin > 15)
# And excluding the Apr 12 game where the regular starters all rested

EXCLUDED_DATES = ["Apr 12, 2026"]  # rest game

reed_starts_competitive = reed_starts.copy()
# Add final margin
def compute_margin(row):
    game_id = row["Game_ID_str"]
    hou, opp = get_final_score(game_id, None)
    if hou and opp:
        return hou - opp
    return None
reed_starts_competitive["final_margin"] = reed_starts_competitive.apply(compute_margin, axis=1)

# Filter
mask = (
    (reed_starts_competitive["final_margin"].abs() <= 15) &
    (~reed_starts_competitive["GAME_DATE"].isin(EXCLUDED_DATES))
)
competitive = reed_starts_competitive[mask]
blowouts = reed_starts_competitive[~mask]

print(f"COMPETITIVE STARTS (final margin <= 15, excluding rest game):")
print(f"  Games: {len(competitive)}")
print(f"  Record: {(competitive['WL'] == 'W').sum()}-{(competitive['WL'] == 'L').sum()}")
print(f"  MIN/G: {competitive['MIN'].mean():.1f}")
print(f"  PTS/G: {competitive['PTS'].mean():.1f}")
print(f"  AST/G: {competitive['AST'].mean():.1f}")
print(f"  +/- per game: {competitive['PLUS_MINUS'].mean():+.1f}")

print(f"\nBLOWOUTS + REST GAME (excluded):")
print(f"  Games: {len(blowouts)}")
print(f"  Record: {(blowouts['WL'] == 'W').sum()}-{(blowouts['WL'] == 'L').sum()}")
print(f"  +/- per game: {blowouts['PLUS_MINUS'].mean():+.1f}")

COMPETITIVE STARTS (final margin <= 15, excluding rest game):
  Games: 14
  Record: 10-4
  MIN/G: 31.7
  PTS/G: 16.3
  AST/G: 5.4
  +/- per game: +4.2

BLOWOUTS + REST GAME (excluded):
  Games: 7
  Record: 7-0
  +/- per game: +17.1


In [239]:
# Reed's bench games (61 games)
reed_bench = reed_logs[~reed_logs["started"]].copy()
reed_bench["date_parsed"] = pd.to_datetime(reed_bench["GAME_DATE"], format="%b %d, %Y")
reed_bench = reed_bench.sort_values("date_parsed").reset_index(drop=True)

print("=" * 145)
print(f"Reed Sheppard's {len(reed_bench)} BENCH games — with starting lineup and final score")
print("=" * 145)
print(f"{'Date':<14} {'Opp':<10} {'WL':<3} {'Score':<10} {'MIN':>4} {'PTS':>4} {'AST':>4} {'TOV':>4} {'+/-':>5}   {'Starting 5'}")
print("-" * 145)

for _, row in reed_bench.iterrows():
    game_id = row["Game_ID_str"]
    history = all_lineups.get(game_id, [])
    starters = sorted(history[0]["lineup"]) if history else []
    short_names = [s.split()[-1] if s != "Jabari Smith Jr." else "Smith Jr." for s in starters]
    lineup_str = " / ".join(short_names)
    
    matchup_short = row["MATCHUP"].replace("HOU vs. ", "vs ").replace("HOU @ ", "@ ")
    hou_pts, opp_pts = get_final_score(game_id, None)
    score_str = f"{hou_pts}-{opp_pts}" if opp_pts is not None else "?"
    
    print(f"{row['GAME_DATE']:<14} {matchup_short:<10} {row['WL']:<3} {score_str:<10} "
          f"{row['MIN']:>4.0f} {row['PTS']:>4.0f} {row['AST']:>4.0f} {row['TOV']:>4.0f} "
          f"{row['PLUS_MINUS']:>+5.0f}   {lineup_str}")

Reed Sheppard's 61 BENCH games — with starting lineup and final score
Date           Opp        WL  Score       MIN  PTS  AST  TOV   +/-   Starting 5
-------------------------------------------------------------------------------------------------------------------------------------------------
Oct 21, 2025   @ OKC      L   124-125      28    9    4    2    +0   Sengun / Thompson / Smith Jr. / Durant / Adams
Oct 24, 2025   vs DET     L   111-115      20    9    1    3   -13   Sengun / Thompson / Smith Jr. / Durant / Adams
Oct 27, 2025   vs BKN     W   137-109      24   15    8    2    +9   Sengun / Thompson / Smith Jr. / Okogie / Durant
Oct 29, 2025   @ TOR      W   139-121      18    7    1    0    +0   Sengun / Thompson / Smith Jr. / Okogie / Durant
Nov 01, 2025   @ BOS      W   128-101      17   12    3    0   +19   Sengun / Thompson / Smith Jr. / Okogie / Durant
Nov 03, 2025   vs DAL     W   110-102      17    5    3    2    +5   Sengun / Thompson / Okogie / Durant / Eason
Nov 05, 

In [240]:
# Houston's record in Reed bench games
bench_w = (reed_bench["WL"] == "W").sum()
bench_l = (reed_bench["WL"] == "L").sum()
print(f"Reed BENCH games: {bench_w}-{bench_l}  ({bench_w/(bench_w+bench_l)*100:.1f}%)")

# Reed minutes distribution in bench games
print(f"\nBench game MIN distribution:")
print(f"  Mean: {reed_bench['MIN'].mean():.1f}")
print(f"  Median: {reed_bench['MIN'].median():.1f}")
print(f"  Games 30+ min off bench: {(reed_bench['MIN'] >= 30).sum()}")
print(f"  Games 20- min off bench: {(reed_bench['MIN'] < 20).sum()}")

Reed BENCH games: 35-26  (57.4%)

Bench game MIN distribution:
  Mean: 24.8
  Median: 25.0
  Games 30+ min off bench: 13
  Games 20- min off bench: 14


<a id="section-0"></a>
# Section 0: Overview (built last, displays first)

Builds the Overview tab assets. These come last in the notebook because they were built after the rest of the report was structured, but they appear first in the final report.

| File | What it shows |
|---|---|
| `00a_timeline_text.html` | The numbered timeline of 7 season-defining moments |
| `00b_timeline_horizontal.html` | Alternative horizontal timeline (NOT used in final report) |

## Build `00a_timeline_text.html` — Numbered text timeline (used in report)

In [581]:
# ============================================================
# Section 00a: Numbered text timeline
# ============================================================

events = [
    {
        "date": "JUL 6, 2025",
        "title": "Houston trades for Kevin Durant",
        "detail": "Jalen Green and Dillon Brooks headline a multi-team package sent out. Houston pivots from a defense-and-development core to a championship contender built around a 36-year-old Durant.",
    },
    {
        "date": "SUMMER 2025",
        "title": "Free agency reshapes the bench",
        "detail": "Houston signs key veterans Clint Capela, Dorian Finney-Smith, and Josh Okogie. The roster is now built to win now, not youth development.",
    },
    {
        "date": "SEP 22, 2025",
        "title": "Fred VanVleet tears ACL",
        "detail": "Houston's starting point guard is ruled out for the season before training camp. There is no replacement point guard on the roster.",
    },
    {
        "date": "OCT 21, 2025",
        "title": "Opening night without a point guard",
        "detail": "Amen Thompson starts at point guard with second-year guard Reed Sheppard coming off the bench.",
    },
    {
        "date": "FEB 5, 2026",
        "title": "Trade deadline passes quietly",
        "detail": "Despite the obvious roster hole, Houston makes no move for a point guard. The front office signals a pivot toward developing young players and waiting for VanVleet's return.",
    },
    {
        "date": "APR 12, 2026",
        "title": "Season ends at 52-30, 5th seed",
        "detail": "Same record as last season. But the West got tougher, dropping Houston from the 2-seed to the 5-seed.",
    },
    {
        "date": "PLAYOFFS",
        "title": "Eliminated in Round 1 by the Lakers in 6 games",
        "detail": "Houston's playoff run ends in six games against an undermanned Lakers team. Kevin Durant only played 1 game due to injury. A first-round exit caps the season.",
    },
]

rows = ""
for i, e in enumerate(events, 1):
    rows += f"""
    <div style="display: grid; grid-template-columns: 60px 1fr; gap: 24px; padding: 20px 0; border-bottom: 1px solid #E5E2DA;">
      <div style="font-family: 'JetBrains Mono', monospace; font-size: 12px; color: #CE1141; font-weight: 600; padding-top: 2px;">
        {i:02d}
      </div>
      <div>
        <div style="font-family: 'JetBrains Mono', monospace; font-size: 11px; letter-spacing: 0.12em; color: #6B7280; text-transform: uppercase; margin-bottom: 4px;">{e['date']}</div>
        <div style="font-family: 'Fraunces', Georgia, serif; font-size: 20px; font-weight: 600; letter-spacing: -0.01em; margin-bottom: 6px;">{e['title']}</div>
        <div style="font-size: 14.5px; line-height: 1.6; color: #374151;">{e['detail']}</div>
      </div>
    </div>"""

body = f"""
  <div class="table-label">Section 00, Introduction</div>
  <h2>The 2025–26 Houston Rockets, In Seven Moments</h2>
  <p class="table-subtitle">A championship window opened with a blockbuster trade — and an underwelming season followed.</p>
  
  <div style="margin-top: 32px; border-top: 1px solid #E5E2DA;">
    {rows}
  </div>
"""

with open(ASSETS_DIR / "00a_timeline_text.html", "w") as f:
    f.write(wrap_html("Season Timeline", body))

print(f"Saved: {ASSETS_DIR / '00a_timeline_text.html'}")

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/00a_timeline_text.html


## Build `00b_timeline_horizontal.html` — Horizontal timeline (NOT used)

Alternative layout explored during design. The numbered text timeline (`00a`) ended up in the final report instead.

In [ ]:
# ============================================================
# Section 00b: Horizontal timeline visual
# ============================================================

# Use the same events but render as a horizontal flow
timeline_events = [
    {"date": "Jul 6, 2025",   "label": "KD Trade",         "type": "boost",   "detail": "Pivot to title window"},
    {"date": "Summer 2025",   "label": "FA Signings",      "type": "neutral", "detail": "Capela, DFS, Okogie"},
    {"date": "Sep 22, 2025",  "label": "VanVleet ACL",     "type": "blow",    "detail": "Out for the season"},
    {"date": "Oct 21, 2025",  "label": "Tip-off",          "type": "neutral", "detail": "Amen starts at PG"},
    {"date": "Feb 5, 2026",   "label": "Deadline Pass",    "type": "blow",    "detail": "No PG acquired"},
    {"date": "Apr 12, 2026",  "label": "52–30, 5th seed",  "type": "neutral", "detail": "Same record, lower seed"},
    {"date": "Playoffs",      "label": "Lost 2–4 vs LAL",  "type": "blow",    "detail": "Out in 6 games"},
]

colors = {
    "boost":   "#4A8B5C",
    "blow":    "#CE1141",
    "neutral": "#6B7280",
}

# Build the SVG-like horizontal layout in flexbox
items_html = ""
for i, e in enumerate(timeline_events):
    color = colors[e["type"]]
    is_last = i == len(timeline_events) - 1
    
    items_html += f"""
    <div style="flex: 1; display: flex; flex-direction: column; align-items: center; position: relative; min-width: 0;">
      <!-- date above -->
      <div style="font-family: 'JetBrains Mono', monospace; font-size: 10px; color: #6B7280; text-transform: uppercase; letter-spacing: 0.1em; margin-bottom: 8px; text-align: center;">{e['date']}</div>
      <!-- dot -->
      <div style="width: 16px; height: 16px; border-radius: 50%; background: {color}; border: 3px solid white; box-shadow: 0 0 0 1px {color}; z-index: 2;"></div>
      <!-- label below -->
      <div style="font-family: 'Fraunces', Georgia, serif; font-size: 14px; font-weight: 600; margin-top: 14px; text-align: center; color: {color};">{e['label']}</div>
      <div style="font-size: 11px; color: #6B7280; margin-top: 4px; text-align: center; line-height: 1.4;">{e['detail']}</div>
    </div>
    """

# Wrap in container with connecting line
body = f"""
  <div class="table-label">Section 00, Introduction</div>
  <h2>The 2025–26 Houston Rockets, In Seven Moments</h2>
  <p class="table-subtitle">A championship window opened with a blockbuster trade — and an underwhelming season followed.</p>
  
  <div style="margin-top: 48px; margin-bottom: 32px; padding: 32px 0; position: relative;">
    <!-- horizontal connecting line -->
    <div style="position: absolute; top: 51px; left: 0; right: 0; height: 2px; background: #E5E2DA; z-index: 1;"></div>
    <!-- events -->
    <div style="display: flex; gap: 12px; position: relative;">
      {items_html}
    </div>
  </div>
  
  <div style="display: flex; gap: 24px; justify-content: center; margin-top: 36px; font-size: 12px; color: #6B7280;">
    <div style="display: flex; align-items: center; gap: 8px;">
      <span style="width: 10px; height: 10px; border-radius: 50%; background: #4A8B5C;"></span>
      <span>Positive moment</span>
    </div>
    <div style="display: flex; align-items: center; gap: 8px;">
      <span style="width: 10px; height: 10px; border-radius: 50%; background: #CE1141;"></span>
      <span>Setback</span>
    </div>
    <div style="display: flex; align-items: center; gap: 8px;">
      <span style="width: 10px; height: 10px; border-radius: 50%; background: #6B7280;"></span>
      <span>Neutral</span>
    </div>
  </div>
"""

with open(ASSETS_DIR / "00b_timeline_horizontal.html", "w") as f:
    f.write(wrap_html("Season Timeline (Horizontal)", body))

print(f"Saved: {ASSETS_DIR / '00b_timeline_horizontal.html'}")

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/00b_timeline_horizontal.html


# Section 02b: Overtime Problem (added later)

Built after the rest of Section 2 was complete. Slots into the Close-Out Problem tab as `02b_overtime_problem.html` between the clutch comparison (02a) and the four collapses (02c).

In [246]:
# Pull OT games and Houston's record in them
# An OT game has team MIN > 240 in the game log (extra minutes from OT)
ot_games = team_gl_2526[team_gl_2526["MIN"] > 240].copy()
print(f"Houston OT games this season: {len(ot_games)}")

# Record
wins = (ot_games["WL"] == "W").sum()
losses = (ot_games["WL"] == "L").sum()
print(f"Record: {wins}-{losses}")

# Show them
print("\nAll OT games:")
for _, row in ot_games.sort_values("GAME_DATE").iterrows():
    print(f"  {row['GAME_DATE']:<14} {row['MATCHUP']:<14} {row['WL']}  HOU {row['PTS']}")

Houston OT games this season: 8
Record: 1-7

All OT games:
  DEC 15, 2025   HOU @ DEN      L  HOU 125
  DEC 18, 2025   HOU @ NOP      L  HOU 128
  DEC 21, 2025   HOU @ SAC      L  HOU 124
  JAN 22, 2026   HOU @ PHI      L  HOU 122
  MAR 05, 2026   HOU vs. GSW    L  HOU 113
  MAR 25, 2026   HOU @ MIN      L  HOU 108
  NOV 16, 2025   HOU vs. ORL    W  HOU 117
  OCT 21, 2025   HOU @ OKC      L  HOU 124


## Preview: OT chart (early draft)

In [248]:
# ============================================================
# Section 02b (NEW): OT record + Minnesota collapse
# ============================================================

body = f"""
  <div class="table-label">The Close-Out Problem, Section 02</div>
  <h2>The Overtime Problem</h2>
  <p class="table-subtitle">When games went past regulation, Houston rarely closed.</p>

  <div class="callout-grid" style="grid-template-columns: 1fr 1.4fr; gap: 24px; margin-top: 32px; align-items: stretch;">
    
    <div class="card" style="display: flex; flex-direction: column; justify-content: center;">
      <div class="label">Record in Overtime</div>
      <div class="big">1–8</div>
      <div class="sub">Including 1 playoff game loss to the Lakers</div>
    </div>
    
    <div class="card" style="display: flex; flex-direction: column; justify-content: center;">
      <div class="label">The 13-Point OT Collapse</div>
      <div style="font-family: 'Fraunces', Georgia, serif; font-size: 22px; font-weight: 600; letter-spacing: -0.01em; margin: 8px 0 10px 0; color: #1A1A1A; line-height: 1.25;">
        Houston led Minnesota by 13 in overtime, and lost.
      </div>
      <div style="font-family: 'JetBrains Mono', monospace; font-size: 12px; color: #6B7280; letter-spacing: 0.08em;">
        MAR 25, 2026  ·  HOU 108 — MIN 110
      </div>
    </div>
    
  </div>

  <p class="table-footnote" style="margin-top: 24px;">
    Houston went 1–7 in regular-season overtime games. The lone OT win came against the Magic on November 16. The eighth overtime loss came in Game 3 of the first-round playoff series against the Lakers — a 6-point lead surrendered with 40 seconds left.
  </p>
"""

with open(ASSETS_DIR / "02b_overtime_problem.html", "w") as f:
    f.write(wrap_html("The Overtime Problem", body))

print(f"Saved: {ASSETS_DIR / '02b_overtime_problem.html'}")

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/02b_overtime_problem.html


## Build `02b_overtime_problem.html` — Production version

In [252]:
body = f"""
  <div class="table-label">The Close-Out Problem, Section 02</div>
  <h2>The Overtime Problem</h2>
  <p class="table-subtitle">When games went past regulation, Houston rarely closed.</p>

  <p style="font-size: 15px; line-height: 1.65; color: #374151; margin-top: 20px; max-width: 800px;">
    Houston went an abysmal 1–8 in overtime games this season. The pattern continued into the postseason: in Game 3 of the first-round series against the Lakers, Houston surrendered a 6-point lead with 40 seconds left and lost in overtime — turning what could have been a 1–2 series into a 0–3 hole.
  </p>

  <div class="callout-grid" style="grid-template-columns: 1fr 1.4fr; gap: 24px; margin-top: 32px; align-items: stretch;">
    
    <div class="card" style="display: flex; flex-direction: column; justify-content: center;">
      <div class="label">Record in Overtime</div>
      <div class="big">1–8</div>
      <div class="sub">Including the playoff loss to the Lakers</div>
    </div>
    
    <div class="card" style="display: flex; flex-direction: column; justify-content: center;">
      <div class="label">The 13-Point OT Collapse</div>
      <div style="font-family: 'Fraunces', Georgia, serif; font-size: 22px; font-weight: 600; letter-spacing: -0.01em; margin: 8px 0 10px 0; color: #1A1A1A; line-height: 1.25;">
        Houston led Minnesota by 13 in overtime, and lost.
      </div>
      <div style="font-family: 'JetBrains Mono', monospace; font-size: 12px; color: #6B7280; letter-spacing: 0.08em;">
        MAR 25, 2026  ·  HOU 108 — MIN 110
      </div>
    </div>
    
  </div>
"""

with open(ASSETS_DIR / "02b_overtime_problem.html", "w") as f:
    f.write(wrap_html("The Overtime Problem", body))

print(f"Saved: {ASSETS_DIR / '02b_overtime_problem.html'}")

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/02b_overtime_problem.html


### Cleanup: rename existing Section 2 files to make room for 02b

In [380]:
import shutil

# Rename the three Section 2 files to make room for 02b
renames = [
    ("02b_four_collapses.html",  "02c_four_collapses.html"),
    ("02c_when_kd_sits.html",    "02d_when_kd_sits.html"),
    ("02d_kd_off_scatter.html",  "02e_kd_off_scatter.html"),
]

for old, new in renames:
    old_path = ASSETS_DIR / old
    new_path = ASSETS_DIR / new
    if old_path.exists() and not new_path.exists():
        shutil.move(str(old_path), str(new_path))
        print(f"Renamed: {old}  →  {new}")
    elif new_path.exists():
        print(f"⚠ Skipped (target exists): {new}")
    else:
        print(f"⚠ Skipped (source missing): {old}")

⚠ Skipped (target exists): 02c_four_collapses.html
⚠ Skipped (target exists): 02d_when_kd_sits.html
⚠ Skipped (target exists): 02e_kd_off_scatter.html


## Build `04b_team_gravity.html` — Houston team gravity comparison (7 players)

In [553]:
# ============================================================
# Section 04b: Houston team gravity comparison
# ============================================================
import plotly.graph_objects as go

team_gravity = [
    {"name": "Kevin Durant",     "gv_mp": 694.8, "grav": 16.2},
    {"name": "Reed Sheppard",    "gv_mp": 536.9, "grav": 6.2},
    {"name": "Jabari Smith Jr.", "gv_mp": 668.2, "grav": 2.9},
    {"name": "Tari Eason",       "gv_mp": 364.2, "grav": 0.6},
    {"name": "Alperen Sengun",   "gv_mp": 591.3, "grav": 0.1},
    {"name": "Josh Okogie",      "gv_mp": 339.3, "grav": -1.1},
    {"name": "Amen Thompson",    "gv_mp": 727.3, "grav": -4.6},
]

# Sort ascending so highest is on top in a horizontal bar chart
players_sorted = sorted(team_gravity, key=lambda p: p["grav"])

names = [p["name"] for p in players_sorted]
gravs = [p["grav"] for p in players_sorted]
gv_mps = [p["gv_mp"] for p in players_sorted]

# Color KD red, others gray. Amen lightest because he's a special case
colors = []
for p in players_sorted:
    if p["name"] == "Kevin Durant":
        colors.append("#CE1141")
    elif p["name"] == "Amen Thompson":
        colors.append("#E5E2DA")  # very light to signal "outlier context"
    else:
        colors.append("#D1D5DB")

# Labels with player name + gravity minutes context
labels = [
    f"<b>{p['name']}</b><br><span style='font-size:11px;color:#6B7280'>{p['gv_mp']:.1f} gravity minutes</span>"
    for p in players_sorted
]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=gravs,
    y=labels,
    orientation="h",
    marker=dict(color=colors, line=dict(width=0)),
    text=[f"{g:+.1f}" for g in gravs],
    textposition="outside",
    textfont=dict(size=14, family="Inter, sans-serif", color="#1A1A1A"),
    hoverinfo="skip",
    cliponaxis=False,
))

# Vertical line at 0 to show positive/negative
fig.add_vline(x=0, line=dict(color="#9CA3AF", width=1, dash="dot"))

fig.update_layout(
    height=380,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Inter, sans-serif", color="#1A1A1A", size=13),
    margin=dict(l=20, r=70, t=30, b=50),
    showlegend=False,
    xaxis=dict(
        title="Average Gravity",
        zeroline=False,
        gridcolor="#F0EDE5",
        range=[min(gravs) - 4, max(gravs) + 4],
    ),
    yaxis=dict(
        showgrid=False,
        zeroline=False,
        tickfont=dict(size=13),
    ),
    bargap=0.35,
)

plotly_html = fig.to_html(include_plotlyjs='cdn', full_html=True, config={'displayModeBar': False, 'responsive': True})

import re
chart_div = re.search(r'<body>(.*?)</body>', plotly_html, re.DOTALL).group(1)

body = f"""
  <div class="table-label">Lineup Construction, Section 04</div>
  <h2>Beyond Durant, No One Pulled Defenders</h2>
  <p class="table-subtitle">Houston's gravity profile by player. Only Kevin Durant drew significant above-expected defensive attention. Of the three players who rotated through the 5th starter spot, Reed Sheppard pulled the most gravity.</p>
  <div class="chart-frame">{chart_div}</div>
  <p class="table-footnote" style="margin-top: 28px;">
    Gravity, a new NBA metric, measures how much defensive pressure a player draws above expected.<br>
    Source: NBA.com <a href="https://www.nba.com/inside-the-game/player/gravity" style="color: #CE1141; text-decoration: none;">Inside the Game</a> tracking data.
  </p>
  <div class="player-prose">
    <div class="player-name">About Amen Thompson's gravity</div>
    <p>Amen's negative gravity isn't a reflection of his impact, it's a function of his shooting profile. As a non-shooter from the perimeter, defenses were content to leave him open beyond the arc, daring him to take threes. The real story is the gap between KD and everyone else: how few teammates forced defenses to stay honest.</p>
  </div>
"""
##
with open(ASSETS_DIR / "04b_team_gravity.html", "w") as f:
    f.write(wrap_html("Team Gravity", body))

print(f"Saved: {ASSETS_DIR / '04b_team_gravity.html'}")
fig.show()

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/04b_team_gravity.html


## Build `04c_lineup_ratings.html` — Starting lineup ratings (3 cards)

In [355]:
# ============================================================
# Build 04c (production) — Starting Lineup Ratings cards
# ============================================================

LINEUP_DATA = [
    {
        "fifth": "Okogie",
        "gp": 41,
        "min": 377,
        "off_rtg": 121.8,
        "def_rtg": 111.5,
        "net_rtg": 10.3,
    },
    {
        "fifth": "Eason",
        "gp": 36,
        "min": 306,
        "off_rtg": 116.4,
        "def_rtg": 107.3,
        "net_rtg": 9.1,
    },
    {
        "fifth": "Sheppard",
        "gp": 44,
        "min": 273,
        "off_rtg": 114.7,
        "def_rtg": 121.1,
        "net_rtg": -6.4,
    },
]

POSITIVE = "#4A8B5C"
NEGATIVE = "#CE1141"


def color_for_net(value):
    return POSITIVE if value >= 0 else NEGATIVE


def fmt_net(value):
    return f"+{value:.1f}" if value >= 0 else f"{value:.1f}"


cards_html = ""
for lineup in LINEUP_DATA:
    net_color = color_for_net(lineup["net_rtg"])
    cards_html += f"""
    <div style="background: white; border-radius: 8px; padding: 24px; border-left: 4px solid {net_color}; box-shadow: 0 1px 3px rgba(0,0,0,0.04), 0 1px 2px rgba(0,0,0,0.06);">
      <div style="font-family: 'Inter', sans-serif; font-size: 13px; color: #6B7280; margin-bottom: 6px; font-weight: 500;">5th man</div>
      <div style="font-family: 'Fraunces', Georgia, serif; font-size: 22px; font-weight: 600; color: #1A1A1A; margin-bottom: 4px;">{lineup["fifth"]}</div>
      <div style="font-family: 'JetBrains Mono', monospace; font-size: 11px; letter-spacing: 0.08em; color: #6B7280; margin-bottom: 20px;">{lineup["gp"]} GP · {lineup["min"]} MIN</div>
      
      <div style="display: flex; justify-content: space-between; align-items: baseline; padding: 8px 0; border-bottom: 1px solid #F0EDE5;">
        <span style="font-size: 13px; color: #6B7280;">OFF RTG</span>
        <span style="font-family: 'JetBrains Mono', monospace; font-size: 18px; font-weight: 700; color: #1A1A1A;">{lineup["off_rtg"]:.1f}</span>
      </div>
      <div style="display: flex; justify-content: space-between; align-items: baseline; padding: 8px 0; border-bottom: 1px solid #F0EDE5;">
        <span style="font-size: 13px; color: #6B7280;">DEF RTG</span>
        <span style="font-family: 'JetBrains Mono', monospace; font-size: 18px; font-weight: 700; color: #1A1A1A;">{lineup["def_rtg"]:.1f}</span>
      </div>
      <div style="display: flex; justify-content: space-between; align-items: baseline; padding: 12px 0 4px;">
        <span style="font-size: 13px; color: #6B7280; font-weight: 600;">NET RTG</span>
        <span style="font-family: 'JetBrains Mono', monospace; font-size: 24px; font-weight: 700; color: {net_color};">{fmt_net(lineup["net_rtg"])}</span>
      </div>
    </div>
    """

body = f"""
  <h2>Starting Lineup Ratings</h2>
  <p class="table-subtitle">Houston's three most used lineups, which included the team's "fixed four" + a rotating 5th.</p>
  <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 20px; margin-top: 24px;">
    {cards_html}
  </div>
  <div class="player-prose">
    <p>Houston's starting lineups faced a tradeoff this season: the perimeter shooting and spacing Sheppard added came at a defensive cost. Whereas the Okogie and Eason lineups posted strong net ratings driven by solid defense.</p>
  </div>
"""

with open(ASSETS_DIR / "04c_lineup_ratings.html", "w") as f:
    f.write(wrap_html("Starting Lineup Ratings", body))

# Clean up preview files
for preview in ["preview_04c_cards.html", "preview_04c_table.html", "preview_04c_cards_with_lineup.html"]:
    p = ASSETS_DIR / preview
    if p.exists():
        p.unlink()
        print(f"Deleted: {preview}")

print(f"\nSaved: 04c_lineup_ratings.html")


Saved: 04c_lineup_ratings.html


<a id="section-5"></a>
# Section 5: The Playoffs (and Beyond)

Builds 1 asset for the Postseason & Beyond tab:

| File | What it shows |
|---|---|
| `05a_playoffs.html` | Game 3 callout card + vertical play-by-play timeline + 'Path Forward' closing prose |

## Build `05a_playoffs.html` — first draft (in-progress)

In [588]:
# ============================================================
# Section 05: The Playoffs + timeline
# ============================================================

# ==========================================
# Vertical Play-by-Play Timeline
# ==========================================

timeline_events = [
    {
        "clock": "40.6",
        "event": "Sengun finishes a running dunk to put Houston up 6",
        "score": "HOU 101 - 95 LAL"
    },
    {
        "clock": "34.4",
        "event": "Tate grabs a defensive rebound from Lakers' miss and passes to Smith Jr.",
        "score": ""
    },
    {
        "clock": "27.8",
        "event": "Smith Jr.'s pass to Thompson is picked off by Marcus Smart",
        "score": ""
    },
    {
        "clock": "25.4",
        "event": "Tate fouls Smart on a 3-point attempt. Smart makes all 3 free throws",
        "score": "HOU 101 - 98 LAL"
    },
    {
        "clock": "25.4",
        "event": "Sheppard checks in for Tate to be the ball handler",
        "score": ""
    },
    {
        "clock": "19.8",
        "event": "Sheppard gets stripped by LeBron",
        "score": ""
    },
    {
        "clock": "13.6",
        "event": "LeBron pulls up from 24 feet and ties the game",
        "score": "HOU 101 - 101 LAL"
    },
    {
        "clock": "0.0",
        "event": "End of regulation",
        "score": "HOU 101 - 101 LAL"
    }
]

timeline_color = "#CE1141"

# ==========================================
# Build Timeline
# ==========================================

items_html = ""

for i, e in enumerate(timeline_events):

    is_last = i == len(timeline_events) - 1

    connector = ""
    if not is_last:
        connector = """
        <div style="
            width: 2px;
            min-height: 40px;
            background: #E5E2DA;
            margin-top: 2px;
        "></div>
        """

    score_html = ""
    if e["score"]:
        score_html = f"""
        <div style="
            width: 100%;
            font-family: 'JetBrains Mono', monospace;
            font-size: 15px;
            font-weight: 700;
            color: {timeline_color};
            white-space: nowrap;
            text-align: right;
        ">
            {e['score']}
        </div>
        """

    items_html += f"""
    <div style="
        display: grid;
        grid-template-columns: 90px 30px 1fr 220px;
        align-items: start;
        margin-bottom: 0;
    ">

        <!-- Clock -->
        <div style="
            text-align: right;
            padding-right: 24px;
        ">
            <div style="
                font-family: 'JetBrains Mono', monospace;
                font-size: 14px;
                font-weight: 700;
                color: {timeline_color};
                letter-spacing: 0.03em;
            ">
                {e['clock']}
            </div>
        </div>

        <!-- Timeline Dot + Connector -->
        <div style="
            display: flex;
            flex-direction: column;
            align-items: center;
        ">
            <div style="
                width: 14px;
                height: 14px;
                border-radius: 50%;
                background: {timeline_color};
                border: 3px solid white;
                box-shadow: 0 0 0 1px {timeline_color};
                z-index: 2;
            "></div>

            {connector}
        </div>

        <!-- Event Description -->
        <div style="
            padding-left: 24px;
            padding-right: 24px;
            padding-bottom: 12px;
        ">
            <div style="
                font-family: Inter;
                font-size: 16px;
                font-weight: 600;
                line-height: 1.45;
                color: #111827;
            ">
                {e['event']}
            </div>
        </div>

        <!-- Score -->
        <div style="
            display: flex;
            justify-content: flex-end;
            padding-top: 2px;
        ">
            {score_html}
        </div>

    </div>
    """





body = f"""
  
  <h2>Game 3 Final Minute Collapse</h2>
  <p class="table-subtitle">The moment that decided the series.</p>
  
  <div class="callout-grid" style="grid-template-columns: 1fr;">
    <div class="card">
      <div class="label">vs LAL, Apr 24, 2026</div>
      <div style="
      font-family: 'Fraunces', Georgia, serif; 
      font-size: 22px; 
      font-weight: 600; 
      color: #1A1A1A; 
      line-height: 1.4; 
      margin-top: 8px;
      ">
        Up 6, with possession, 40 seconds left in regulation. Lost in overtime.
      </div>
    </div>
  </div>
  
  <!-- TIMELINE INSERTED HERE -->
  <div style="
    margin-top: 32px;
    margin-bottom: 32px;
    max-width: 1100px;
  ">
    {items_html}
  </div>

  <div class="player-prose" style="margin-top: 16px;">
    <p>Two ball-handling errors. A 6-point lead with the ball became a tie game in 27 seconds. Both turnovers were the kind a true point guard is built to prevent. This was where VanVleet's absence hit hardest.</p>
  </div>

  <h2 style="margin-top: 48px;">The Path Forward</h2>
  <div class="player-prose" style="margin-top: 16px;">
    <p>Houston's championship aspirations didn't end with Game 6. They ended at the trade deadline, when the front office chose not to address the structural problems. The playoffs only confirmed what the regular season already showed and the young core isn't ready to lead a deep run on its own. How Houston balances KD's win-now timeline with the young core's development will define how this team gets back to championship contention.</p>
  </div>
"""

with open(ASSETS_DIR / "05a_playoffs.html", "w") as f:
    f.write(wrap_html("Postseason & Beyond", body))

print(f"Saved: {ASSETS_DIR / '05a_playoffs.html'}")

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/05a_playoffs.html


## Build `05a_playoffs.html` — final version with vertical PBP timeline

In [509]:
# ==========================================
# Vertical Play-by-Play Timeline
# ==========================================

timeline_events = [
    {
        "clock": "40.6",
        "event": "Sengun finishes a running dunk to put Houston up 6",
        "score": "HOU 101 - 95 LAL"
    },
    {
        "clock": "34.4",
        "event": "Tate grabs a defensive rebound from Lakers' miss and passes to Smith Jr.",
        "score": ""
    },
    {
        "clock": "27.8",
        "event": "Smith Jr.'s pass to Thompson is picked off by Marcus Smart",
        "score": ""
    },
    {
        "clock": "25.4",
        "event": "Tate fouls Smart on a 3-point attempt. Smart makes all 3 free throws",
        "score": "HOU 101 - 98 LAL"
    },
    {
        "clock": "25.4",
        "event": "Sheppard checks in for Tate to be the ball handler",
        "score": ""
    },
    {
        "clock": "19.8",
        "event": "Sheppard gets stripped by LeBron",
        "score": ""
    },
    {
        "clock": "13.6",
        "event": "LeBron pulls up from 24 feet and ties the game",
        "score": "HOU 101 - 101 LAL"
    },
    {
        "clock": "0.0",
        "event": "End of regulation",
        "score": "HOU 101 - 101 LAL"
    }
]

timeline_color = "#CE1141"

# ==========================================
# Build Timeline
# ==========================================

items_html = ""

for i, e in enumerate(timeline_events):

    is_last = i == len(timeline_events) - 1

    connector = ""
    if not is_last:
        connector = """
        <div style="
            width: 2px;
            min-height: 60px;
            background: #E5E2DA;
            margin-top: 2px;
        "></div>
        """

    score_html = ""
    if e["score"]:
        score_html = f"""
        <div style="
            width: 100%;
            font-family: 'JetBrains Mono', monospace;
            font-size: 15px;
            font-weight: 700;
            color: {timeline_color};
            white-space: nowrap;
            text-align: right;
        ">
            {e['score']}
        </div>
        """

    items_html += f"""
    <div style="
        display: grid;
        grid-template-columns: 90px 30px 1fr 220px;
        align-items: start;
        margin-bottom: 0;
    ">

        <!-- Clock -->
        <div style="
            text-align: right;
            padding-right: 24px;
        ">
            <div style="
                font-family: 'JetBrains Mono', monospace;
                font-size: 14px;
                font-weight: 700;
                color: {timeline_color};
                letter-spacing: 0.03em;
            ">
                {e['clock']}
            </div>
        </div>

        <!-- Timeline Dot + Connector -->
        <div style="
            display: flex;
            flex-direction: column;
            align-items: center;
        ">
            <div style="
                width: 14px;
                height: 14px;
                border-radius: 50%;
                background: {timeline_color};
                border: 3px solid white;
                box-shadow: 0 0 0 1px {timeline_color};
                z-index: 2;
            "></div>

            {connector}
        </div>

        <!-- Event Description -->
        <div style="
            padding-left: 24px;
            padding-right: 24px;
            padding-bottom: 30px;
        ">
            <div style="
                font-family: Inter;
                font-size: 16px;
                font-weight: 600;
                line-height: 1.45;
                color: #111827;
            ">
                {e['event']}
            </div>
        </div>

        <!-- Score -->
        <div style="
            display: flex;
            justify-content: flex-end;
            padding-top: 2px;
        ">
            {score_html}
        </div>

    </div>
    """

# ==========================================
# Page Body
# ==========================================

body = f"""


<div style="
    margin-top: 48px;
    margin-bottom: 32px;
    max-width: 1100px;
">
    {items_html}
</div>

<div class="player-prose" style="margin-top: 16px;">
    <p>This game reflects the same pattern from the regular season's close out problems. A potential 1-2 series turned into a 0-3 hole from two costly turnovers.</p>
  </div>
"""

# ==========================================
# Export
# ==========================================

with open(ASSETS_DIR / "05b_play_by_play_timeline.html", "w") as f:
    f.write(
        wrap_html(
            "Houston Rockets Play-by-Play Timeline",
            body
        )
    )

print(f"Saved: {ASSETS_DIR / '05b_play_by_play_timeline.html'}")

Saved: /Users/stevenyan/Desktop/rockets_season_review/report_assets/05b_play_by_play_timeline.html


<a id="stitcher"></a>
# Full Report Stitcher

Combines all individual asset HTML files into a single tabbed report (`rockets_season_review.html`). Defines the tab order, applies the shared report-level CSS, and wraps each tab's contents in the navigation shell.

## Stitcher implementation — build the tabbed HTML

In [589]:
# ============================================================
# Stitch all assets into one tabbed HTML report
# ============================================================
import re
from pathlib import Path

# Section structure
SECTIONS = [
    {
        "key": "overview",
        "label": "Overview",
        "title": "The 2025–26 Houston Rockets",
        "intro": "A championship window opened with a blockbuster trade — and an underwhelming season followed.",
        "show_section_intro": False,  # asset has its own header
        "assets": [
            "00a_timeline_text.html",
        ],
    },
    {
        "key": "lineup",
        "label": "Lineup Construction",
        "title": "Lineup Construction",
        "intro": "Houston entered the season expecting Fred VanVleet at point guard. He tore his ACL before training camp, and the roster offered no replacement. Amen Thompson, a 6'7\" wing, handled primary ball-handling duties all season, and the resulting lineups created a structural spacing problem.",
        "show_section_intro": True,
        "assets": [
            "04a_starting_lineup.html",
            "04c_lineup_ratings.html",
            "04b_team_gravity.html",
        ],
    },
    {
        "key": "playstyle",
        "label": "Playstyle Identity",
        "title": "Playstyle Identity",
        "intro": "Houston played an old-school brand of basketball, much like last season. They dominated the paint at the same elite level — but their three-point shooting, already a weakness, got worse.",
        "show_section_intro": True,
        "assets": [
            "01a_interior_identity.html",
            "01b_three_point_problem.html",
            "01c_three_ball_didnt_fall.html",
            "01d_heatmap_starters.html",
            "01e_heatmap_supporting_cast.html",
            "01f_heatmap_reliable_shooters.html",
        ],
    },
    {
        "key": "closeout",
        "label": "The Close-Out Problem",
        "title": "The Close-Out Problem",
        "intro": "When games tightened, Houston regressed. Clutch-time performance fell off a cliff, overtime games turned into a death sentence, and a handful of late collapses defined the season's most painful losses.",
        "show_section_intro": True,
        "assets": [
            "02a_clutch_comparison.html",
            "02b_overtime_problem.html",
            "02c_four_collapses.html",
            "02d_when_kd_sits.html",
            "02d_kd_off_scatter.html",
        ],
    },
    {
        "key": "kd",
        "label": "KD Reliance",
        "title": "KD Reliance",
        "intro": "Kevin Durant carried more of Houston's offense than any peer his age. When he sat, the wheels frequently came off.",
        "show_section_intro": True,
        "assets": [
            "03b_kd_headline.html",
            "03c_kd_vs_stars.html",
            "03d_kd_involvement.html",
            "03e_kd_gravity.html",
            "03a_wheels_came_off.html",
        ],
    },
    {
        "key": "playoffs",
        "label": "Postseason & Beyond",
        "title": "Postseason & Beyond",
        "intro": "Houston entered the first round with a real opportunity. The Lakers were without Luka for the entire series and without Reaves for the first four games. Even with KD playing only one game due to injury, Houston's young core should have been talented enough to beat an undermanned Lakers team. They weren't. Houston lost in six, falling behind 0-3 before winning two of the final three games.",
        "show_section_intro": True,
        "assets": [
            "05a_playoffs.html",
        ],
    },
]


def extract_body_content(html_path):
    """Pull the inner content of <body> from an asset file."""
    content = Path(html_path).read_text()
    body_match = re.search(r"<body[^>]*>(.*?)</body>", content, re.DOTALL)
    if not body_match:
        return ""
    return body_match.group(1).strip()


# Build tabs
tab_buttons = []
tab_contents = []

for i, section in enumerate(SECTIONS):
    is_active = (i == 0)
    
    tab_buttons.append(f"""
      <button class="tab-button {'active' if is_active else ''}" data-tab="{section['key']}">
        {section['label']}
      </button>""")
    
    asset_blocks = []
    for asset_filename in section["assets"]:
        asset_path = ASSETS_DIR / asset_filename
        if not asset_path.exists():
            asset_blocks.append(f"<!-- Missing: {asset_filename} -->")
            continue
        body_content = extract_body_content(asset_path)
        asset_blocks.append(f'<div class="asset-block">{body_content}</div>')
    
    section_intro_html = ""
    if section.get("show_section_intro") and section.get("intro"):
        section_intro_html = f"""
        <div class="section-intro">
          <h1 class="section-title">{section['title']}</h1>
          <p class="section-lede">{section['intro']}</p>
        </div>"""
    
    tab_contents.append(f"""
      <div class="tab-content {'active' if is_active else ''}" id="tab-{section['key']}">
        {section_intro_html}
        {''.join(asset_blocks)}
      </div>""")


styles = """
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    font-family: 'Inter', sans-serif;
    background: #FAFAF7;
    color: #1A1A1A;
    line-height: 1.5;
  }
  
  /* Report header */
  .report-header {
    background: #1A1A1A;
    color: white;
    padding: 32px 40px 20px;
  }
  .report-header .report-eyebrow {
    font-family: 'JetBrains Mono', monospace;
    font-size: 11px;
    letter-spacing: 0.18em;
    text-transform: uppercase;
    color: #CE1141;
    margin-bottom: 8px;
  }
  .report-header h1 {
    font-family: 'Fraunces', Georgia, serif;
    font-size: 28px;
    font-weight: 600;
    letter-spacing: -0.015em;
    color: white;
  }
  
  /* Tab bar */
  .tab-bar {
    background: #1A1A1A;
    padding: 0 40px;
    position: sticky;
    top: 0;
    z-index: 100;
    border-bottom: 1px solid #2A2A2A;
    display: flex;
    gap: 4px;
    overflow-x: auto;
  }
  .tab-button {
    background: transparent;
    border: none;
    color: #9CA3AF;
    font-family: 'Inter', sans-serif;
    font-size: 14px;
    font-weight: 500;
    padding: 16px 18px;
    cursor: pointer;
    border-bottom: 3px solid transparent;
    white-space: nowrap;
    transition: color 0.15s, border-color 0.15s;
  }
  .tab-button:hover { color: white; }
  .tab-button.active {
    color: white;
    border-bottom-color: #CE1141;
  }
  
  /* Tab content */
  .tab-content { display: none; }
  .tab-content.active { display: block; }
  
  /* Section intro */
  .section-intro {
    max-width: 1100px;
    margin: 0 auto;
    padding: 48px 40px 24px;
  }
  .section-title {
    font-family: 'Fraunces', Georgia, serif;
    font-size: 36px;
    font-weight: 600;
    letter-spacing: -0.02em;
    margin-bottom: 12px;
  }
  .section-lede {
    font-size: 18px;
    color: #4B5563;
    line-height: 1.6;
    max-width: 1020px;
  }
  
  /* Asset block container */
  .asset-block { padding: 16px 0; }
  .asset-block .wrapper {
    max-width: 1100px;
    margin: 0 auto;
    padding: 24px 40px;
    background: transparent;
  }
  
  /* HIDE the per-asset section labels (redundant with tabs) */
  .asset-block .table-label { display: none; }
  
  /* ===== Asset-level styles ===== */
  
  .asset-block h2 {
    font-family: 'Fraunces', Georgia, serif;
    font-size: 26px;
    font-weight: 600;
    letter-spacing: -0.015em;
    margin-bottom: 8px;
    color: #1A1A1A;
  }
  
  .asset-block .table-subtitle,
  .asset-block .subtitle {
    color: #6B7280;
    font-size: 15px;
    margin-bottom: 24px;
    line-height: 1.6;
  }
  .asset-block .table-footnote,
  .asset-block .footnote,
  .asset-block .callout-footer,
  .asset-block .legend-note {
    color: #6B7280;
    font-size: 12px;
    margin-top: 14px;
    font-style: italic;
    line-height: 1.5;
  }
  
  /* Legend swatch (for "When KD Sits" chart) */
  .asset-block .legend-swatch {
    display: inline-block;
    width: 12px;
    height: 12px;
    background: #E5E2DA;
    border: 1px solid #D1D5DB;
    border-radius: 2px;
    margin-right: 6px;
    vertical-align: middle;
  }
  
  /* Tables */
  .asset-block table {
    width: 100%;
    border-collapse: collapse;
    margin-top: 16px;
    background: white;
    border-radius: 8px;
    overflow: hidden;
    box-shadow: 0 1px 3px rgba(0,0,0,0.04), 0 1px 2px rgba(0,0,0,0.06);
  }
  .asset-block thead {
    background: #1A1A1A;
    color: white;
  }
  .asset-block th {
    text-align: right;
    padding: 14px 18px;
    font-family: 'JetBrains Mono', monospace;
    font-size: 11px;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    font-weight: 500;
  }
  .asset-block th:first-child { text-align: left; }
  .asset-block td {
    padding: 14px 18px;
    border-bottom: 1px solid #F0EDE5;
    text-align: right;
    font-size: 14px;
  }
  .asset-block tbody tr:last-child td { border-bottom: none; }
  .asset-block tbody tr:hover { background: #FAFAF7; }
  .asset-block td.metric {
    text-align: left;
    font-weight: 500;
    color: #1A1A1A;
  }
  .asset-block td.value {
    font-family: 'JetBrains Mono', monospace;
    color: #374151;
  }
  
  /* Stat cards (callout-grid) */
  .asset-block .callout-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 20px;
    margin-top: 24px;
  }
  .asset-block .card {
    background: white;
    border-radius: 8px;
    padding: 24px;
    border-left: 4px solid #CE1141;
    box-shadow: 0 1px 3px rgba(0,0,0,0.04), 0 1px 2px rgba(0,0,0,0.06);
  }
  .asset-block .card .label {
    font-family: 'Inter', sans-serif;
    font-size: 13px;
    color: #6B7280;
    margin-bottom: 12px;
    font-weight: 500;
  }
  .asset-block .card .big {
    font-family: 'JetBrains Mono', monospace;
    font-size: 42px;
    font-weight: 700;
    letter-spacing: -0.02em;
    color: #1A1A1A;
    line-height: 1;
    margin-bottom: 8px;
  }
  .asset-block .card .sub {
    font-family: 'Inter', sans-serif;
    font-size: 12px;
    color: #6B7280;
  }
  
  /* Chart frame */
  .asset-block .chart-frame {
    background: white;
    border-radius: 8px;
    padding: 8px;
    box-shadow: 0 1px 3px rgba(0,0,0,0.04), 0 1px 2px rgba(0,0,0,0.06);
  }
  .asset-block .chart-frame .plotly-graph-div,
  .asset-block .chart-frame .js-plotly-plot {
    width: 100% !important;
  }
  
  /* Player prose boxes */
  .asset-block .player-prose {
    margin-top: 28px;
    margin-bottom: 28px;
    padding: 20px 24px;
    background: white;
    border-left: 3px solid #CE1141;
    border-radius: 4px;
  }
  .asset-block .player-prose .player-name {
    font-family: 'Fraunces', Georgia, serif;
    font-size: 18px;
    font-weight: 600;
    margin-bottom: 8px;
    color: #1A1A1A;
  }
  .asset-block .player-prose p {
    font-size: 15.5px;
    line-height: 1.65;
    color: #374151;
  }
"""


full_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Houston Rockets 2025–26 Season Review</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600;9..144,700&family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@500;700&display=swap" rel="stylesheet">
<style>{styles}</style>
</head>
<body>

<header class="report-header">
  <div class="report-eyebrow">Season Review · 2025–26</div>
  <h1>Houston Rockets</h1>
</header>

<nav class="tab-bar">
  {''.join(tab_buttons)}
</nav>

<main>
  {''.join(tab_contents)}
</main>

<script>
  const buttons = document.querySelectorAll('.tab-button');
  const contents = document.querySelectorAll('.tab-content');
  
  function resizePlotlyCharts(container) {{
    if (!window.Plotly) return;
    const plotlyDivs = container.querySelectorAll('.js-plotly-plot');
    plotlyDivs.forEach(div => {{
      try {{
        window.Plotly.Plots.resize(div);
      }} catch (e) {{}}
    }});
  }}
  
  buttons.forEach(button => {{
    button.addEventListener('click', () => {{
      const targetTab = button.dataset.tab;
      
      buttons.forEach(b => b.classList.remove('active'));
      contents.forEach(c => c.classList.remove('active'));
      
      button.classList.add('active');
      const activeContent = document.getElementById('tab-' + targetTab);
      activeContent.classList.add('active');
      
      setTimeout(() => resizePlotlyCharts(activeContent), 50);
      
      window.scrollTo({{ top: 0, behavior: 'instant' }});
    }});
  }});
  
  window.addEventListener('load', () => {{
    const activeTab = document.querySelector('.tab-content.active');
    if (activeTab) resizePlotlyCharts(activeTab);
  }});
  window.addEventListener('resize', () => {{
    const activeTab = document.querySelector('.tab-content.active');
    if (activeTab) resizePlotlyCharts(activeTab);
  }});
</script>

</body>
</html>"""

output_path = PROJECT_ROOT / "rockets_season_review.html"
output_path.write_text(full_html)

print(f"Saved: {output_path}")
print("Open it in your browser to see the report.")

Saved: /Users/stevenyan/Desktop/rockets_season_review/rockets_season_review.html
Open it in your browser to see the report.


<a id="appendix"></a>
# Appendix: Stitcher Fixes & Verification

End-of-build cells used to:
- Patch specific styling issues in individual asset files
- Verify specific statistics quoted in the prose (paint share, 3PT share, ASB splits)

In [590]:
# Fix the inline max-width on 02b's prose paragraph
path = ASSETS_DIR / "02b_overtime_problem.html"
content = path.read_text()

# Remove the inline max-width from the prose paragraph
content = content.replace(
    'style="font-size: 15px; line-height: 1.65; color: #374151; margin-top: 20px; max-width: 800px;"',
    'style="font-size: 15px; line-height: 1.65; color: #374151; margin-top: 20px;"'
)

path.write_text(content)
print(f"Fixed: {path}")

Fixed: /Users/stevenyan/Desktop/rockets_season_review/report_assets/02b_overtime_problem.html


## Verify class-naming consistency across the 'problem' asset files

In [591]:
# Find the actual class used in the three problem files
import re

problem_files = [
    "01c_three_ball_didnt_fall.html",
    "02d_when_kd_sits.html",
    "02e_kd_off_scatter.html",
]

for filename in problem_files:
    path = ASSETS_DIR / filename
    content = path.read_text()
    print(f"--- {filename} ---")
    # Look for the final paragraph
    paragraphs = re.findall(r'<p[^>]*>.*?</p>', content, re.DOTALL)
    if paragraphs:
        # Show the last paragraph (likely the footnote)
        print(paragraphs[-1][:300])
    print()

--- 01c_three_ball_didnt_fall.html ---
<p class="callout-footer">League average: 13.7 games per team under 10 3PM in 2025–26.</p>

--- 02d_when_kd_sits.html ---
<p class="legend-note"><span class="legend-swatch"></span> Shaded regions indicate when Kevin Durant was off the court.</p>

--- 02e_kd_off_scatter.html ---
<p class="legend-note">Each dot represents one game. Dot size reflects total minutes Durant spent on the bench. Annotated games had a swing of 13 points or more.</p>



## Stat verification cells

These compute specific numbers that appear in the report's prose, so they can be sanity-checked against the assets.

In [592]:
# ============================================================
# Verify ASB dates and pull all stats for shot chart prose
# ============================================================
import pandas as pd

ASB_LAST = pd.Timestamp("2026-02-11")
ASB_FIRST = pd.Timestamp("2026-02-19")

ts = team_gl_2526.copy()
ts["date_parsed"] = pd.to_datetime(ts["GAME_DATE"], format="%b %d, %Y")
ts = ts.sort_values("date_parsed").reset_index(drop=True)

last_pre = ts[ts["date_parsed"] <= ASB_LAST]["date_parsed"].max()
first_post = ts[ts["date_parsed"] >= ASB_FIRST]["date_parsed"].min()
print(f"Houston last pre-ASB game: {last_pre.strftime('%b %d, %Y')}")
print(f"Houston first post-ASB game: {first_post.strftime('%b %d, %Y')}")

# ============================================================
# Eason: pre/post ASB splits
# ============================================================
EASON_ID = 1631106

eason_logs = player_gamelogs_rockets[
    (player_gamelogs_rockets["Player_ID"] == EASON_ID) &
    (player_gamelogs_rockets["SEASON"] == "2025-26") &
    (player_gamelogs_rockets["SEASON_TYPE"] == "Regular Season")
].copy()
eason_logs["date_parsed"] = pd.to_datetime(eason_logs["GAME_DATE"], format="%b %d, %Y")
eason_logs = eason_logs.sort_values("date_parsed").reset_index(drop=True)

pre = eason_logs[eason_logs["date_parsed"] <= ASB_LAST]
post = eason_logs[eason_logs["date_parsed"] >= ASB_FIRST]

print(f"\n=== EASON ===")
print(f"Total games: {len(eason_logs)}")
print(f"\nPre-ASB ({len(pre)} games):")
print(f"  3PM/3PA: {pre['FG3M'].sum()}/{pre['FG3A'].sum()} = {pre['FG3M'].sum()/max(pre['FG3A'].sum(),1)*100:.1f}%")
print(f"  PTS/G: {pre['PTS'].mean():.1f}")
print(f"\nPost-ASB ({len(post)} games):")
print(f"  3PM/3PA: {post['FG3M'].sum()}/{post['FG3A'].sum()} = {post['FG3M'].sum()/max(post['FG3A'].sum(),1)*100:.1f}%")
print(f"  PTS/G: {post['PTS'].mean():.1f}")

# First 11 games for the "led the league" claim
first11 = eason_logs.head(11)
print(f"\nFirst 11 games:")
print(f"  3PM/3PA: {first11['FG3M'].sum()}/{first11['FG3A'].sum()} = {first11['FG3M'].sum()/max(first11['FG3A'].sum(),1)*100:.1f}%")

# Find the 21-miss streak — walk through games and find longest miss streak
print(f"\n=== Searching for longest missed-3 streak ===")
# Need PBP data for this, or we can approximate by counting from box scores
# A "consecutive miss" can span across games. Let's at least find the worst stretches.

# Find worst 7-game stretch by 3PT%
print("\nWorst 7-game 3PT% stretches (min 10 attempts):")
for i in range(len(eason_logs) - 6):
    window = eason_logs.iloc[i:i+7]
    pct = window['FG3M'].sum() / max(window['FG3A'].sum(), 1) * 100
    attempts = window['FG3A'].sum()
    if attempts >= 10:
        first_date = window['GAME_DATE'].iloc[0]
        last_date = window['GAME_DATE'].iloc[-1]
        misses = window['FG3A'].sum() - window['FG3M'].sum()
        print(f"  {first_date} → {last_date}: {window['FG3M'].sum()}/{attempts} = {pct:.1f}% ({misses} misses)")

Houston last pre-ASB game: Feb 11, 2026
Houston first post-ASB game: Feb 19, 2026

=== EASON ===
Total games: 60

Pre-ASB (31 games):
  3PM/3PA: 69/150 = 46.0%
  PTS/G: 12.2

Post-ASB (29 games):
  3PM/3PA: 24/110 = 21.8%
  PTS/G: 8.6

First 11 games:
  3PM/3PA: 27/53 = 50.9%

=== Searching for longest missed-3 streak ===

Worst 7-game 3PT% stretches (min 10 attempts):
  Oct 21, 2025 → Nov 05, 2025: 19/34 = 55.9% (15 misses)
  Oct 24, 2025 → Nov 07, 2025: 21/34 = 61.8% (13 misses)
  Oct 27, 2025 → Nov 09, 2025: 21/38 = 55.3% (17 misses)
  Oct 29, 2025 → Nov 12, 2025: 20/37 = 54.1% (17 misses)
  Nov 01, 2025 → Nov 14, 2025: 19/38 = 50.0% (19 misses)
  Nov 03, 2025 → Dec 21, 2025: 18/35 = 51.4% (17 misses)
  Nov 05, 2025 → Dec 23, 2025: 16/35 = 45.7% (19 misses)
  Nov 07, 2025 → Dec 25, 2025: 13/32 = 40.6% (19 misses)
  Nov 09, 2025 → Dec 27, 2025: 12/31 = 38.7% (19 misses)
  Nov 12, 2025 → Dec 29, 2025: 13/29 = 44.8% (16 misses)
  Nov 14, 2025 → Jan 01, 2026: 11/25 = 44.0% (14 misses)
 

### Amen Thompson: % of points from the paint

In [593]:
# ============================================================
# 1. Amen Thompson: % of points from paint (close-range)
# ============================================================
AMEN_ID = 1641708

amen_shots = shots[
    (shots["PLAYER_ID"] == AMEN_ID) &
    (shots["SEASON_LABEL"] == "2025-26") &
    (shots["SEASON_TYPE"] == "Regular Season") &
    (shots["TEAM_ID"] == ROCKETS_ID)
].copy()

# Paint = "In The Paint (Non-RA)" + "Restricted Area"
paint_zones = ["Restricted Area", "In The Paint (Non-RA)"]
paint_shots = amen_shots[amen_shots["SHOT_ZONE_BASIC"].isin(paint_zones)]
non_paint_shots = amen_shots[~amen_shots["SHOT_ZONE_BASIC"].isin(paint_zones)]

paint_makes = paint_shots["SHOT_MADE_FLAG"].sum()
paint_pts_from_fg = paint_makes * 2  # all paint makes are 2-pointers

# Need to also add FTs for total points. Use his game log
amen_logs = player_gamelogs_rockets[
    (player_gamelogs_rockets["Player_ID"] == AMEN_ID) &
    (player_gamelogs_rockets["SEASON"] == "2025-26") &
    (player_gamelogs_rockets["SEASON_TYPE"] == "Regular Season")
]
amen_total_pts = amen_logs["PTS"].sum()

# Points NOT from paint = non-paint FGs * 2 (or 3) + all FTs
non_paint_pts = sum(
    3 if shots["SHOT_TYPE"].iloc[i] == "3PT Field Goal" else 2
    for i in non_paint_shots.index
    if shots["SHOT_MADE_FLAG"].iloc[i] == 1
)
# Use a simpler approach: compute from grouping
non_paint_pts = 0
for _, row in non_paint_shots.iterrows():
    if row["SHOT_MADE_FLAG"] == 1:
        non_paint_pts += 3 if row["SHOT_TYPE"] == "3PT Field Goal" else 2

ft_pts = amen_total_pts - paint_pts_from_fg - non_paint_pts

print(f"=== AMEN THOMPSON ===")
print(f"Total points (season): {amen_total_pts}")
print(f"Paint FG points: {paint_pts_from_fg} ({paint_makes} makes × 2)")
print(f"Non-paint FG points: {non_paint_pts}")
print(f"Free throw points: {ft_pts}")
print(f"\n% of points from paint FGs: {paint_pts_from_fg / amen_total_pts * 100:.1f}%")
print(f"% of points from paint + FTs (close-range total): {(paint_pts_from_fg + ft_pts) / amen_total_pts * 100:.1f}%")

# ============================================================
# 2. KD + Jabari + Reed combined 3PT share
# ============================================================
KD_ID = 201142
JABARI_ID = 1631095
REED_ID = 1642263

trio_logs = player_gamelogs_rockets[
    (player_gamelogs_rockets["Player_ID"].isin([KD_ID, JABARI_ID, REED_ID])) &
    (player_gamelogs_rockets["SEASON"] == "2025-26") &
    (player_gamelogs_rockets["SEASON_TYPE"] == "Regular Season")
]

trio_3pa = trio_logs["FG3A"].sum()
trio_3pm = trio_logs["FG3M"].sum()
team_3pa = team_gl_2526["FG3A"].sum()
team_3pm = team_gl_2526["FG3M"].sum()

print(f"\n=== KD + JABARI + REED 3PT SHARE ===")
print(f"Trio 3PA: {trio_3pa}, Team 3PA: {team_3pa}")
print(f"  → {trio_3pa / team_3pa * 100:.1f}% of team's 3-point attempts")
print(f"Trio 3PM: {trio_3pm}, Team 3PM: {team_3pm}")
print(f"  → {trio_3pm / team_3pm * 100:.1f}% of team's 3-point makes")

# Individual 3PT% for context
print(f"\nIndividual splits:")
for pid, name in [(KD_ID, "KD"), (JABARI_ID, "Jabari"), (REED_ID, "Reed")]:
    p = trio_logs[trio_logs["Player_ID"] == pid]
    pct = p["FG3M"].sum() / max(p["FG3A"].sum(), 1) * 100
    print(f"  {name}: {p['FG3M'].sum()}/{p['FG3A'].sum()} = {pct:.1f}%")

=== AMEN THOMPSON ===
Total points (season): 1443
Paint FG points: 1004 (502 makes × 2)
Non-paint FG points: 135
Free throw points: 304

% of points from paint FGs: 69.6%
% of points from paint + FTs (close-range total): 90.6%

=== KD + JABARI + REED 3PT SHARE ===
Trio 3PA: 1514, Team 3PA: 2580
  → 58.7% of team's 3-point attempts
Trio 3PM: 590, Team 3PM: 940
  → 62.8% of team's 3-point makes

Individual splits:
  KD: 186/450 = 41.3%
  Jabari: 177/488 = 36.3%
  Reed: 227/576 = 39.4%


### Dorian Finney-Smith: 3PT% verification

In [594]:
DFS_ID = 1627827
dfs_logs = player_gamelogs_rockets[
    (player_gamelogs_rockets["Player_ID"] == DFS_ID) &
    (player_gamelogs_rockets["SEASON"] == "2025-26") &
    (player_gamelogs_rockets["SEASON_TYPE"] == "Regular Season")
].copy()
dfs_logs["date_parsed"] = pd.to_datetime(dfs_logs["GAME_DATE"], format="%b %d, %Y")
dfs_logs = dfs_logs.sort_values("date_parsed").reset_index(drop=True)

print(f"DFS games this season: {len(dfs_logs)}")
print(f"First game: {dfs_logs['GAME_DATE'].iloc[0]}")
print(f"3PT: {dfs_logs['FG3M'].sum()}/{dfs_logs['FG3A'].sum()} = {dfs_logs['FG3M'].sum()/max(dfs_logs['FG3A'].sum(),1)*100:.1f}%")
print(f"PPG: {dfs_logs['PTS'].mean():.1f}")
print(f"MPG: {dfs_logs['MIN'].mean():.1f}")

DFS games this season: 37
First game: Dec 25, 2025
3PT: 27/100 = 27.0%
PPG: 3.3
MPG: 16.7
